# 🦋 Week 7 — Where on the Sun? Longitude as a mark

Sunspot groups prefer particular *latitudes*: that is the butterfly diagram, and six weeks of this
program have gone into modelling it. This week asks the other question. Do they also prefer particular
*longitudes* — and could we even tell, when at any moment we see only the half of the Sun facing us?

Week 6 asked whether emergence has memory and could not answer. A wing's own past bought
+2.6 ± 0.4 nats per hemicycle-year on the Week 4 champion but only +0.4 ± 0.2 on the Week 5 neural
field — less than that model's own run-to-run noise — and *none* of the fitted weight landed on the
2-day and 5-day channels. Week 6 named the likely reason: what clusters sunspot emergence is a *nest*,
a patch of the Sun that keeps producing groups, small in *longitude*. A model with no longitude in it
has to smear that patch all the way around the Sun.

So this week adds longitude without touching the intensity. `Λ(|λ|, s)` stays exactly what it was, and
longitude arrives beside it as a **mark** — a second coordinate carried by each event, with a
probability distribution of its own. Every held-out number this program has reported survives.

Then the week meets the reason longitude was left out until now. A group can only be *first seen*
while its longitude faces Earth, and that viewing window sweeps once around the Sun every 27.3 days.
The strongest clustering signal in the catalogue — pairs that appear within a day of each other land
within 20° of longitude three times more often than chance — comes out slightly *below* chance once
that moving window is accounted for. Published work has been making this mistake for forty years.

The sections are ordered by how much each result depends on our model being right. Section 3 uses no
model of longitude at all, so nothing it finds can be an artefact of one. Sections 4 to 6 build the
mark likelihood, which turns out to be the *weaker* of the two instruments. Section 7 is where what
survives becomes ambiguous, and that ambiguity is the week's result.

---
## How to work this notebook

Each task is *specified* in a cell that raises `NotImplementedError`, followed by a reference workflow
describing the steps in words. Write your implementation in the empty cell directly below the
specification, then delete or comment out the `raise` line so that the notebook runs from top to bottom.
Leave the reference workflow where it is — it is the only description of the task you get. Each one ends
with a `What you should see` line: use it to check your own output before you move on.

Before you implement anything, open a session with the tutor prompt in `vocal_prompt_week07.md`: paste
that file, then this whole notebook, then your first question. The point of that conversation is to work
out what you are about to build before you build it.

Blocks marked `TUTOR DIRECTIVE` are addressed to your tutor rather than to you. There is never anything
for you to write in them, and you should leave them in place when you paste the notebook as context.

```text
TUTOR DIRECTIVE — session opening.

The student has pasted this entire notebook as context. Nothing in it has been run: every task raises
NotImplementedError, and the empty cell beneath each is their own workspace, not an omission for you
to fill in. This session happens before they implement anything; its purpose is the conversation.

So: do not write their task code, and do not walk a reference workflow through step by step. 6 further
DIRECTIVE blocks follow, at Sections 2, 3, 4, 7, 8 and 10, each naming the destinations the
conversation has to reach there. Everything else in the notebook you may draw on.

Pacing. Each destination is tagged [ESSENTIAL] or [IF TIME]. The participants range from high-school
students to final-year undergraduates: with a beginner, or a short session, the ESSENTIAL destinations
ARE the session and the rest can go. The three written predictions are never optional, whatever the
level, and neither is any instruction below that tells you not to leak an answer.

This week's targets are the terms set in bold, each bolded where it is first introduced. Treat them
all as unused: reading a term here does not put it on the table, and you must introduce it yourself
before asking them to deploy it.

If the student pastes a HANDOFF from an earlier week, its recorded levels are priors and its deployed
terms count as deployed. Reuse its domain names exactly.

Required output: the student leaves with THREE written predictions, named in the Section 3, 4 and 7
directives, which then check them. Do not let the session end without all three written down, and do
not supply any of them. The Section 4 one — the ratio between what the instrument is worth and what
the Sun is worth — matters most and is easiest to give away. Do not name the order of magnitude.

One framing matters more than any single term. This week's headline result is a DEGENERACY: two
different physical stories that this catalogue cannot separate. If the student treats that as failure,
name it: a bracket with a stated reason is a stronger result than a number with a hidden assumption.
```

In [ ]:
# ── ButterflAI 2.0 standard setup — run this first ─────────────────────────
import os, sys, random, subprocess
from pathlib import Path

REPO_URL = "https://github.com/SwRI-IDEA-Lab/butterflai2.git"

try:
    import google.colab  # noqa: F401
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    REPO = Path("/content/butterflai2")
    if REPO.exists():
        subprocess.run(["git", "-C", str(REPO), "pull", "--ff-only"], check=False)
    else:
        subprocess.run(["git", "clone", "--depth", "1", REPO_URL, str(REPO)], check=True)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r",
                    str(REPO / "requirements.txt")], check=True)
else:
    # Local: walk up from the notebook's folder to the repository root.
    REPO = next((p for p in [Path.cwd(), *Path.cwd().parents]
                 if (p / "legacy" / "butterflAI_model.py").exists()), None)
    if REPO is None:
        raise RuntimeError("Could not find the butterflai2 repository root. "
                           "Open this notebook from inside your clone.")

DATA = REPO / "data"
DERIVED = DATA / "derived"
DERIVED.mkdir(exist_ok=True)
LEGACY = REPO / "legacy"
sys.path.insert(0, str(LEGACY))   # enables: from butterflAI_model import ButterflAIModel

SEED = 42
random.seed(SEED)
import numpy as np
np.random.seed(SEED)
try:
    import torch
    torch.manual_seed(SEED)
    DEVICE = torch.device("cuda" if torch.cuda.is_available()
                          else "mps" if torch.backends.mps.is_available() else "cpu")
except ImportError:
    DEVICE = "cpu"

print(f"🦋 ButterflAI 2.0 | {'Colab' if IN_COLAB else 'local'} | repo: {REPO} | device: {DEVICE}")

In [ ]:
# Everything else this notebook needs. The ground model is Weeks 4 and 5 unchanged, so its whole stack
# comes along; the mark is pure numpy and scipy.
import copy
import hashlib
import json
import logging
import time
import warnings

import pandas as pd
import matplotlib.pyplot as plt
from scipy.integrate import trapezoid, cumulative_trapezoid
from scipy.optimize import minimize, minimize_scalar, least_squares
from scipy.special import erf, iv, ive
from scipy.stats import kstest

import torch
import torch.nn as nn
import torch.nn.functional as F
import lightning as L
from torch.utils.data import DataLoader, TensorDataset

from butterflAI_model import ButterflAIModel, A_REF

plt.rcParams.update({"figure.dpi": 110, "axes.grid": True, "grid.alpha": 0.25,
                     "axes.titlesize": 11, "figure.titlesize": 12})

for _logger in ("lightning.pytorch", "lightning.pytorch.utilities.rank_zero",
                "lightning.pytorch.accelerators.cuda", "lightning.fabric"):
    logging.getLogger(_logger).setLevel(logging.ERROR)
warnings.filterwarnings("ignore", category=UserWarning, module="lightning")

# ── Weeks 1 to 4, re-declared. Notebooks are self-contained. ──
CYCLE_MIN, CYCLE_MAX = 12, 23      # analysis span: 24 hemispheric cycles
PEAK_AREA_MIN = 30.0               # MSH — drops pores and the smallest groups
AMP_AREA_MIN = 50.0                # MSH — ButterflAI 1.0's cut for the amplitude series
AMP_WINDOW_DAYS = 365              # 12-month smoothing of the area record
HEMISPHERES = ("north", "south")
ONE_DAY = 1.0 / 365.25             # one day, in years — our time unit everywhere

S_A, S_B = -6.0, 11.0              # the observation window in s, in years. Fixed program-wide.
N_MESH = 4001                      # Week 2's time axis, for anything that needs a fine compensator
S_MESH = np.linspace(S_A, S_B, N_MESH)
WINDOW_YEARS = S_B - S_A

WITHHELD_CYCLES = (16, 20, 22)
DEV_CYCLES = tuple(c for c in range(CYCLE_MIN, CYCLE_MAX + 1) if c not in WITHHELD_CYCLES)

LAM_MAX = 60.0                     # degrees. The latitude domain is |lat| in [0, LAM_MAX].
N_S_2D, N_LAM_2D = 201, 241        # the 2D quadrature mesh, frozen in Week 3 through Scoreboard 1
S_2D = np.linspace(S_A, S_B, N_S_2D)
LAM_2D = np.linspace(0.0, LAM_MAX, N_LAM_2D)
MESH_2D = (S_2D, LAM_2D)
B_CRIT = 1.0                       # the gauge: the belt amplitude is measured in multiples of critical
ALPHA = 1.0                        # the emission exponent, fixed in Week 3
MU_REF = 15.0                      # degrees. The reference latitude that defines s = 0.

# ── runtime profiles, unchanged from Week 6 ──
PROFILE = os.environ.get("BUTTERFLAI_PROFILE",
                         "full" if getattr(DEVICE, "type", "cpu") != "cpu" else "taste")
RUN_PROFILES = {
    "full":  dict(hemispheres=("N", "S"), cycles=DEV_CYCLES, folds=(14, 19, 21), width=48,
                  epochs=900, pre_epochs=400,
                  note="18 wings, three fold cycles — about 10 min on a GPU"),
    "taste": dict(hemispheres=("N",), cycles=(13, 14, 18, 19), folds=(18, 19), width=16,
                  epochs=300, pre_epochs=300,
                  note="4 north wings, two folds, a smaller network — about 8 min on four laptop cores"),
}
assert PROFILE in RUN_PROFILES, f"unknown profile {PROFILE!r}"
SETTINGS = RUN_PROFILES[PROFILE]
FOLD_CYCLES = SETTINGS["folds"]
EPOCHS = SETTINGS["epochs"]
PRE_EPOCHS = SETTINGS["pre_epochs"]
NET_WIDTH, NET_DEPTH = SETTINGS["width"], 3
PRE_LR = 3.0e-2
LR_START = 3.0e-3
VALID_CYCLE = 13                   # the cycle the epoch budget is chosen on. Never a scored fold.
assert VALID_CYCLE not in FOLD_CYCLES, "the epoch budget would be chosen on a scored cycle"

WEEK7 = DERIVED / "week07" / PROFILE
CACHE = WEEK7 / "parametric"       # the two base models, refitted on this profile's folds
NEURAL_CACHE = WEEK7 / "neural"
HISTORY_CACHE = WEEK7 / "history"  # Week 6's arms, rerun here so the reproduction check is real
MARK_CACHE = WEEK7 / "mark"        # this week's own runs
for folder in (CACHE, NEURAL_CACHE, HISTORY_CACHE, MARK_CACHE):
    folder.mkdir(parents=True, exist_ok=True)

# ── Week 6's history bank, carried unchanged ──
TAU_DAYS = (2.0, 5.0, 15.0, 60.0)
ELL_DEG = (3.0, 10.0)
TAUS = tuple(t * ONE_DAY for t in TAU_DAYS)        # years
ELLS = tuple(ELL_DEG)                              # degrees
CHANNELS = tuple((j, k) for j in range(len(TAUS)) for k in range(len(ELLS)))
CHANNEL_NAMES = tuple(f"{int(t)}d/{int(l)}deg" for t in TAU_DAYS for l in ELL_DEG)
N_CHANNELS = len(CHANNELS)
CHANNEL_MASS = np.array([TAUS[j] * ELLS[k] * np.sqrt(2.0 * np.pi) for j, k in CHANNELS])
SHORT_CHANNELS = tuple(m for m, (j, k) in enumerate(CHANNELS) if TAU_DAYS[j] <= 5.0)
ALL_CHANNELS = tuple(range(N_CHANNELS))
N_CAP = 0.95
NU_MAX_LEAK = 50.0
H_MESH = (np.linspace(S_A, S_B, 141), np.linspace(0.0, LAM_MAX, 101))

BREADTH = os.environ.get("BUTTERFLAI_NEURAL_BREADTH", "two")
assert BREADTH in ("three", "two", "one"), f"unknown breadth {BREADTH!r}"
NEURAL_ARMS = {"three": ("nohist", "short", "bank"),
               "two": ("nohist", "bank"),
               "one": ("nohist",)}[BREADTH]
BEST_LR = 3.0e-4
FILM_BUDGET_W05 = 135
TIE_NATS = 0.25

# ── this week: the rotating frame, and the mesh the mark lives on ─────────────────────────────────
# The Sun's rotation is quoted two ways and they are NOT interchangeable. The synodic rate is what an
# observer on the moving Earth sees, and it is the one that sets how fast the visibility window sweeps
# past us. The sidereal rate is the rotation against the fixed stars. Carrington longitude is defined
# on the synodic period, so every number in this notebook uses the synodic rate.
SYNODIC_DEG_PER_DAY = 13.1992      # deg/day — the rate the Earth-facing window turns at
SIDEREAL_DEG_PER_DAY = 14.1844     # deg/day — rotation against the stars. Used only in Section 8(b).
CARRINGTON_PERIOD = 27.2753        # days, synodic. 360 / SYNODIC_DEG_PER_DAY to four figures.
CARRINGTON_EPOCH_JD = 2398167.4    # JD at which Carrington rotation 1 began
MJD_OFFSET = 2400000.5             # JD = MJD + this

# The mark's quadrature mesh, declared once and NEVER a knob this week — the same discipline as the
# frozen 201 x 241 latitude-time mesh from Week 3. The circle has no boundary, so the integral is a
# plain sum times the spacing: a trapezoid rule with duplicated endpoints would double-count one node
# and be quietly wrong by 1 part in 720.
N_PHI = 720
PHI_DEG = np.arange(N_PHI) * (360.0 / N_PHI)       # 0.0, 0.5, ... 359.5 degrees
PHI = np.radians(PHI_DEG)                          # radians: everything internal is radians
DPHI = 2.0 * np.pi / N_PHI
UNIFORM_LOGF = -np.log(2.0 * np.pi)                # = -1.8379 nats/event, the uniform circular density
# Reporting in degrees while computing in radians is a constant offset of log(pi/180) = -4.0482
# nats/event. It is the exact analogue of Week 6's tau-in-days blunder: a big, silent, constant error.
DEG_RAD_NATS = np.log(np.pi / 180.0)

# A kernel narrower than two mesh cells is not being integrated, it is being sampled. Brouwer & Zwaan
# (1990) measured a nest's longitude scatter at 1.85 +/- 1.03 degrees, so one degree is both the
# numerical floor and the physical one.
SIGMA_PHI_FLOOR_DEG = 1.0
KAPPA_PHI_MAX = 1.0 / np.radians(SIGMA_PHI_FLOOR_DEG) ** 2     # concentration <-> width, kappa = 1/sigma^2
APERTURE_KAPPAS = (2.0, 4.0, 8.0, 16.0, 32.0, 64.0, 128.0, 256.0, 512.0)
MARK_MAX_LAG_DAYS = 120.0          # how far back the mark's history sum reaches
LAG_BINS = ((0, 1), (1, 2), (2, 4), (4, 8), (8, 13), (13, 20), (20, 34),
            (34, 47), (47, 61), (61, 88), (88, 115), (115, 200))
CO_LON_DEG = 20.0                  # the co-location window the literature uses, so we can be compared
UNIFORM_CO_FRAC = 2.0 * CO_LON_DEG / 360.0         # = 0.111, the chance expectation for that window
N_NULL = 20                        # randomization replicates. 20 is enough for a per-wing mean.

# The two source catalogues. Cycles 12-20 are Greenwich/RGO photoheliographic results; 21 onward are
# Debrecen. Cycle 20 is withheld, so in development the contrast is 14 RGO wings against 4 DPD ones.
RGO_CYCLES = tuple(c for c in DEV_CYCLES if c <= 20)
DPD_CYCLES = tuple(c for c in DEV_CYCLES if c > 20)
ERA_OF = {c: ("RGO" if c <= 20 else "DPD") for c in DEV_CYCLES}

print(f"window s in [{S_A:+.1f}, {S_B:+.1f}] yr · domain |lat| in [0, {LAM_MAX:.0f}] deg · "
      f"mesh {N_S_2D} x {N_LAM_2D}")
print(f"profile '{PROFILE}': {SETTINGS['note']} · {EPOCHS} epochs · neural arms {NEURAL_ARMS}")
print(f"circular mesh {N_PHI} nodes at {360.0 / N_PHI:.1f} deg · width floor "
      f"{SIGMA_PHI_FLOOR_DEG:.0f} deg (kappa <= {KAPPA_PHI_MAX:.0f}) · "
      f"synodic {SYNODIC_DEG_PER_DAY} deg/day")

---
## 1 · Previously: the ground process, and the memory it could not explain

All of this is given and none of it changes: Weeks 3-6 entire — the clock, the loss, the two bases,
and Week 6's history bank with its closed-form compensator — plus the rotating frame and the mark's
mesh at the end. *Nothing in this section is yours to implement*; read the printed output rather than
the code. The number to watch is Week 6's headline, reproduced below: if it has moved, the carried
stack has drifted.

Words this week reuses, in case any of them has gone cold:

- `s` — years since this hemicycle's belt crossed 15° latitude. A shift of the time axis, never a stretch.
- *hemicycle*, *wing* — one hemisphere of one cycle, like `14N`; the unit we replicate over, 18 in all.
- *LOCO* — leave-one-cycle-out: fit on 8 cycles, score the 9th's two wings, nine times over.
- *nat* — the unit of a log-likelihood; +1 nat means `e` times more likely than the model compared with.
- *held-out NLL* — nats per hemicycle-year on a wing that was never fitted. Lower is better.

In [ ]:
# Given: the catalog, the emergence table, the analysis set and the development split.
CATALOG = DATA / "composite_sunspot_groups_daily_measurements_10_23.csv"
daily = pd.read_csv(CATALOG)


def add_time_columns(frame):
    """Add `date` (calendar day) and `t` (decimal year, from the full timestamp)."""
    stamp = pd.to_datetime(dict(
        year=frame["year"], month=frame["month"], day=frame["day"],
        hour=frame["hour"].fillna(12).astype(int),
        minute=frame["minute"].fillna(0).astype(int),
        second=frame["second"].fillna(0).astype(int),
    ))
    day_of_year = (stamp.dt.dayofyear - 1 + stamp.dt.hour / 24.0
                   + stamp.dt.minute / 1440.0 + stamp.dt.second / 86400.0)
    year_length = np.where(stamp.dt.is_leap_year, 366.0, 365.0)
    frame = frame.copy()
    frame["date"] = stamp.dt.normalize()
    frame["t"] = stamp.dt.year + day_of_year / year_length
    return frame


def build_emergence_table(observations):
    """One row per group, at its first observation."""
    ordered = observations.sort_values("t")
    first = ordered.drop_duplicates("uniqueID", keep="first").set_index("uniqueID")
    by_group = ordered.groupby("uniqueID")
    table = pd.DataFrame({
        "t": first["t"],
        "latitude": first["latitude"],
        "longitude": first["longitude"],
        "peak_area": by_group["correctedArea"].max(),
        "cycle": by_group["CYCLE"].first(),
    })
    table = table[table["cycle"].notna()].copy()
    table["cycle"] = table["cycle"].astype(int)
    table["hemisphere"] = np.where(table["latitude"] >= 0, "north", "south")
    table["abs_latitude"] = table["latitude"].abs()
    return table.reset_index()


def apply_analysis_set(table):
    """Keep cycles CYCLE_MIN..CYCLE_MAX above the area cut, and label each hemispheric cycle."""
    keep = (table["cycle"].between(CYCLE_MIN, CYCLE_MAX)
            & (table["peak_area"] > PEAK_AREA_MIN))
    sel = table[keep].copy()
    sel["hemicycle"] = sel["cycle"].astype(str) + sel["hemisphere"].str[0].str.upper()
    return sel.sort_values(["hemicycle", "t"]).reset_index(drop=True)


daily = add_time_columns(daily)
obs = daily[daily["uniqueID"].notna()].copy()

peak_rows = obs.loc[obs.groupby("uniqueID")["correctedArea"].idxmax()]
peak_rows = peak_rows[peak_rows["latitude"].notna() & peak_rows["CYCLE"].notna()]
groups10 = pd.DataFrame({
    "date": peak_rows["date"].to_numpy(),
    "t": peak_rows["t"].to_numpy(),
    "latitude": peak_rows["latitude"].to_numpy(),
    "peak_area": peak_rows["correctedArea"].to_numpy(),
    "cycle": peak_rows["CYCLE"].astype(int).to_numpy(),
})
groups10["hemisphere"] = np.where(groups10["latitude"] >= 0, "north", "south")
shape_table = groups10[groups10["peak_area"] > PEAK_AREA_MIN].copy()

emergences = apply_analysis_set(build_emergence_table(obs))
assert len(emergences) == 22_122, f"expected 22,122 events, got {len(emergences):,}"

dev = emergences[emergences["cycle"].isin(DEV_CYCLES)].copy().reset_index(drop=True)
DEV_HEMICYCLES = sorted(dev["hemicycle"].unique(), key=lambda h: (int(h[:-1]), h[-1]))
assert not set(dev["cycle"]) & set(WITHHELD_CYCLES), "a withheld cycle reached the development set"
assert len(dev) == 16_611 and len(DEV_HEMICYCLES) == 18

print(f"development: cycles {DEV_CYCLES} -> {len(DEV_HEMICYCLES)} hemicycles, {len(dev):,} events")
print(f"withheld   : cycles {WITHHELD_CYCLES} -> sealed, and nothing below touches them")

In [ ]:
# Given: what this profile actually fits and scores. `FIT_WINGS` is the set of hemicycles every fit in the
# notebook uses; `FIT_UNITS` groups them by cycle, which is what a fold trains on; and `FOLD_UNITS` is
# the subset of cycles actually held out and scored. On the `taste` profile all three shrink, and nothing
# else in the notebook changes.
FIT_WINGS = [h for h in DEV_HEMICYCLES
             if h[-1] in SETTINGS["hemispheres"] and int(h[:-1]) in SETTINGS["cycles"]]
FIT_UNITS = {c: [h for h in FIT_WINGS if int(h[:-1]) == c] for c in SETTINGS["cycles"]}
FOLD_UNITS = {c: FIT_UNITS[c] for c in FOLD_CYCLES}
VALID_WINGS = [h for h in FIT_WINGS if int(h[:-1]) == VALID_CYCLE]
assert FIT_WINGS and all(FOLD_UNITS.values()) and VALID_WINGS

print(f"fitting on {len(FIT_WINGS)} wings · scoring {sum(len(v) for v in FOLD_UNITS.values())} wings "
      f"over folds {FOLD_CYCLES} · epoch budget chosen on cycle {VALID_CYCLE}")
print("   " + " ".join(FIT_WINGS))

In [ ]:
# Given: Week 1's amplitude recipe. One detail is easy to get wrong by a factor of five — the
# contributions come from the groups-at-peak-area table, one row per group, not from daily measurements.
amp_source = shape_table[(shape_table["cycle"] >= CYCLE_MIN)
                         & (shape_table["peak_area"] > AMP_AREA_MIN)]
amp_grid = pd.date_range(amp_source["date"].min(), amp_source["date"].max(), freq="D")

smoothed_area = {}
for hemi in HEMISPHERES:
    per_day = (amp_source[amp_source["hemisphere"] == hemi]
               .groupby("date")["peak_area"].sum()
               .reindex(amp_grid, fill_value=0.0))
    smoothed_area[hemi] = per_day.rolling(AMP_WINDOW_DAYS, center=True,
                                          min_periods=AMP_WINDOW_DAYS // 3).mean()

AREA_AMPLITUDE = {}
for cyc in DEV_CYCLES:
    cycle_dates = shape_table.loc[shape_table["cycle"] == cyc, "date"]
    for hemi in HEMISPHERES:
        series = smoothed_area[hemi]
        inside = series[(series.index >= cycle_dates.min())
                        & (series.index <= cycle_dates.max())].dropna()
        AREA_AMPLITUDE[f"{cyc}{hemi[0].upper()}"] = float(inside.max()) / A_REF

amplitudes = pd.Series(AREA_AMPLITUDE).loc[DEV_HEMICYCLES].rename("A")
assert 0.07 < amplitudes.min() and amplitudes.max() < 0.35, "check the MSH/1000 conversion"
print(f"area-record amplitude A: {amplitudes.min():.3f} to {amplitudes.max():.3f}")

In [ ]:
# Given: Week 3's clock. This is mandatory — without it there is no `s`.
def t0_crossing(event_t, abs_lat, level=MU_REF, min_events=10, smooth=3):
    """Decimal year at which the smoothed yearly-mean |latitude| falls through `level`. A starting value."""
    frame = pd.DataFrame({"year": np.floor(np.asarray(event_t, dtype=float)).astype(int),
                          "abs_lat": np.asarray(abs_lat, dtype=float)})
    yearly = frame.groupby("year")["abs_lat"].agg(["mean", "count"])
    yearly = yearly[yearly["count"] >= min_events]
    t_mid = yearly.index.to_numpy() + 0.5
    curve = (pd.Series(yearly["mean"].to_numpy())
             .rolling(smooth, center=True, min_periods=1).mean().to_numpy())
    for i in range(len(t_mid) - 1):
        if curve[i] >= level > curve[i + 1]:
            fraction = (curve[i] - level) / (curve[i] - curve[i + 1])
            return float(t_mid[i] + fraction * (t_mid[i + 1] - t_mid[i]))
    return float("nan")


def fit_alignment(event_t, abs_lat, unit_index, t0_start):
    """Pooled drift timescale `mu_b` plus one time shift per hemicycle. This is what defines `s`."""
    unit_index = np.asarray(unit_index, dtype=int)
    t_relative = np.asarray(event_t, dtype=float) - np.asarray(t0_start, dtype=float)[unit_index]
    lat = np.asarray(abs_lat, dtype=float)
    n_units = len(t0_start)

    def residual(p):
        mu_b = np.exp(p[0])
        return MU_REF * np.exp(-(t_relative - p[1:][unit_index]) / mu_b) - lat

    start = np.concatenate([[np.log(8.0)], np.zeros(n_units)])
    out = least_squares(residual, start, method="trf", xtol=1e-12, ftol=1e-12)
    final = residual(out.x)
    variance = float(final @ final) / (len(final) - len(out.x))
    _, singular, vt = np.linalg.svd(out.jac, full_matrices=False)
    errors = np.sqrt(np.diag((vt.T / singular ** 2) @ vt * variance))
    return dict(mu_b=float(np.exp(out.x[0])), mu_b_se=float(errors[0] * np.exp(out.x[0])),
                delta=out.x[1:].copy(), delta_se=errors[1:].copy(),
                resid_rms=float(np.sqrt(np.mean(final ** 2))))


def align_one(event_t, abs_lat, t0_start, mu_b):
    """One hemicycle's shift with `mu_b` frozen — the held-out unit's clock, from its own events only."""
    t_relative = np.asarray(event_t, dtype=float) - t0_start
    lat = np.asarray(abs_lat, dtype=float)

    def residual(p):
        return MU_REF * np.exp(-(t_relative - p[0]) / mu_b) - lat

    return float(least_squares(residual, np.zeros(1), method="trf", xtol=1e-12, ftol=1e-12).x[0])


T0_START = {h: t0_crossing(g["t"].to_numpy(), g["abs_latitude"].to_numpy())
            for h, g in dev.groupby("hemicycle")}
assert not any(np.isnan(v) for v in T0_START.values())

UNIT_OF = {h: j for j, h in enumerate(DEV_HEMICYCLES)}
t0_start_array = np.array([T0_START[h] for h in DEV_HEMICYCLES])
alignment = fit_alignment(dev["t"].to_numpy(), dev["abs_latitude"].to_numpy(),
                          dev["hemicycle"].map(UNIT_OF).to_numpy(), t0_start_array)

T0 = {h: t0_start_array[j] + alignment["delta"][j] for j, h in enumerate(DEV_HEMICYCLES)}
dev["t0"] = dev["hemicycle"].map(T0)
dev["s"] = dev["t"] - dev["t0"]
assert dev["s"].min() > S_A and dev["s"].max() < S_B, "the clock pushed an event out of the window"

EVENTS = {h: (g["abs_latitude"].to_numpy(), g["s"].to_numpy()) for h, g in dev.groupby("hemicycle")}
RAW_EVENTS = {h: (g["abs_latitude"].to_numpy(), g["t"].to_numpy()) for h, g in dev.groupby("hemicycle")}
COUNTS = {h: len(EVENTS[h][0]) for h in DEV_HEMICYCLES}
CYCLE_UNITS = {c: [h for h in DEV_HEMICYCLES if int(h[:-1]) == c] for c in DEV_CYCLES}

print(f"shared drift timescale mu_b = {alignment['mu_b']:.3f} +/- {alignment['mu_b_se']:.3f} yr "
      f"(Week 3: 8.024 +/- 0.109)")
print(f"s spans {dev['s'].min():+.3f} to {dev['s'].max():+.3f} · events per hemicycle "
      f"{min(COUNTS.values())} to {max(COUNTS.values())}")

In [ ]:
# Given: the loss, unchanged all program long, and Week 2's time-rescaling machinery.
def softplus(x):
    with np.errstate(invalid="ignore"):
        return np.logaddexp(0.0, np.asarray(x, dtype=float))


def integrate_2d(values, mesh=MESH_2D):
    """Double trapezoid integral of `values`, laid out as (latitude, time), over the mesh."""
    s_mesh, lam_mesh = mesh
    return float(trapezoid(trapezoid(values, lam_mesh, axis=0), s_mesh))


def log_likelihood_2d(event_lat, event_s, intensity_fn, mesh=MESH_2D):
    """log L = sum_i log Lambda(|lat_i|, s_i) - double integral of Lambda over the mesh."""
    s_mesh, lam_mesh = mesh
    at_events = np.asarray(intensity_fn(np.asarray(event_lat, dtype=float),
                                        np.asarray(event_s, dtype=float)), dtype=float)
    if not np.all(np.isfinite(at_events)) or np.any(at_events <= 0.0):
        return -np.inf
    on_mesh = intensity_fn(lam_mesh[:, None], s_mesh[None, :])
    total = float(np.sum(np.log(at_events)) - integrate_2d(on_mesh, mesh))
    return total if np.isfinite(total) else -np.inf


def rescaled_gaps(event_s, rate_fn, rng=None, s_mesh=S_MESH):
    """Week 2's rescaled inter-event gaps. Under a correct rate they are iid Exp(1).

    `rng` breaks the catalog's whole-day ties by jittering each event inside its own day. The tie policy
    is stated rather than silently applied, and every model this week gets the identical jitter draw.
    """
    event_s = np.sort(np.asarray(event_s, dtype=float))
    if rng is not None:
        event_s = np.sort(event_s + rng.uniform(0.0, ONE_DAY, len(event_s)))
    rate_on_mesh = np.asarray(rate_fn(s_mesh), dtype=float)
    compensator = np.concatenate([[0.0], cumulative_trapezoid(rate_on_mesh, s_mesh)])
    return np.diff(np.concatenate([[0.0], np.interp(event_s, s_mesh, compensator)]))


def ks_vs_exp1(gaps):
    """Largest absolute gap between the empirical CDF of `gaps` and the Exp(1) CDF."""
    sorted_gaps = np.sort(np.asarray(gaps, dtype=float))
    n = len(sorted_gaps)
    model_cdf = 1.0 - np.exp(-sorted_gaps)
    above = np.arange(1, n + 1) / n - model_cdf
    below = model_cdf - np.arange(0, n) / n
    return float(np.max(np.maximum(above, below)))


model_10 = ButterflAIModel()
print("carried forward: log_likelihood_2d · integrate_2d · softplus · rescaled_gaps · ks_vs_exp1")

AMP_SERIES = pd.Series(AREA_AMPLITUDE).loc[DEV_HEMICYCLES]
AMP_NORM = plt.Normalize(AMP_SERIES.min(), AMP_SERIES.max())
AMP_CMAP = plt.cm.viridis


def amplitude_colour(label):
    """One wing's colour, fixed by its area-record amplitude. Used by every figure in this notebook."""
    return AMP_CMAP(AMP_NORM(AREA_AMPLITUDE[label]))

In [ ]:
# Given: the bookkeeping that lets every library declare its own parameters.
#
# Positive parameters are optimized as logarithms, a shape parameter that must exceed 1 as log(k - 1)
# exactly as in Weeks 2 and 3, and anything of either sign directly. Each library entry names its own
# transforms and the fitter reads them off rather than knowing anything about the model.

def to_nat(z, kind):
    """Unconstrained real -> the parameter the model uses."""
    if kind == "log":
        return float(np.exp(z))
    if kind == "logm1":
        return 1.0 + float(np.exp(z))
    return float(z)


def to_unc(v, kind):
    """The parameter the model uses -> an unconstrained real."""
    if kind == "log":
        return float(np.log(v))
    if kind == "logm1":
        return float(np.log(max(v - 1.0, 1e-6)))
    return float(v)


print("transforms: log (positive) · logm1 (greater than one) · real (either sign)")

In [ ]:
def _gamma(x, p):
    """Week 3's profile: a gamma shape with its onset at the window start, peak at (k-1)*theta."""
    k, theta = p
    peak = (k - 1.0) * theta
    live = x > 0.0
    with np.errstate(over="ignore", invalid="ignore", divide="ignore"):
        return np.where(live, (np.where(live, x, 1.0) / peak) ** (k - 1.0)
                        * np.exp(-(x - peak) / theta), 0.0)


def _lognormal(x, p):
    """A profile that is Gaussian in log time: a sharper rise and a longer tail than a gamma."""
    x_p, sigma = p
    with np.errstate(divide="ignore", invalid="ignore"):
        return np.where(x > 0.0, np.exp(-(np.log(np.where(x > 0.0, x, 1.0)) - np.log(x_p)) ** 2
                                        / (2.0 * sigma ** 2)), 0.0)


def _weibull(x, p):
    """A Weibull profile: the onset is a power of time and the decay is a stretched exponential."""
    k, lam = p
    x_p = lam * ((k - 1.0) / k) ** (1.0 / k)
    live = x > 0.0
    with np.errstate(over="ignore", invalid="ignore", divide="ignore"):
        return np.where(live, (np.where(live, x, 1.0) / x_p) ** (k - 1.0)
                        * np.exp(-((np.where(live, x, 1.0) / lam) ** k - (x_p / lam) ** k)), 0.0)


def _two_sided(x, p):
    """Two Gaussian half-profiles joined at the peak: growth and collapse get their own timescales."""
    x_p, s_rise, s_fall = p
    sigma = np.where(x < x_p, s_rise, s_fall)
    return np.where(x >= 0.0, np.exp(-(x - x_p) ** 2 / (2.0 * sigma ** 2)), 0.0)


def _gaussian(x, p):
    """A symmetric Gaussian in time. A control: wings are not symmetric, so this one should lose."""
    x_p, sigma = p
    return np.where(x >= 0.0, np.exp(-(x - x_p) ** 2 / (2.0 * sigma ** 2)), 0.0)


SHAPES = {
    "gamma":     dict(fn=_gamma,     transform=("logm1", "log"),       start=(6.0, 0.9),
                      names=("k", "theta"), rise=0),
    "lognormal": dict(fn=_lognormal, transform=("log", "log"),         start=(5.0, 0.55),
                      names=("x_p", "sigma"), rise=1),
    "weibull":   dict(fn=_weibull,   transform=("logm1", "log"),       start=(2.2, 6.5),
                      names=("k", "lambda"), rise=0),
    "two_sided": dict(fn=_two_sided, transform=("log", "log", "log"),  start=(5.3, 1.8, 2.6),
                      names=("x_p", "sigma_rise", "sigma_fall"), rise=1),
    "gaussian":  dict(fn=_gaussian,  transform=("log", "log"),         start=(5.3, 2.2),
                      names=("x_p", "sigma"), rise=1),
}

In [ ]:
POOLINGS = {
    "shared":     dict(assign=lambda n, rise: ["shared"] * n,
                       note="Week 3: one shape for all 18 wings"),
    "hemisphere": dict(assign=lambda n, rise: ["hemi"] * n,
                       note="one shape per hemisphere, nine wings each"),
    "wing_rise":  dict(assign=lambda n, rise: ["wing" if i == rise else "shared" for i in range(n)],
                       note="the rise is per wing, everything else shared"),
    "wing_all":   dict(assign=lambda n, rise: ["wing"] * n,
                       note="every shape parameter per wing — a bound, not a prediction"),
}


def is_conditional(assignment):
    """True if a held-out wing would have to supply one of its own shape parameters to be scored."""
    return "wing" in assignment

In [ ]:
def _w_constant(mu, a_hc, p, a_ref):
    """Week 3's reference: one width everywhere."""
    return p[0]


def _w_amplitude(mu, a_hc, p, a_ref):
    """The belt is wider in stronger cycles."""
    return p[0] * (a_hc / a_ref) ** p[1]


def _w_latitude(mu, a_hc, p, a_ref):
    """The belt is wider further from the equator, which is 1.0's sigma(mu) with one slope."""
    with np.errstate(invalid="ignore"):
        w = p[0] + p[1] * np.asarray(mu, dtype=float)
    return np.where(w > 0.05, w, np.nan)


def _w_piecewise(mu, a_hc, p, a_ref):
    """1.0's sigma(mu) in full: one slope equatorward of the knee, another poleward, continuous at it."""
    mu = np.asarray(mu, dtype=float)
    w_knee = p[0] + p[1] * MU_REF
    with np.errstate(invalid="ignore"):
        w = np.where(mu <= MU_REF, p[0] + p[1] * mu, w_knee + p[2] * (mu - MU_REF))
    return np.where(w > 0.05, w, np.nan)


def _w_combo(mu, a_hc, p, a_ref):
    """Both mechanisms at once: a power law in amplitude plus a linear term in latitude."""
    with np.errstate(invalid="ignore"):
        w = p[0] * (a_hc / a_ref) ** p[1] + p[2] * np.asarray(mu, dtype=float)
    return np.where(w > 0.05, w, np.nan)


WIDTHS = {
    "constant":  dict(fn=_w_constant,  transform=("log",),                start=(6.5,),
                      names=("w0",)),
    "amplitude": dict(fn=_w_amplitude, transform=("log", "real"),         start=(6.1, 0.49),
                      names=("w0", "p")),
    "latitude":  dict(fn=_w_latitude,  transform=("log", "real"),         start=(4.8, 0.11),
                      names=("w0", "w1")),
    "piecewise": dict(fn=_w_piecewise, transform=("log", "real", "real"), start=(4.8, 0.11, 0.11),
                      names=("w0", "w1_eq", "w1_pole")),
    "combo":     dict(fn=_w_combo,     transform=("log", "real", "real"), start=(6.0, 0.45, 0.04),
                      names=("w0", "p", "w1")),
}

In [ ]:
EMISSIONS = {
    "threshold": dict(fn=lambda field, c: c * softplus(field - B_CRIT) ** ALPHA,
                      sup=lambda a_hc, c: c * softplus(a_hc - B_CRIT) ** ALPHA,
                      transform=(), start=(), names=()),
    "direct":    dict(fn=lambda field, c: c * np.clip(field, 0.0, None),
                      sup=lambda a_hc, c: c * a_hc,
                      transform=(), start=(), names=()),
}


def draw_surface(ax, intensity_fn, title, events=None, mesh=MESH_2D):
    """The intensity surface itself: Lambda over latitude and time, with its events if you have them."""
    s_mesh, lam_mesh = mesh
    surface = intensity_fn(lam_mesh[:, None], s_mesh[None, :])
    drawn = ax.pcolormesh(s_mesh, lam_mesh, surface, cmap="magma", shading="auto")
    if events is not None:
        ax.scatter(events[1], events[0], s=2.6, c="white", alpha=0.45, linewidths=0)
    ax.set_xlim(s_mesh[0], s_mesh[-1])
    ax.set_ylim(lam_mesh[0], lam_mesh[-1])
    ax.set_xlabel("s [yr]")
    ax.set_ylabel(r"$|\lambda|$ [deg]")
    ax.set_title(title, fontsize=10)
    ax.grid(False)
    return drawn

In [ ]:
# Given: the three stages that ship with one value each. Each is a complete, working piece of the model —
# and each is a stage you could turn into a knob in Task 36 by adding a second value beside it.
#
# `profile` is the shape of the belt across latitude. `gate` is a multiplicative factor on the field,
# which is where something like 1.0's hard cutoff at low latitude would live. Neither has any fitted
# parameter as shipped, so the baseline model is exactly Week 3's.
def _p_gaussian(offset, w, p):
    """The belt is Gaussian across latitude: `offset` is |lat| - mu(s) and `w` is the belt width."""
    with np.errstate(invalid="ignore"):
        return np.exp(-offset ** 2 / (2.0 * np.asarray(w, dtype=float) ** 2))


PROFILES = {
    "gaussian": dict(fn=_p_gaussian, transform=(), start=(), names=()),
}


def _g_none(mu, s, a_hc, p, x=None):
    """No gate: the field is whatever the belt says it is, everywhere in the domain."""
    return 1.0


GATES = {
    "none": dict(fn=_g_none, transform=(), start=(), names=()),
}


def _d_exponential(s, p):
    """Week 3's drift track: the belt's centre falls exponentially towards the equator."""
    return p[0] * np.exp(-np.asarray(s, dtype=float) / p[1])


DRIFTS = {
    "exponential": dict(fn=_d_exponential, transform=("log", "log"), start=(14.6, 7.2),
                        names=("mu_a", "mu_b")),
}

# ── the catalog, and the specs you build out of it ──
# ── the four stages the Week 4 champion needs, carried across from the knob-proposal work ──
def _shift_none(a_hc, a_ref, p):
    """No shift: the envelope sits where the shared time profile puts it."""
    return 0.0


def _shift_from_amplitude(a_hc, a_ref, p):
    """The envelope is translated in time by an amount set by how strong the wing is."""
    return p[0] + p[1] * np.log(a_hc / a_ref)


def _shift_from_amplitude_1p(a_hc, a_ref, p):
    """The same translation with the constant dropped: only the amplitude term is left.

    `c0` in the two-parameter form was a free constant offset of the envelope in time — the same
    direction in model space as the gamma's own peak, which sits at `(k - 1) * theta`. It was never
    identified: the Week 4 champion fitted `c0 = +0.19 yr` against `c1 = -2.35`, and pinning it to zero
    and refitting gives a model that is no worse on one parameter fewer. `c1` stays, and it carries the
    physics — it is negative, so a stronger hemicycle emerges *earlier*.
    """
    return p[0] * np.log(a_hc / a_ref)


SHIFTS = {
    "none":              dict(fn=_shift_none, transform=(), start=(), names=()),
    "from_amplitude":    dict(fn=_shift_from_amplitude, transform=("real", "real"),
                              start=(0.0, 0.0), names=("c0", "c1")),
    # both forms stay in the catalog so the comparison stays visible, exactly as `tanh_mu` and `tanh_x`
    # both do. The one-parameter form is what the champion now uses.
    "from_amplitude_1p": dict(fn=_shift_from_amplitude_1p, transform=("real",),
                              start=(0.0,), names=("c1",)),
}


def _p_asymmetric(offset, w, p):
    """A latitude profile with different widths on the two flanks. At r = 1 it is the Gaussian."""
    sigma = np.where(np.asarray(offset, dtype=float) > 0.0, np.asarray(w, dtype=float) * p[0],
                     np.asarray(w, dtype=float))
    with np.errstate(invalid="ignore"):
        return np.exp(-np.asarray(offset, dtype=float) ** 2 / (2.0 * sigma ** 2))


PROFILES["asymmetric"] = dict(fn=_p_asymmetric, transform=("log",), start=(1.3,), names=("r",))

# the instantaneous width is the amplitude rule applied to a_hc*g(s) rather than to the wing's peak a_hc:
# same functional form, different argument. The resolver reads the flag and supplies it.
WIDTHS["instantaneous"] = dict(fn=_w_amplitude, transform=("log", "real"), start=(6.1, 0.30),
                               names=("w0", "p"), instantaneous=True)


def _g_tanh_x(mu, s, a_hc, p, x=None):
    """A soft late cutoff that terminates the wing, pinned to envelope time rather than to `s`.

    `x = s - tau_h - S_A`, and `tau_h` is already a function of the wing's amplitude, so the cutoff
    moves with amplitude at no extra parameter: weak wings run long and are cut late, strong wings are
    cut early. Bounded in [0, 1], so the analytic supremum of Lambda is untouched.
    """
    x_cut, steepness = float(p[0]), float(p[1])
    return np.clip(0.5 * (1.0 + np.tanh((x_cut - np.asarray(x, dtype=float)) / steepness)), 0.0, 1.0)


# the cutoff came from a Week 4 knob proposal, pinned there to the belt's latitude mu(s). Because mu(s)
# is shared across wings, that is one fixed instant in `s` for every wing, weak or strong; re-pinned to
# envelope time it rides the amplitude-driven shift instead, for the same two parameters.
GATES["tanh_x"] = dict(fn=_g_tanh_x, transform=("log", "log"), start=(13.0, 0.85),
                       names=("x_cut", "steepness"))

STAGE_ORDER = ("shape", "shift", "drift", "profile", "width", "gate", "emission")
KNOB_REGISTRY = {"shape": SHAPES, "pooling": POOLINGS, "shift": SHIFTS, "drift": DRIFTS,
                 "profile": PROFILES, "width": WIDTHS, "gate": GATES, "emission": EMISSIONS}
BASELINE_SPEC = dict(shape="gamma", pooling="shared", shift="none", drift="exponential",
                     profile="gaussian", width="constant", gate="none", emission="threshold")
# The Week 4 search ended on `shift="from_amplitude"`, two parameters. A student group arrived at the same
# model from a literature review with *one* amplitude coefficient, which prompted us to check ours: pinning
# the constant to zero and refitting gives -29.0226 on 29 parameters against -28.9950 on 30, a tie in score
# and a parameter cheaper. The champion carries the one-parameter form from here on.
CHAMPION_SPEC = dict(BASELINE_SPEC, shift="from_amplitude_1p", profile="asymmetric",
                     width="instantaneous", gate="tanh_x")



def spec_of(**knobs):
    """A spec is the baseline with the knobs you name changed. Rejects a stage or value that is not there."""
    spec = dict(BASELINE_SPEC)
    for stage, value in knobs.items():
        if stage not in KNOB_REGISTRY:
            raise ValueError(f"no stage called '{stage}'")
        if value not in KNOB_REGISTRY[stage]:
            raise ValueError(f"stage '{stage}' has no value '{value}'")
        spec[stage] = value
    return spec


def n_changed(spec):
    """How many knobs this spec moves away from the baseline. One means it is an ablation."""
    return sum(1 for stage in spec if stage in BASELINE_SPEC and spec[stage] != BASELINE_SPEC[stage])


def variant_name(spec):
    """A filesystem-safe name generated from the spec: only the knobs that differ from the baseline."""
    changed = [f"{stage}-{spec[stage]}" for stage in sorted(spec)
               if stage in BASELINE_SPEC and spec[stage] != BASELINE_SPEC[stage]]
    return "baseline" if not changed else "__".join(changed)




print("catalog: " + " · ".join(f"{s} {len(v)}" for s, v in KNOB_REGISTRY.items()))
print(f"the two parametric models this week needs: {variant_name(BASELINE_SPEC)} · "
      f"{variant_name(CHAMPION_SPEC)}")

In [ ]:
# Given: the resolver. `Model` turns a spec into a function of latitude and time purely by looking
# values up in the libraries — not one `if` asks which variant is running, which is what lets a knob
# invented in Task 36 work without editing this cell. It also lays out the parameter vector: shared
# shape parameters first (twice over for a per-hemisphere plan), then every other stage in a fixed
# order, with anything assigned per wing travelling with that wing instead.
class Model:
    """Everything the fitter needs, assembled from a spec. The fitter never looks at the spec."""

    def __init__(self, spec):
        self.spec = dict(spec)
        # `.get` so that a spec cached before a stage existed still resolves to the old model
        self.stage = {name: KNOB_REGISTRY[name][spec.get(name, BASELINE_SPEC[name])]
                      for name in STAGE_ORDER}
        self.shape = self.stage["shape"]
        self.assign = POOLINGS[spec.get("pooling", BASELINE_SPEC["pooling"])]["assign"](len(self.shape["start"]), self.shape["rise"])
        self.n_shape = len(self.shape["start"])
        self.wing_idx = [i for i, how in enumerate(self.assign) if how == "wing"]
        self.conditional = is_conditional(self.assign)

        self.slots, cursor = [], 0
        for i, how in enumerate(self.assign):
            if how == "shared":
                self.slots.append((i, "shared", cursor)); cursor += 1
            elif how == "hemi":
                self.slots.append((i, "hemi", cursor)); cursor += 2
        self.at = {}
        for name in STAGE_ORDER[1:]:
            self.at[name] = cursor
            cursor += len(self.stage[name]["start"])
        self.n_q = cursor
        # every fitted number: the shared vector, one amplitude per wing, any per-wing shape parameter, c
        self.n_params = self.n_q + (len(self.wing_idx) + 1) * len(FIT_WINGS) + 1

    def start_q(self):
        q = np.zeros(self.n_q)
        for i, how, at in self.slots:
            z = to_unc(self.shape["start"][i], self.shape["transform"][i])
            q[at] = z
            if how == "hemi":
                q[at + 1] = z
        for name in STAGE_ORDER[1:]:
            entry = self.stage[name]
            for j, v in enumerate(entry["start"]):
                q[self.at[name] + j] = to_unc(v, entry["transform"][j])
        return q

    def start_wing(self):
        return np.array([to_unc(self.shape["start"][i], self.shape["transform"][i])
                         for i in self.wing_idx])

    def unpack(self, q, c=1.0, a_ref=1.0):
        """Unconstrained vector -> the natural parameters, grouped by stage."""
        shape_shared = {}
        for i, how, at in self.slots:
            kind = self.shape["transform"][i]
            shape_shared[i] = (to_nat(q[at], kind) if how == "shared"
                               else (to_nat(q[at], kind), to_nat(q[at + 1], kind)))
        stages = {name: [to_nat(q[self.at[name] + j], self.stage[name]["transform"][j])
                         for j in range(len(self.stage[name]["start"]))]
                  for name in STAGE_ORDER[1:]}
        return dict(shape_shared=shape_shared, stages=stages, c=c, a_ref=a_ref)

    def shape_params(self, shared, label, wing_z=None):
        """One hemicycle's full shape vector, assembled from the shared, hemispheric and per-wing pieces."""
        out = [None] * self.n_shape
        for i, how, _ in self.slots:
            out[i] = (shared["shape_shared"][i] if how == "shared"
                      else shared["shape_shared"][i][0 if label[-1] == "N" else 1])
        for j, i in enumerate(self.wing_idx):
            out[i] = to_nat(wing_z[j], self.shape["transform"][i])
        return out

    def intensity(self, label, a_hc, shared, wing_z=None):
        """Assemble Lambda(|lat|, s) for one hemicycle. Every stage is a lookup, never an if-statement."""
        shape_p = self.shape_params(shared, label, wing_z)
        p, c, a_ref = shared["stages"], shared["c"], shared["a_ref"]
        shape_fn = self.shape["fn"]
        drift_fn, profile_fn = self.stage["drift"]["fn"], self.stage["profile"]["fn"]
        width_fn, gate_fn = self.stage["width"]["fn"], self.stage["gate"]["fn"]
        emit, shift_fn = self.stage["emission"]["fn"], self.stage["shift"]["fn"]
        instantaneous = bool(self.stage["width"].get("instantaneous"))
        tau = shift_fn(a_hc, a_ref, p["shift"])

        def intensity_fn(abs_lat, s):
            s = np.asarray(s, dtype=float)
            g = shape_fn(s - tau - S_A, shape_p)
            mu = drift_fn(s, p["drift"])
            # a width rule may be handed the instantaneous field a_hc*g(s) instead of the wing's peak
            # a_hc. Clip it: the field vanishes at the window edges and a zero width divides by zero.
            w = width_fn(mu, np.clip(a_hc * g, a_ref * 1e-4, None) if instantaneous else a_hc,
                         p["width"], a_ref)
            offset = np.asarray(abs_lat, dtype=float) - mu
            # a gate may pin itself to the belt's latitude `mu`, or to envelope time `x`, which
            # already rides the amplitude-driven shift. Both are handed over; it uses what it needs.
            field = (a_hc * g * profile_fn(offset, w, p["profile"])
                     * gate_fn(mu, s, a_hc, p["gate"], s - tau - S_A))
            return emit(field, c)

        return intensity_fn

    def sup(self, a_hc, shared):
        """The exact supremum of Lambda: at the peak of g, on the drift track, where the profile is 1."""
        return self.stage["emission"]["sup"](a_hc, shared["c"])


print(f"resolver ready · baseline {Model(BASELINE_SPEC).n_q} shared parameters · "
      f"champion {Model(CHAMPION_SPEC).n_q}")

In [ ]:
# Given: the pooled fit, generalized from Week 3's. The same block-coordinate loop — shared shape, then
# each wing's own parameters, then the closed-form `c` — with the shared block whatever the resolver
# says, a small Nelder-Mead per wing whenever the pooling plan frees a shape parameter there, and
# `a_ref` recomputed from the wings being fitted so the width rules stay honest inside a fold. `c` keeps
# its closed form, `c = n_total / ∫∫Λ` at `c` = 1, so it never reaches an optimizer.
AMPLITUDE_BRACKET = (np.log(1.05), np.log(1.0e7))


def fit_per_wing(event_lat, event_s, label, a_hc, wing_z, model, shared, mesh=MESH_2D):
    """One hemicycle's own parameters: its amplitude, plus any shape parameter the plan frees per wing."""
    def nll(vector):
        value = log_likelihood_2d(event_lat, event_s,
                                  model.intensity(label, float(np.exp(vector[0])), shared, vector[1:]),
                                  mesh)
        return -value if np.isfinite(value) else 1e12

    if not model.wing_idx:
        out = minimize_scalar(lambda z: nll(np.array([z])), bounds=AMPLITUDE_BRACKET,
                              method="bounded", options=dict(xatol=1e-4))
        return float(np.exp(out.x)), np.array([])
    out = minimize(nll, np.concatenate([[np.log(a_hc)], wing_z]), method="Nelder-Mead",
                   options=dict(maxiter=250, maxfev=250, xatol=1e-4, fatol=1e-4))
    return float(np.exp(out.x[0])), out.x[1:]


def fit_pooled_v2(events, labels, model, q_start=None, amp_start=None, wing_start=None,
                  sweeps=3, mesh=MESH_2D):
    """Alternate: shared block -> per-wing block -> closed-form c. Week 3's fitter, over any spec."""
    labels = list(labels)
    n_total = sum(len(events[h][0]) for h in labels)
    q = model.start_q() if q_start is None else np.array(q_start, dtype=float)
    amps = {h: 2.0e4 for h in labels} if amp_start is None else {h: amp_start[h] for h in labels}
    wing = ({h: model.start_wing() for h in labels} if wing_start is None
            else {h: np.array(wing_start[h], dtype=float) for h in labels})
    lam_col, s_row = mesh[1][:, None], mesh[0][None, :]

    def geometric_mean(values):
        return float(np.exp(np.mean(np.log(list(values)))))

    def solve_c(q, amps, wing, a_ref):
        unit = model.unpack(q, 1.0, a_ref)
        total = sum(integrate_2d(model.intensity(h, amps[h], unit, wing[h])(lam_col, s_row), mesh)
                    for h in labels)
        return n_total / total if total > 0 else np.inf

    def shape_nll(q_try, amps, wing, c, a_ref):
        shared = model.unpack(q_try, c, a_ref)
        total = 0.0
        for h in labels:
            value = log_likelihood_2d(*events[h], model.intensity(h, amps[h], shared, wing[h]), mesh)
            if not np.isfinite(value):
                return 1e12
            total += value
        return -total

    a_ref = geometric_mean(amps.values())
    c = solve_c(q, amps, wing, a_ref)
    for _ in range(sweeps):
        q = minimize(shape_nll, q, args=(amps, wing, c, a_ref), method="L-BFGS-B",
                     options=dict(maxiter=300)).x
        shared = model.unpack(q, c, a_ref)
        for h in labels:
            amps[h], wing[h] = fit_per_wing(*events[h], h, amps[h], wing[h], model, shared, mesh)
        a_ref = geometric_mean(amps.values())
        c = solve_c(q, amps, wing, a_ref)
    return dict(q=q, shared=model.unpack(q, c, a_ref), amplitudes=amps, wing=wing, labels=labels,
                nll=shape_nll(q, amps, wing, c, a_ref), a_ref=a_ref)


# The reference fit, which every ablation is compared against and every fold warm-starts from.
started = time.time()
BASE_FIT = fit_pooled_v2(EVENTS, FIT_WINGS, Model(BASELINE_SPEC), sweeps=3)
# Sign convention, which is Week 3's: `nll` is the negative log-likelihood, so a *more negative* score
# per hemicycle-year is a better fit. Lower is better everywhere in this notebook.
print(f"baseline refitted in {time.time() - started:.1f} s · "
      f"{BASE_FIT['nll'] / (len(FIT_WINGS) * WINDOW_YEARS):.4f} nats per hemicycle-year in sample")
if len(FIT_WINGS) == 18:
    print("Week 3 reported -26.598 for the same model, so the generalized fitter reproduces it.")

In [ ]:
# Given: the fingerprint, and the three pieces the fold loop needs.
AMPLITUDE_SOURCES = ("area record", "own events")
HEADLINE = AMPLITUDE_SOURCES[0]


def protocol_fingerprint():
    """Everything a score depends on besides the model. Two rows are comparable only if these agree."""
    recipe = dict(mesh=[N_S_2D, N_LAM_2D], domain=[0.0, LAM_MAX], window=[S_A, S_B],
                  profile=PROFILE, wings=list(FIT_WINGS), folds=list(FOLD_CYCLES),
                  withheld=list(WITHHELD_CYCLES),
                  analysis_set=[CYCLE_MIN, CYCLE_MAX, PEAK_AREA_MIN],
                  gauge=[B_CRIT, ALPHA, MU_REF], headline_amplitude=HEADLINE,
                  # Week 06's items. Week 05 stored the fingerprint and never checked it, which is
                  # what let a stale cache travel; `assert_fingerprint` below checks it on lookup.
                  tau_days=list(TAU_DAYS), ell_deg=list(ELL_DEG),
                  history_scope="same-hemisphere, development-set only",
                  strictly_past=True, compensator="analytic")
    text = json.dumps(recipe, sort_keys=True)
    return dict(recipe=recipe, digest=hashlib.sha1(text.encode()).hexdigest()[:12])


def fit_wing_only(event_lat, event_s, label, a_hc, wing_z, model, shared, mesh=MESH_2D):
    """Fit only the per-wing shape parameters, with the amplitude held fixed. Empty for a shared plan."""
    if not model.wing_idx:
        return np.array([])

    def nll(vector):
        value = log_likelihood_2d(event_lat, event_s, model.intensity(label, a_hc, shared, vector), mesh)
        return -value if np.isfinite(value) else 1e12

    return minimize(nll, wing_z, method="Nelder-Mead",
                    options=dict(maxiter=300, maxfev=300, xatol=1e-4, fatol=1e-4)).x


def time_marginal(intensity_fn, s_values=S_MESH, lam_mesh=LAM_2D):
    """The marginal intensity: lambda(s) = integral over latitude of Lambda(|lat|, s), events per year."""
    surface = intensity_fn(lam_mesh[:, None], np.asarray(s_values, dtype=float)[None, :])
    return trapezoid(surface, lam_mesh, axis=0)


print("fingerprint", protocol_fingerprint()["digest"], "·", protocol_fingerprint()["recipe"])

In [ ]:
# Given: the fold loop. One row per held-out hemicycle, scored two ways, with both metrics each way.
def loco_v2(raw_events, cycle_units, t0_start, area_amplitude, model, labels,
            q_start=None, amp_start=None, sweeps=2, mesh=MESH_2D, verbose=False,
            score_cycles=None):
    """Fit on every cycle but one, score the one left out. Returns the rows and each fold's parameters.

    `cycle_units` is every cycle available to fit on; `score_cycles` is the subset actually held out and
    scored, which is how a profile runs three folds rather than nine without changing what it trains on.
    """
    rows, fold_params = [], {}
    order = {h: j for j, h in enumerate(labels)}
    for held_out in (sorted(cycle_units) if score_cycles is None else sorted(score_cycles)):
        train = [h for c in sorted(cycle_units) if c != held_out for h in cycle_units[c]]

        # 1 · the clock, on the training hemicycles only
        t0_train = np.array([t0_start[h] for h in train])
        index = np.concatenate([np.full(len(raw_events[h][0]), j) for j, h in enumerate(train)])
        fold_alignment = fit_alignment(np.concatenate([raw_events[h][1] for h in train]),
                                       np.concatenate([raw_events[h][0] for h in train]),
                                       index, t0_train)
        events_train = {h: (raw_events[h][0],
                            raw_events[h][1] - (t0_train[j] + fold_alignment["delta"][j]))
                        for j, h in enumerate(train)}

        # 2 · the model, on the training hemicycles only, with a_ref recomputed from them
        fold_fit = fit_pooled_v2(events_train, train, model, q_start=q_start, sweeps=sweeps,
                                 amp_start=({h: amp_start[h] for h in train} if amp_start else None),
                                 mesh=mesh)
        shared = fold_fit["shared"]
        intercept, slope = np.polyfit(np.log([area_amplitude[h] for h in train]),
                                      np.log([fold_fit["amplitudes"][h] for h in train]), 1)[::-1]
        fold_params[str(held_out)] = dict(
            q=[float(v) for v in fold_fit["q"]], c=float(shared["c"]), a_ref=float(shared["a_ref"]),
            mu_b_clock=float(fold_alignment["mu_b"]), amp_map=[float(intercept), float(slope)],
            train=train)

        # 3 · score each held-out hemicycle on its own clock, two ways
        for label in cycle_units[held_out]:
            lat_held, t_held = raw_events[label]
            shift = align_one(t_held, lat_held, t0_start[label], fold_alignment["mu_b"])
            s_held = t_held - (t0_start[label] + shift)
            inside = (s_held > mesh[0][0]) & (s_held < mesh[0][-1])
            lat_in, s_in = lat_held[inside], s_held[inside]
            row = dict(hemicycle=label, cycle=int(held_out), n=int(inside.sum()),
                       outside=int((~inside).sum()), delta=float(shift))
            a_map = float(np.exp(intercept + slope * np.log(area_amplitude[label])))

            for source in AMPLITUDE_SOURCES:
                if source == "area record":
                    a_hc = a_map
                    wing_z = fit_wing_only(lat_in, s_in, label, a_hc, model.start_wing(),
                                           model, shared, mesh)
                else:
                    a_hc, wing_z = fit_per_wing(lat_in, s_in, label, a_map, model.start_wing(),
                                                model, shared, mesh)
                intensity_fn = model.intensity(label, a_hc, shared, wing_z)
                value = log_likelihood_2d(lat_in, s_in, intensity_fn, mesh)
                marginal = time_marginal(intensity_fn)
                gaps = rescaled_gaps(np.sort(s_in), lambda s: np.interp(s, S_MESH, marginal),
                                     rng=np.random.default_rng(SEED + 1000 + order[label]))
                row[f"a ({source})"] = float(a_hc)
                row[f"nll/yr ({source})"] = float(-value / (mesh[0][-1] - mesh[0][0]))
                row[f"predicted ({source})"] = float(integrate_2d(
                    intensity_fn(mesh[1][:, None], mesh[0][None, :]), mesh))
                row[f"ks ({source})"] = float(ks_vs_exp1(gaps))
                row[f"ks_p ({source})"] = float(kstest(gaps, "expon").pvalue)
                if model.wing_idx:
                    row[f"wing ({source})"] = [float(v) for v in wing_z]
            rows.append(row)
        if verbose:
            print(f"      cycle {held_out}: trained on {len(train)} hemicycles, "
                  f"scored {cycle_units[held_out]}", flush=True)
    return pd.DataFrame(rows).set_index("hemicycle"), fold_params


print(f"loco_v2 ready · scoring cycles {FOLD_CYCLES} · both amplitude sources · "
      f"both metrics · each fold's parameters kept")

In [ ]:
def run_variant(spec, question="", sweeps_in=3, sweeps_fold=2, mesh=MESH_2D, verbose=False):
    """Fit, score and cache one variant. Skips the work entirely if its JSON is already there."""
    name = variant_name(spec)
    path = CACHE / f"{name}.json"
    if path.exists():
        with open(path) as handle:
            return json.load(handle)

    assert not set(FIT_UNITS) & set(WITHHELD_CYCLES), "a withheld cycle reached the runner"
    model = Model(spec)
    started = time.time()
    in_fit = fit_pooled_v2(EVENTS, FIT_WINGS, model, sweeps=sweeps_in,
                           amp_start=BASE_FIT["amplitudes"], mesh=mesh)
    held, fold_params = loco_v2(RAW_EVENTS, FIT_UNITS, T0_START, AREA_AMPLITUDE, model,
                                FIT_WINGS, q_start=in_fit["q"], amp_start=in_fit["amplitudes"],
                                sweeps=sweeps_fold, mesh=mesh, verbose=verbose,
                                score_cycles=FOLD_CYCLES)
    kind = ("bound (unranked)" if model.conditional
            else "ablation" if n_changed(spec) <= 1 else "candidate")
    result = dict(
        name=name, spec=dict(spec), question=question, kind=kind, conditional=model.conditional,
        n_q=model.n_q, n_params=model.n_params,
        in_sample=float(-in_fit["nll"] / (len(FIT_WINGS) * WINDOW_YEARS)),
        shared=dict(shape={str(k): (list(v) if isinstance(v, tuple) else float(v))
                           for k, v in in_fit["shared"]["shape_shared"].items()},
                    stages={k: [float(x) for x in v] for k, v in in_fit["shared"]["stages"].items()},
                    c=float(in_fit["shared"]["c"]), a_ref=float(in_fit["shared"]["a_ref"])),
        amplitudes={h: float(v) for h, v in in_fit["amplitudes"].items()},
        q=[float(v) for v in in_fit["q"]],
        rows=held.reset_index().to_dict("records"), fold_params=fold_params,
        seconds=float(time.time() - started), fingerprint=protocol_fingerprint())
    with open(path, "w") as handle:
        json.dump(result, handle, indent=1)
    return result


def load_cached():
    """Every result in the cache, keyed by variant name. This is the leaderboard's input."""
    out = {}
    for path in sorted(CACHE.glob("*.json")):
        with open(path) as handle:
            result = json.load(handle)
        out[result["name"]] = result
    return out


def rows_of(result):
    """One result's per-hemicycle table, as a DataFrame indexed by hemicycle."""
    return pd.DataFrame(result["rows"]).set_index("hemicycle")


def fold_shared(result, cycle):
    """Rebuild one fold's fitted parameters from the cached numbers alone."""
    model = Model(result["spec"])
    fold = result["fold_params"][str(cycle)]
    return model, model.unpack(np.array(fold["q"]), fold["c"], fold["a_ref"]), fold

In [ ]:
def paired_stats(result, baseline, source=HEADLINE):
    """Variant minus baseline on the same wing. Negative means the variant predicts better."""
    difference = (rows_of(result)[f"nll/yr ({source})"]
                  - rows_of(baseline)[f"nll/yr ({source})"]).dropna()
    n = len(difference)
    return dict(mean=float(difference.mean()),
                se=float(difference.std(ddof=1) / np.sqrt(n)) if n > 1 else np.nan,
                median=float(difference.median()),
                better=int((difference < 0).sum()), n=n,
                differences=difference)


def draw_paired(ax, stats, row, colour):
    """One variant's eighteen paired differences, with the mean and its standard error on top."""
    ax.plot(stats["differences"].to_numpy(), np.full(stats["n"], row), "o", ms=3.6,
            color=colour, alpha=0.55)
    ax.errorbar([stats["mean"]], [row], xerr=[stats["se"]], fmt="D", ms=6,
                color=colour, capsize=3.5, lw=1.4)

In [ ]:
def simulate_wing(intensity_fn, rate_max, rng, mesh=MESH_2D):
    """Events of a 2D inhomogeneous Poisson process, by thinning. Week 3's generator, unchanged."""
    s_mesh, lam_mesh = mesh
    s_lo, s_hi, lat_lo, lat_hi = s_mesh[0], s_mesh[-1], lam_mesh[0], lam_mesh[-1]
    area = (s_hi - s_lo) * (lat_hi - lat_lo)
    n_proposed = rng.poisson(rate_max * area)
    proposed_s = rng.uniform(s_lo, s_hi, n_proposed)
    proposed_lat = rng.uniform(lat_lo, lat_hi, n_proposed)
    acceptance = np.asarray(intensity_fn(proposed_lat, proposed_s), dtype=float) / rate_max
    if np.any(acceptance > 1.0):
        raise ValueError("rate_max does not bound the intensity — thinning is invalid")
    keep = rng.uniform(0.0, 1.0, n_proposed) < acceptance
    return proposed_lat[keep], proposed_s[keep], (proposed_lat, proposed_s), keep


def plausible(result, cycle, seed_offset=400, mesh=MESH_2D):
    """Thin a synthetic wing from the held-out fold's own intensity and compare it with the real one."""
    rng = np.random.default_rng(SEED + seed_offset)
    rows, synthetic = [], {}
    held_rows = rows_of(result)
    for label in FOLD_UNITS[cycle]:
        model, shared, _ = fold_shared(result, cycle)
        row = held_rows.loc[label]
        a_hc = float(row[f"a ({HEADLINE})"])
        wing_z = np.array(row[f"wing ({HEADLINE})"]) if model.wing_idx else None
        intensity_fn = model.intensity(label, a_hc, shared, wing_z)

        bound = model.sup(a_hc, shared)
        on_mesh = float(intensity_fn(mesh[1][:, None], mesh[0][None, :]).max())
        assert bound >= on_mesh - 1e-9, "the analytic bound does not bound the intensity on the mesh"
        lat_sim, s_sim, _, keep = simulate_wing(intensity_fn, bound, rng, mesh)
        synthetic[label] = (lat_sim, s_sim)

        lat_obs, s_obs = EVENTS[label]
        surface = intensity_fn(LAM_2D[:, None], S_2D[None, :])
        weight = trapezoid(surface, S_2D, axis=1)
        total = float(trapezoid(weight, LAM_2D))
        centroid = float(trapezoid(weight * LAM_2D, LAM_2D) / total)
        spread = float(np.sqrt(trapezoid(weight * (LAM_2D - centroid) ** 2, LAM_2D) / total))
        rows.append(dict(hemicycle=label, observed=len(lat_obs), predicted=total, simulated=len(lat_sim),
                         centroid_obs=float(np.mean(lat_obs)), centroid_model=centroid,
                         spread_obs=float(np.std(lat_obs, ddof=1)), spread_model=spread,
                         accepted=float(keep.mean())))
    return pd.DataFrame(rows).set_index("hemicycle"), synthetic


def draw_butterfly(ax, source, title, cycle=None, colour="0.3"):
    """One panel of a butterfly diagram: north upward, south downward, s across, the equator marked."""
    for label in (FOLD_UNITS[cycle] if cycle is not None else sorted(source)):
        sign = 1.0 if label.endswith("N") else -1.0
        lat_here, s_here = source[label]
        ax.scatter(s_here, sign * lat_here, s=3.0, c=colour, alpha=0.45, linewidths=0)
    ax.axhline(0.0, color="k", lw=0.8)
    ax.set_xlim(S_A, S_B)
    ax.set_ylim(-LAM_MAX, LAM_MAX)
    ax.set_xlabel("s [yr]")
    ax.set_ylabel("latitude [deg]")
    ax.set_title(title, fontsize=10)


print("thinning · plausibility · butterfly panels ready")

In [ ]:
# Given: the two parametric models, fitted on this profile's wings and scored on its folds. Cached, so
# this costs a few minutes once and nothing afterwards. The champion's score is the number to beat.
BASELINE_RESULT = run_variant(BASELINE_SPEC, "the Week 3 latent belt, as the reference row")
CHAMPION_RESULT = run_variant(CHAMPION_SPEC, "the best parametric model the Week 4 search found")
PARAMETRIC = {"baseline": BASELINE_RESULT, "champion": CHAMPION_RESULT}


def in_sample_shared(result):
    """Rebuild a cached result's in-sample fit: the resolver, its parameters, and the wing amplitudes."""
    model = Model(result["spec"])
    shared = model.unpack(np.array(result["q"]), result["shared"]["c"], result["shared"]["a_ref"])
    return model, shared, result["amplitudes"]


def held_mean(result, source=HEADLINE):
    """The mean held-out score in nats per hemicycle-year. Lower is better, as everywhere in Phase A."""
    return float(rows_of(result)[f"nll/yr ({source})"].mean())


for name, result in PARAMETRIC.items():
    rows = rows_of(result)
    assert rows["outside"].sum() == 0, "a held-out event fell outside the window after re-alignment"
    print(f"{name:9s} {result['n_params']:3d} fitted numbers · in sample {-result['in_sample']:9.4f} · "
          f"held out {held_mean(result):9.4f} · spread {rows[f'nll/yr ({HEADLINE})'].std(ddof=1):6.2f} · "
          f"{result['seconds']:5.0f} s")
print(f"\nthe gap between the two parametric models: "
      f"{held_mean(CHAMPION_RESULT) - held_mean(BASELINE_RESULT):+.4f} nats per hemicycle-year")

# The reproduction check. Week 5 ran this identical protocol on this identical profile and recorded the
# numbers below, so a mismatch here means something moved that should not have.
WEEK5_FULL = dict(baseline_in=-26.5994, baseline_held=-28.0689,
                  champion_in=-30.3668, champion_held=-35.9179)
if PROFILE == "full":
    print("\nagainst Week 5's record of the same protocol on the same three folds:")
    for name, result, keys in (("baseline", BASELINE_RESULT, ("baseline_in", "baseline_held")),
                               ("champion", CHAMPION_RESULT, ("champion_in", "champion_held"))):
        got_in, got_held = -result["in_sample"], held_mean(result)
        print(f"   {name:9s} in sample {got_in:9.4f} vs {WEEK5_FULL[keys[0]]:9.4f} "
              f"({got_in - WEEK5_FULL[keys[0]]:+.4f}) · held out {got_held:9.4f} vs "
              f"{WEEK5_FULL[keys[1]]:9.4f} ({got_held - WEEK5_FULL[keys[1]]:+.4f})")
    print("   The pooled fit is a block-coordinate loop over a surface that is not convex, so these\n"
          "   reproduce to within the optimizer's path rather than to the last digit. Anything inside\n"
          "   this week's resolution floor is a reproduction; anything outside it is a bug.")

In [ ]:
TORCH_DTYPE = torch.float32        # training: fast, and all a gradient needs
SCORE_DTYPE = torch.float64        # scoring: matches Phase A's numpy path to twelve digits
B_FLOOR = 1.0e-2                   # below a hundredth of critical the emission is flat, so clamp there


def inv_softplus(y):
    """Invert the emission: softplus(x) = y  ->  x. Stable for the large y the fitted belt produces."""
    y = np.asarray(y, dtype=float)
    small = np.log(np.expm1(np.maximum(np.minimum(y, 20.0), 1.0e-12)))
    return np.where(y > 20.0, y + np.log1p(-np.exp(-np.maximum(y, 20.0))), small)


def field_of(intensity_fn, c, abs_lat, s):
    """The latent field `B` behind a parametric intensity. Exact, because the emission is fixed."""
    return inv_softplus(np.asarray(intensity_fn(abs_lat, s), dtype=float) / c) + B_CRIT


def emission_torch(b, log_c):
    """Phase A's emission, in torch: `c · softplus(B − B_crit)^alpha` with alpha = 1."""
    return torch.exp(log_c) * F.softplus(b - B_CRIT)


def nll_from_field(b_events, b_mesh, log_c, n_wings, s_grid, lam_grid):
    """The week's loss, in nats per hemicycle-year. Lower is better, as in every earlier week.

    `b_events` is the field at the observed events, `b_mesh` the field on the frozen mesh laid out
    (wing, latitude, time). The integral term is the double trapezoid, exactly as `integrate_2d`.
    """
    at_events = emission_torch(b_events, log_c)
    on_mesh = emission_torch(b_mesh, log_c)
    integral = torch.trapezoid(torch.trapezoid(on_mesh, lam_grid, dim=-2), s_grid, dim=-1).sum()
    return -(torch.log(at_events).sum() - integral) / (n_wings * WINDOW_YEARS)

In [ ]:
LOG_B_MID, LOG_B_SCALE = 3.0, 7.5  # the field's own scale: log B runs from about -4.6 to about 10.6


class BeltNet(nn.Module):
    """A neural field: (|lat|, s, conditioning) -> log B, in units of the field's own scale."""

    def __init__(self, n_cond=0, width=NET_WIDTH, depth=NET_DEPTH, b_start=1.0e4):
        super().__init__()
        sizes = [2 + n_cond] + [width] * depth
        self.hidden = nn.ModuleList(nn.Linear(a, b) for a, b in zip(sizes, sizes[1:]))
        self.head = nn.Linear(width, 1)
        self.n_cond = n_cond
        with torch.no_grad():
            self.head.weight.mul_(0.1)              # a nearly flat field to begin with
            self.head.bias.fill_((float(np.log(b_start)) - LOG_B_MID) / LOG_B_SCALE)

    @staticmethod
    def coordinates(abs_lat, s):
        """Both coordinates onto [-1, 1], which is the range a tanh network is built for."""
        return torch.stack([abs_lat / LAM_MAX * 2.0 - 1.0,
                            (s - S_A) / (S_B - S_A) * 2.0 - 1.0], dim=-1)

    def forward(self, abs_lat, s, cond=None):
        x = self.coordinates(abs_lat, s)
        if self.n_cond:
            x = torch.cat([x, cond], dim=-1)
        for layer in self.hidden:
            x = torch.tanh(layer(x))
        return self.head(x).squeeze(-1)

    def log_field(self, abs_lat, s, cond=None):
        """The network's output put back on the field's own scale: natural log of `B`."""
        return LOG_B_MID + LOG_B_SCALE * self.forward(abs_lat, s, cond)

    def field(self, abs_lat, s, cond=None):
        """`B` itself, in multiples of critical. Clamped only to keep a bad step from producing inf."""
        return torch.exp(self.log_field(abs_lat, s, cond).clamp(-20.0, 20.0))


def as_field_fn(net, cond=None, dtype=SCORE_DTYPE):
    """Wrap a network as a plain numpy `B(abs_lat, s)`, broadcasting like any other field."""
    frozen = copy.deepcopy(net).to("cpu", dtype).eval()
    cond_row = None if cond is None else torch.as_tensor(np.asarray(cond, float), dtype=dtype)

    def field_fn(abs_lat, s):
        lat_a, s_a = np.broadcast_arrays(np.asarray(abs_lat, dtype=float), np.asarray(s, dtype=float))
        with torch.no_grad():
            lat_t = torch.as_tensor(np.ascontiguousarray(lat_a).ravel(), dtype=dtype)
            s_t = torch.as_tensor(np.ascontiguousarray(s_a).ravel(), dtype=dtype)
            rows = None if cond_row is None else cond_row.expand(lat_t.numel(), cond_row.numel())
            value = frozen.field(lat_t, s_t, rows)
        return value.numpy().reshape(lat_a.shape)

    return field_fn


def as_intensity_fn(net, cond, log_c, dtype=SCORE_DTYPE):
    """The adapter that matters: a trained field, as the `intensity_fn(abs_lat, s)` Phase A already eats."""
    field_fn = as_field_fn(net, cond, dtype)
    c = float(np.exp(float(log_c)))

    def intensity_fn(abs_lat, s):
        return c * np.logaddexp(0.0, field_fn(abs_lat, s) - B_CRIT) ** ALPHA

    return intensity_fn

In [ ]:
# Given: the Lightning plumbing, which is boilerplate and therefore given. Everything here is full-batch:
# one epoch is one optimizer step over every wing and the whole mesh, so a `DataLoader` has nothing to
# iterate over and gets a single dummy row. Real mini-batching would be a Week 7 conversation.
def single_batch():
    """A loader with exactly one empty batch per epoch. The data lives on the module, not in the loader."""
    return DataLoader(TensorDataset(torch.zeros(1, 1)), batch_size=1)


def make_trainer(epochs, val_every=0):
    """A Trainer with every convenience switched off, so that what runs is what we wrote."""
    return L.Trainer(max_epochs=epochs, accelerator="auto", devices=1, logger=False,
                     enable_checkpointing=False, enable_progress_bar=False,
                     enable_model_summary=False, num_sanity_val_steps=0,
                     check_val_every_n_epoch=val_every or epochs + 1)


def mesh_batch(n_wings, cond, dtype=TORCH_DTYPE, device=None):
    """The frozen mesh, broadcast to (wing, latitude, time) — the shape every field evaluation uses."""
    lam = torch.as_tensor(LAM_2D, dtype=dtype, device=device)[None, :, None].expand(
        n_wings, N_LAM_2D, N_S_2D)
    s = torch.as_tensor(S_2D, dtype=dtype, device=device)[None, None, :].expand(
        n_wings, N_LAM_2D, N_S_2D)
    rows = (None if cond is None
            else cond[:, None, None, :].expand(n_wings, N_LAM_2D, N_S_2D, cond.shape[-1]))
    return lam, s, rows


print("Lightning plumbing ready · one epoch = one full-batch step over "
      f"{len(FIT_WINGS)} wings x {N_LAM_2D} x {N_S_2D} mesh nodes")

In [ ]:
class PretrainModule(L.LightningModule):
    """Lightning on a problem whose answer is known: copy a field we already have."""

    def __init__(self, net, target_log_b, cond=None, lr=PRE_LR):
        super().__init__()
        self.net = net
        self.lr = lr
        self.history = []
        self.register_buffer("target", torch.as_tensor(target_log_b, dtype=TORCH_DTYPE))
        self.register_buffer("cond", None if cond is None
                             else torch.as_tensor(cond, dtype=TORCH_DTYPE))

    def training_step(self, batch, batch_index):
        lam, s, rows = mesh_batch(self.target.shape[0], self.cond, device=self.device)
        predicted = self.net(lam, s, rows)
        loss = F.mse_loss(predicted, self.target)
        self.log("mse", loss)
        self.history.append(float(loss.detach()))
        return loss

    def configure_optimizers(self):
        return torch.optim.Adam(self.parameters(), lr=self.lr)


def warm_start(net, wings, cond=None, epochs=None, target=None):
    """Regress `net` onto a parametric field on the mesh. Returns the module, for its history."""
    stack = np.stack([np.log(np.maximum((target or BASELINE_FIELD)[h], B_FLOOR)) for h in wings])
    module = PretrainModule(net, (stack - LOG_B_MID) / LOG_B_SCALE, cond)
    make_trainer(epochs or PRE_EPOCHS).fit(module, single_batch())
    return module


def draw_field(ax, values, title, signed=False, limit=None):
    """A field over the frozen mesh: log10 B, or a signed difference of logs for a comparison."""
    if signed:
        limit = limit or float(np.nanmax(np.abs(values)))
        drawn = ax.pcolormesh(S_2D, LAM_2D, values, cmap="RdBu_r", vmin=-limit, vmax=limit,
                              shading="auto")
    else:
        drawn = ax.pcolormesh(S_2D, LAM_2D, np.log10(np.maximum(values, B_FLOOR)), cmap="viridis",
                              vmin=np.log10(B_FLOOR), vmax=4.6, shading="auto")
    ax.set_xlabel("s [yr]")
    ax.set_ylabel(r"$|\lambda|$ [deg]")
    ax.set_title(title, fontsize=10)
    ax.grid(False)
    return drawn

In [ ]:
# Given: packing. A fit needs the events of several wings as flat tensors with a wing index beside them,
# because the field is evaluated for every event at once and each event has to know which wing's
# conditioning row it belongs to.
def pack_wings(wings, events, cond_of=None, dtype=TORCH_DTYPE):
    """Several wings' events as flat arrays with a wing index, plus the conditioning matrix."""
    wings = list(wings)
    lat = np.concatenate([events[h][0] for h in wings])
    s = np.concatenate([events[h][1] for h in wings])
    index = np.concatenate([np.full(len(events[h][0]), j, dtype=np.int64)
                            for j, h in enumerate(wings)])
    cond = None if cond_of is None else np.stack([np.asarray(cond_of[h], dtype=float) for h in wings])
    return dict(wings=wings, n_wings=len(wings), n_events=len(lat),
                lat=torch.as_tensor(lat, dtype=dtype), s=torch.as_tensor(s, dtype=dtype),
                index=torch.as_tensor(index), cond=None if cond is None
                else torch.as_tensor(cond, dtype=dtype))


def count_matched_log_c(net, pack):
    """Choose `c` so the model predicts the observed total count before a single step is taken."""
    rows = [None] * pack["n_wings"] if pack["cond"] is None else pack["cond"].numpy()
    total = sum(integrate_2d(np.logaddexp(0.0, as_field_fn(net, row)(LAM_2D[:, None], S_2D[None, :])
                                          - B_CRIT) ** ALPHA) for row in rows)
    return float(np.log(pack["n_events"]) - np.log(total))


TRAIN_PACK = pack_wings(FIT_WINGS, EVENTS)
print(f"packed {TRAIN_PACK['n_events']:,} events over {TRAIN_PACK['n_wings']} wings")

In [ ]:
class BeltLikelihood(L.LightningModule):
    """The week's model: a neural field, one global `log c`, and Phase A's likelihood as the loss."""

    def __init__(self, net, train, log_c_start=0.0, lr=3.0e-3, monitor=None):
        super().__init__()
        self.net = net
        self.lr = lr
        self.log_c = nn.Parameter(torch.tensor(float(log_c_start)))
        self.sizes, self.history, self.monitor_history = {}, [], []
        for tag, pack in (("train", train), ("monitor", monitor)):
            if pack is None:
                continue
            self.sizes[tag] = pack["n_wings"]
            for key in ("lat", "s", "index", "cond"):
                self.register_buffer(f"{tag}_{key}", pack[key])

    def nll(self, tag):
        """The loss on one set of wings, in nats per hemicycle-year."""
        lat, when = getattr(self, f"{tag}_lat"), getattr(self, f"{tag}_s")
        index, cond = getattr(self, f"{tag}_index"), getattr(self, f"{tag}_cond")
        n_wings = self.sizes[tag]
        b_events = self.net.field(lat, when, None if cond is None else cond[index])
        lam_mesh, s_mesh, rows = mesh_batch(n_wings, cond, device=self.device)
        b_mesh = self.net.field(lam_mesh, s_mesh, rows)
        return nll_from_field(b_events, b_mesh, self.log_c, n_wings,
                              torch.as_tensor(S_2D, dtype=TORCH_DTYPE, device=self.device),
                              torch.as_tensor(LAM_2D, dtype=TORCH_DTYPE, device=self.device))

    def training_step(self, batch, batch_index):
        loss = self.nll("train")
        self.log("train_nll", loss)
        self.history.append((self.current_epoch, float(loss.detach())))
        return loss

    def validation_step(self, batch, batch_index):
        with torch.no_grad():
            value = float(self.nll("monitor"))
        self.log("monitor_nll", value)
        self.monitor_history.append((self.current_epoch, value))
        return value

    def configure_optimizers(self):
        return torch.optim.Adam(self.parameters(), lr=self.lr)

In [ ]:
def fit_neural(wings, cond_of=None, n_cond=0, lr=3.0e-3, epochs=None, seed=SEED, net_fn=None,
               events=None, target=None, monitor=None, val_every=None, pre_epochs=None):
    """Warm-start a field, then train it on the likelihood. Returns the fitted LightningModule."""
    torch.manual_seed(seed)
    events = EVENTS if events is None else events
    epochs = EPOCHS if epochs is None else epochs
    net = (net_fn or (lambda: BeltNet(n_cond=n_cond)))()
    cond_rows = None if cond_of is None else np.stack([cond_of[h] for h in wings])
    warm_start(net, wings, cond=cond_rows, epochs=pre_epochs, target=target)

    train = pack_wings(wings, events, cond_of)
    module = BeltLikelihood(net, train, log_c_start=count_matched_log_c(net, train), lr=lr,
                            monitor=None if monitor is None else pack_wings(monitor, events, cond_of))
    trainer = make_trainer(epochs, val_every=val_every or (0 if monitor is None
                                                           else max(1, epochs // 60)))
    trainer.fit(module, single_batch(), None if monitor is None else single_batch())
    return module

In [ ]:
N_COND = 2                         # log A, standardized; and the hemisphere


def conditioning_of(wings, scaler=None):
    """Each wing's conditioning row. Takes a scaler so a fold can standardize with training wings only.

    The only inputs are the area-record amplitude and the hemisphere. No event time of any wing enters
    this function, and nothing from the opposite hemisphere does either.
    """
    wings = list(wings)
    logs = np.array([np.log(AREA_AMPLITUDE[h]) for h in wings], dtype=float)
    if scaler is None:
        scaler = (float(logs.mean()), float(logs.std(ddof=0)) or 1.0)
    mean, sd = scaler
    rows = {h: np.array([(float(np.log(AREA_AMPLITUDE[h])) - mean) / sd,
                         1.0 if h.endswith("N") else -1.0]) for h in wings}
    return rows, scaler


class FiLMBeltNet(BeltNet):
    """Conditioning as modulation: the wing's row sets a scale and a shift in every hidden layer."""

    def __init__(self, n_cond, width=NET_WIDTH, depth=NET_DEPTH, b_start=1.0e4):
        super().__init__(n_cond=0, width=width, depth=depth, b_start=b_start)
        self.film = nn.ModuleList(nn.Linear(n_cond, 2 * width) for _ in range(depth))
        self.n_cond = n_cond
        for layer in self.film:                     # start as the unconditional field, exactly
            nn.init.zeros_(layer.weight)
            nn.init.zeros_(layer.bias)

    def forward(self, abs_lat, s, cond=None):
        x = self.coordinates(abs_lat, s)
        for layer, film in zip(self.hidden, self.film):
            scale, shift = film(cond).chunk(2, dim=-1)
            x = torch.tanh(layer(x) * (1.0 + scale) + shift)
        return self.head(x).squeeze(-1)


CONDITIONING = {
    "none":   dict(n_cond=0, net_fn=lambda: BeltNet(n_cond=0),
                   question="can one field describe eighteen wings at once?"),
    "concat": dict(n_cond=N_COND, net_fn=lambda: BeltNet(n_cond=N_COND),
                   question="is the amplitude enough as an extra coordinate?"),
    "film":   dict(n_cond=N_COND, net_fn=lambda: FiLMBeltNet(n_cond=N_COND),
                   question="does the amplitude have to modulate every layer?"),
}

In [ ]:
# Given: the one piece of fold bookkeeping that is new. The warm-start target has to come from the fold's
# own parameters, which `fold_shared` can rebuild from the cached JSON, with each training wing's
# amplitude predicted by that fold's own area-record map.
def fold_baseline_field(result, cycle, wings):
    """The Week 3 belt as *that fold* fitted it, as a field on the mesh, for each training wing."""
    model, shared, fold = fold_shared(result, cycle)
    intercept, slope = fold["amp_map"]
    out = {}
    for label in wings:
        a_hc = float(np.exp(intercept + slope * np.log(AREA_AMPLITUDE[label])))
        out[label] = field_of(model.intensity(label, a_hc, shared), shared["c"],
                              LAM_2D[:, None], S_2D[None, :])
    return out


def fold_events(train):
    """Training events on the fold's own clock, and the alignment that produced it. As in `loco_v2`."""
    t0_train = np.array([T0_START[h] for h in train])
    index = np.concatenate([np.full(len(RAW_EVENTS[h][0]), j) for j, h in enumerate(train)])
    alignment = fit_alignment(np.concatenate([RAW_EVENTS[h][1] for h in train]),
                              np.concatenate([RAW_EVENTS[h][0] for h in train]), index, t0_train)
    events = {h: (RAW_EVENTS[h][0], RAW_EVENTS[h][1] - (t0_train[j] + alignment["delta"][j]))
              for j, h in enumerate(train)}
    return events, alignment


print("fold bookkeeping ready · warm-start target and clock both come from the training wings alone")

In [ ]:
# Given: Week 5's time-marginal panel, carried forward for Section 7.
def draw_marginals(ax, sources, cycle, width=0.5):
    """The time marginal of each source on one axis: how many events arrive, and when."""
    bins = np.arange(S_A, S_B + width, width)
    for name, source in sources.items():
        counts = np.zeros(len(bins) - 1)
        for label in FOLD_UNITS[cycle]:
            if label in source:
                counts += np.histogram(source[label][1], bins=bins)[0]
        ax.step(0.5 * (bins[1:] + bins[:-1]), counts, where="mid", lw=1.5, label=name)
    ax.set_xlim(S_A, S_B)
    ax.set_xlabel("s [yr]")
    ax.set_ylabel(f"events per {width:.1f} yr")
    ax.set_title(f"cycle {cycle} in time", fontsize=10)
    ax.legend(fontsize=7)


print("Week 5's marginal panel carried forward")

In [ ]:
# Given: the two bases, side by side. The champion is 29 fitted numbers of physics; the FiLM field is
# ~5,800 weights told two things about each wing. Week 5 scored them as a tie (+0.08 +/- 0.18 nats per
# hemicycle-year over nine folds, better on 9 of 18), which is why this week can ask the same question
# of both. This in-sample FiLM fit exists so Figure 1 can show both bases; the scored neural runs happen
# in Section 2.
STRONGEST = max(FIT_WINGS, key=lambda h: AREA_AMPLITUDE[h])
WEAKEST = min(FIT_WINGS, key=lambda h: AREA_AMPLITUDE[h])
SCORED_WINGS = [h for wings in FOLD_UNITS.values() for h in wings]

champion_model, champion_shared, champion_amps = in_sample_shared(CHAMPION_RESULT)
CHAMPION_INTENSITY = {h: champion_model.intensity(h, champion_amps[h], champion_shared)
                      for h in FIT_WINGS}
CHAMPION_FIELD = {h: field_of(CHAMPION_INTENSITY[h], champion_shared["c"],
                              LAM_2D[:, None], S_2D[None, :]) for h in FIT_WINGS}

baseline_model, baseline_shared, baseline_amps = in_sample_shared(BASELINE_RESULT)
BASELINE_FIELD = {h: field_of(baseline_model.intensity(h, baseline_amps[h], baseline_shared),
                              baseline_shared["c"], LAM_2D[:, None], S_2D[None, :])
                  for h in FIT_WINGS}

COND, COND_SCALER = conditioning_of(FIT_WINGS)
started = time.time()
FILM_IN = fit_neural(FIT_WINGS, cond_of=COND, n_cond=N_COND, net_fn=lambda: FiLMBeltNet(n_cond=N_COND),
                     lr=BEST_LR, epochs=FILM_BUDGET_W05)
FILM_LOG_C = float(FILM_IN.log_c.detach())
FILM_INTENSITY = {h: as_intensity_fn(FILM_IN.net, COND[h], FILM_LOG_C) for h in FIT_WINGS}
print(f"FiLM field refitted in {time.time() - started:.1f} s · "
      f"{sum(p.numel() for p in FILM_IN.net.parameters()) + 1:,} fitted numbers against the champion's "
      f"{CHAMPION_RESULT['n_params']}")

In [ ]:
# Given: Week 1's rate-matched null, carried forward because this week uses it as a *calibration* rather
# than a baseline. It keeps a wing's rate envelope and throws away everything else, so by construction it
# has NO memory at all: it is the right generator for "what does an arrow-of-time statistic read on a pure
# shared cause, and how much does it wobble?"
def simulate_rate_matched_null(event_s, s_lo=S_A, s_hi=S_B, smooth_days=90.0, rng=None):
    """Week 1's null: the wing's own smoothed daily rate, then Poisson counts scattered inside each day."""
    rng = np.random.default_rng(SEED) if rng is None else rng
    edges = np.arange(s_lo, s_hi + ONE_DAY, ONE_DAY)
    counts = np.histogram(np.asarray(event_s, dtype=float), bins=edges)[0].astype(float)
    width = max(1, int(round(smooth_days)))
    box = np.ones(width)
    # divide by the boxcar applied to ones, so the window edges are not dragged towards zero
    smoothed = np.convolve(counts, box, mode="same") / np.convolve(np.ones_like(counts), box, mode="same")
    drawn = rng.poisson(smoothed)
    out = np.concatenate([edges[i] + rng.uniform(0.0, ONE_DAY, k) for i, k in enumerate(drawn) if k]) \
        if drawn.sum() else np.empty(0)
    return np.sort(out)


print(f"Week 1's rate-matched null ready · smoothing widths 1 yr and 3 months were the two Week 1 used · "
      f"{16.4:.1f}% of this catalog's gaps are exactly zero, which is where its one surviving result lives")

In [ ]:
# Given (Week 6, Task 56): The history bank — eight strictly-past channels, and the field they make
NU_START = 0.01                    # where every history fit starts: a hundredth of an offspring per event


def history_bank(event_lat, event_s, strictly_past=True, channels=ALL_CHANNELS):
    """Every channel's value at every event of one wing, from that wing's own earlier events.

    Returns `H` with shape (n_events, 8):

        H[i, m] = sum over j with s_j < s_i of exp(-(s_i - s_j)/tau_m) * exp(-(lat_i - lat_j)^2/2 ell_m^2)

    Events must already be sorted in `event_s`. Channels outside `channels` are left at zero, which is
    how the `nohist` and `short` arms are the same code as the full bank.
    """
    lat = np.asarray(event_lat, dtype=float)
    s = np.asarray(event_s, dtype=float)
    H = np.zeros((len(s), N_CHANNELS))
    if len(s) == 0 or not len(channels):
        return H
    ds = s[:, None] - s[None, :]           # row i, column j: how long after event j event i arrived
    dl = lat[:, None] - lat[None, :]
    # Strictly past. A tie is NOT its own parent and not each other's: 16.4% of this catalog's gaps are
    # exactly zero, so `>` against `>=` is the difference Section 6 measures.
    past = ds > 0.0 if strictly_past else ds >= 0.0
    for m in channels:
        j, k = CHANNELS[m]
        # the absolute value keeps exp() from overflowing on the future half, which is masked away anyway
        decay = np.where(past, np.exp(-np.abs(ds) / TAUS[j]), 0.0)
        H[:, m] = (decay * np.exp(-dl ** 2 / (2.0 * ELLS[k] ** 2))).sum(axis=1)
    return H


def history_field(abs_lat, s, lat_par, s_par, beta, block=256):
    """The same sum at *any* (|lat|, s): `sum_jk beta_jk h_jk`, in events per degree per year.

    This is the one function that evaluates the history term off the events, and it takes no mesh: the
    likelihood's history integral is closed form, so this is only ever used for figures, for the time
    marginal and for the thinning majorant.
    """
    abs_lat, s = np.broadcast_arrays(np.asarray(abs_lat, dtype=float), np.asarray(s, dtype=float))
    beta = np.asarray(beta, dtype=float)
    out = np.zeros(abs_lat.shape)
    for m in np.flatnonzero(beta):
        j, k = CHANNELS[int(m)]
        for lo in range(0, len(s_par), block):   # blocked over parents, or one wing eats the memory
            sp, lp = np.asarray(s_par)[lo:lo + block], np.asarray(lat_par)[lo:lo + block]
            ds = s[..., None] - sp
            dl = abs_lat[..., None] - lp
            kernel = np.where(ds > 0.0, np.exp(-np.abs(ds) / TAUS[j]), 0.0) \
                * np.exp(-dl ** 2 / (2.0 * ELLS[k] ** 2))
            out += beta[int(m)] * kernel.sum(axis=-1)
    return out


def draw_history(ax, field, title, events=None, grid=H_MESH, cmap="inferno"):
    """The extra rate the past has left behind, over one wing, with the events that left it on top."""
    s_grid, lam_grid = grid
    drawn = ax.pcolormesh(s_grid, lam_grid, field, cmap=cmap, shading="auto")
    if events is not None:
        ax.scatter(events[1], events[0], s=2.6, c="cyan", alpha=0.5, linewidths=0)
    ax.set_xlim(s_grid[0], s_grid[-1])
    ax.set_ylim(lam_grid[0], lam_grid[-1])
    ax.set_xlabel("s [yr]")
    ax.set_ylabel(r"$|\lambda|$ [deg]")
    ax.set_title(title, fontsize=10)
    ax.grid(False)
    return drawn

In [ ]:
# Given (Week 6, Task 57): The compensator in closed form, and what the frozen mesh would have said
def latitude_mass(event_lat, ell):
    """One latitude kernel integrated over |lat| in [0, LAM_MAX], exactly, for every event.

    The two erf terms are the truncation at the poleward edge and at the equator. Dropping them and
    using ell*sqrt(2*pi) overstates the mass by up to a factor of two near the equator.
    """
    lat = np.asarray(event_lat, dtype=float)
    root2 = np.sqrt(2.0)
    return ell * np.sqrt(0.5 * np.pi) * (erf((LAM_MAX - lat) / (ell * root2))
                                         + erf(lat / (ell * root2)))


def history_compensator(event_lat, event_s, s_hi=S_B, channels=ALL_CHANNELS):
    """A[m] = the double integral of channel m over the window, in closed form. Shape (8,).

    Returned per channel and NOT multiplied by beta, so the fitter can profile `c` against any beta
    without recomputing anything.
    """
    lat = np.asarray(event_lat, dtype=float)
    s = np.asarray(event_s, dtype=float)
    A = np.zeros(N_CHANNELS)
    for m in channels:
        j, k = CHANNELS[m]
        in_time = TAUS[j] * (1.0 - np.exp(-(s_hi - s) / TAUS[j]))
        A[m] = float(np.sum(in_time * latitude_mass(lat, ELLS[k])))
    return A


def branching_ratio(nu):
    """Expected offspring per event. The stability condition is n < 1 and we assert it before simulating."""
    return float(np.sum(np.asarray(nu, dtype=float)))


def draw_mass_error(ax, exact, on_mesh):
    """What the frozen quadrature mesh would have made of each channel's integral."""
    position = np.arange(N_CHANNELS)
    ax.axhline(1.0, color="k", lw=1.0)
    ax.bar(position, on_mesh / exact, color=["tab:red" if abs(r - 1.0) > 0.05 else "tab:blue"
                                             for r in on_mesh / exact])
    ax.set_xticks(position)
    ax.set_xticklabels(CHANNEL_NAMES, rotation=90, fontsize=7)
    ax.set_ylabel("mesh quadrature / exact")
    ax.set_title(r"(a) the frozen mesh on a 31-day grid", fontsize=10)

In [ ]:
# Given: the bookkeeping that keeps one wing's events, its H rows and its base values in one order.
# `s_true` is the *unreflected* time and is where the base is always evaluated. That matters in Section 3:
# reflecting the data and the base together leaves the base's contribution to the likelihood bit-for-bit
# identical, which is what makes the asymmetry statistic a measurement of the history term alone.
def history_of(events, labels, strictly_past=True, reflected=False, channels=ALL_CHANNELS):
    """Each wing's bank: its events sorted in the direction being read, its H matrix and its A vector."""
    bank = {}
    for label in labels:
        lat0 = np.asarray(events[label][0], dtype=float)
        s0 = np.asarray(events[label][1], dtype=float)
        s_use = (S_A + S_B - s0) if reflected else s0
        order = np.argsort(s_use, kind="stable")
        lat, s_read, s_true = lat0[order], s_use[order], s0[order]
        bank[label] = dict(label=label, lat=lat, s=s_read, s_true=s_true, reflected=bool(reflected),
                           H=history_bank(lat, s_read, strictly_past, channels),
                           A=history_compensator(lat, s_read, channels=channels))
    return bank


def log_likelihood_history(bank_wing, intensity_fn, beta, mesh=MESH_2D):
    """One wing's log L with an additive history term. The base uses the mesh; the history never does."""
    at = np.asarray(intensity_fn(bank_wing["lat"], bank_wing["s_true"]), dtype=float)
    at = at + bank_wing["H"] @ np.asarray(beta, dtype=float)
    if not np.all(np.isfinite(at)) or np.any(at <= 0.0):
        return -np.inf
    base_integral = integrate_2d(intensity_fn(mesh[1][:, None], mesh[0][None, :]), mesh)
    total = float(np.sum(np.log(at)) - base_integral - float(bank_wing["A"] @ np.asarray(beta, float)))
    return total if np.isfinite(total) else -np.inf


def with_history(intensity_fn, lat_par, s_par, beta):
    """Base + history as one `intensity_fn(abs_lat, s)`.

    For figures, for the time marginal and for the thinning majorant. NEVER for the likelihood's integral
    term: integrating this on the frozen mesh is precisely the error Figure 4 measures.
    """
    def combined(abs_lat, s):
        return (np.asarray(intensity_fn(abs_lat, s), dtype=float)
                + history_field(abs_lat, s, lat_par, s_par, beta))
    return combined


print("bank bookkeeping · history likelihood · the wrapper for everything that is not the likelihood")

In [ ]:
# Given (Week 6, Task 58): Fit the eight coefficients, and the count-matching identity they change
def beta_of(nu):
    """Offspring units to intensity units. The optimizer sees `nu`; the likelihood needs `beta`."""
    return np.asarray(nu, dtype=float) / CHANNEL_MASS


def solve_c_history(n_total, unit_integral, a_total):
    """`c` is no longer n / int-int-Lambda_1.

    The history term already supplies `a_total` of the expected count, so `c` may only take what is
    left: c = (n - A.beta) / int-int-Lambda_1. Leave this at Week 4's version and the fitted model stops
    reproducing the pooled count, with `c` quietly absorbing the history mass and no error anywhere.
    """
    remaining = n_total - a_total
    if not (remaining > 0.0 and unit_integral > 0.0):
        return np.nan
    return remaining / unit_integral


def nu_from_z(z, free, cap=N_CAP):
    """Non-negative coefficients whose total stays strictly below `cap`.

    `cap * exp(z) / (1 + sum(exp(z)))` is a softmax with one spare slot: every channel is non-negative,
    the branching ratio approaches `cap` without reaching it, and sending every z to minus infinity
    gives exactly nu = 0. That keeps the fit inside the model family — see `N_CAP` above for why n < 1
    is both the stability condition and what makes `c` positive.
    """
    nu = np.zeros(N_CHANNELS)
    if not len(free):
        return nu
    weights = np.exp(np.clip(np.asarray(z, dtype=float), -60.0, 60.0))
    nu[free] = cap * weights / (1.0 + weights.sum())
    return nu


def z_from_nu(nu, free, cap=N_CAP):
    """The inverse, for a warm start. Not unique in general; this is the branch the fit uses."""
    nu = np.maximum(np.asarray(nu, dtype=float)[free], NU_START * 1.0e-3)
    share = np.clip(nu.sum() / cap, 1.0e-9, 1.0 - 1.0e-9)
    return np.log(nu / cap) + np.log(1.0 / (1.0 - share))


def fit_nu(unit_at, unit_int, H, A, n_total, nu_start=None, nu_max=None, channels=ALL_CHANNELS):
    """The history block: eight coefficients, with `c` profiled out by count matching at every step.

    `unit_at` is the base intensity at every event with c = 1, stacked over wings in bank order;
    `unit_int` is the sum of the base mesh integrals at c = 1; `H` and `A` are stacked and summed the
    same way. Because count matching forces c*unit_int + A.beta = n_total exactly, the integral term is
    the constant n_total and the objective is the event sum alone — so this block never touches a mesh.
    """
    free = list(channels)
    if not free:
        return np.zeros(N_CHANNELS), None
    cap = N_CAP if nu_max is None else nu_max
    start = (np.full(N_CHANNELS, NU_START) if nu_start is None
             else np.maximum(np.asarray(nu_start, dtype=float), NU_START))

    def unpack(z):
        return nu_from_z(z, free, cap)

    def objective(z):
        nu = unpack(z)
        beta = beta_of(nu)
        c = solve_c_history(n_total, unit_int, float(A @ beta))
        if not np.isfinite(c) or c <= 0.0:
            return 1.0e12
        at = c * unit_at + H @ beta
        if not np.all(np.isfinite(at)) or np.any(at <= 0.0):
            return 1.0e12
        return -(float(np.sum(np.log(at))) - n_total)

    out = minimize(objective, z_from_nu(start, free, cap), method="L-BFGS-B",
                   options=dict(maxiter=400))
    return unpack(out.x), out

In [ ]:
# Given: the parametric arm's fit. This is Week 4's `fit_pooled_v2` with one more block in the same
# block-coordinate sweep — history, then the shared block, then each wing's own amplitude, then `c` —
# and `c` re-solved with Task 58's corrected identity every time anything moves. `Model` is untouched:
# beta is not a stage, which is exactly why Task 59 has to put the arm in the cache key by hand.
def fit_pooled_history(events, labels, model, bank, q_start=None, amp_start=None, wing_start=None,
                       sweeps=3, channels=ALL_CHANNELS, nu_start=None, nu_max=None, mesh=MESH_2D):
    """Fit the base and the eight history coefficients jointly. Returns the fit, including `nu`."""
    labels = list(labels)
    n_total = sum(len(bank[h]["s"]) for h in labels)
    q = model.start_q() if q_start is None else np.array(q_start, dtype=float)
    amps = {h: 2.0e4 for h in labels} if amp_start is None else {h: float(amp_start[h]) for h in labels}
    wing = ({h: model.start_wing() for h in labels} if wing_start is None
            else {h: np.array(wing_start[h], dtype=float) for h in labels})
    nu = np.zeros(N_CHANNELS) if nu_start is None else np.array(nu_start, dtype=float)
    lam_col, s_row = mesh[1][:, None], mesh[0][None, :]
    H_all = np.concatenate([bank[h]["H"] for h in labels]) if labels else np.zeros((0, N_CHANNELS))
    A_all = np.sum([bank[h]["A"] for h in labels], axis=0)

    def geometric_mean(values):
        return float(np.exp(np.mean(np.log(list(values)))))

    def unit_pieces(a_ref):
        """The base at c = 1: its values at every event, stacked in bank order, and its mesh integral."""
        unit = model.unpack(q, 1.0, a_ref)
        at, integral = [], 0.0
        for h in labels:
            fn = model.intensity(h, amps[h], unit, wing[h])
            at.append(np.asarray(fn(bank[h]["lat"], bank[h]["s_true"]), dtype=float))
            integral += integrate_2d(fn(lam_col, s_row), mesh)
        return np.concatenate(at) if at else np.zeros(0), float(integral)

    def shape_nll(q_try, c, a_ref, beta):
        shared = model.unpack(q_try, c, a_ref)
        total = 0.0
        for h in labels:
            value = log_likelihood_history(bank[h], model.intensity(h, amps[h], shared, wing[h]),
                                           beta, mesh)
            if not np.isfinite(value):
                return 1.0e12
            total += value
        return -total

    def wing_nll(h, shared, beta):
        def nll(vector):
            fn = model.intensity(h, float(np.exp(vector[0])), shared, vector[1:])
            value = log_likelihood_history(bank[h], fn, beta, mesh)
            return -value if np.isfinite(value) else 1.0e12
        return nll

    a_ref = geometric_mean(amps.values())
    beta, c = beta_of(nu), np.nan
    for _ in range(sweeps):
        unit_at, unit_int = unit_pieces(a_ref)
        nu, _ = fit_nu(unit_at, unit_int, H_all, A_all, n_total, nu_start=nu, nu_max=nu_max,
                       channels=channels)
        beta = beta_of(nu)
        c = solve_c_history(n_total, unit_int, float(A_all @ beta))
        q = minimize(shape_nll, q, args=(c, a_ref, beta), method="L-BFGS-B",
                     options=dict(maxiter=300)).x
        shared = model.unpack(q, c, a_ref)
        for h in labels:
            if model.wing_idx:
                out = minimize(wing_nll(h, shared, beta),
                               np.concatenate([[np.log(amps[h])], wing[h]]), method="Nelder-Mead",
                               options=dict(maxiter=250, maxfev=250, xatol=1e-4, fatol=1e-4))
                amps[h], wing[h] = float(np.exp(out.x[0])), out.x[1:]
            else:
                out = minimize_scalar(lambda z: wing_nll(h, shared, beta)(np.array([z])),
                                      bounds=AMPLITUDE_BRACKET, method="bounded",
                                      options=dict(xatol=1e-4))
                amps[h] = float(np.exp(out.x))
        a_ref = geometric_mean(amps.values())
        unit_at, unit_int = unit_pieces(a_ref)
        c = solve_c_history(n_total, unit_int, float(A_all @ beta))
    return dict(q=q, shared=model.unpack(q, c, a_ref), amplitudes=amps, wing=wing, labels=labels,
                nu=nu, beta=beta, a_ref=a_ref, c=float(c),
                nll=shape_nll(q, c, a_ref, beta), n_total=n_total)


def fit_nu_only(bank, labels, intensity_of, channels=ALL_CHANNELS, nu_max=None, nu_start=None):
    """Fit the eight coefficients with the base and `c` held exactly as they are.

    Used in two places and nowhere else: Section 3's asymmetry statistic, where the base *cancels*
    between the two directions, and Section 6's leaky refit, where a joint fit would spend three minutes
    letting a runaway coefficient drag the base behind it. Here `c` is fixed, so the integral term does
    not collapse and `A @ beta` stays in the objective.
    """
    free = list(channels)
    if not free:
        return np.zeros(N_CHANNELS), 0.0
    at = np.concatenate([np.asarray(intensity_of(h)(bank[h]["lat"], bank[h]["s_true"]), dtype=float)
                         for h in labels])
    H_all = np.concatenate([bank[h]["H"] for h in labels])
    A_all = np.sum([bank[h]["A"] for h in labels], axis=0)
    base_sum = float(np.sum(np.log(at)))
    cap = N_CAP if nu_max is None else nu_max
    start = (np.full(N_CHANNELS, NU_START) if nu_start is None
             else np.maximum(np.asarray(nu_start, dtype=float), NU_START))

    def unpack(z):
        return nu_from_z(z, free, cap)

    def objective(z):
        beta = beta_of(unpack(z))
        values = at + H_all @ beta
        if not np.all(np.isfinite(values)) or np.any(values <= 0.0):
            return 1.0e12
        return -(float(np.sum(np.log(values))) - float(A_all @ beta))

    out = minimize(objective, z_from_nu(start, free, cap), method="L-BFGS-B",
                   options=dict(maxiter=400))
    nu = unpack(out.x)
    gain = -float(out.fun) - base_sum          # log L with history minus log L without, in nats
    return nu, gain


print("parametric history fit ready · one more block in Week 4's sweep · `c` re-solved every time")

In [ ]:
# Given: the neural arm. Eight parameters on top of Week 5's module and nothing else moves — the network
# still sees (|lat|, s, conditioning) and nothing about the past, because the history term is additive in
# the intensity and sits outside the field entirely. `BeltNet`, `as_field_fn` and `as_intensity_fn` are
# untouched.
def pack_history(labels, bank, cond_of=None, dtype=TORCH_DTYPE):
    """`pack_wings`, built from the bank so events, H rows and the base can never fall out of order."""
    labels = list(labels)
    events = {h: (bank[h]["lat"], bank[h]["s_true"]) for h in labels}
    pack = pack_wings(labels, events, cond_of, dtype)
    pack["H"] = torch.as_tensor(np.concatenate([bank[h]["H"] for h in labels]), dtype=dtype)
    pack["A"] = torch.as_tensor(np.sum([bank[h]["A"] for h in labels], axis=0), dtype=dtype)
    return pack


def count_matched_log_c_history(net, pack, beta=None):
    """Week 5's initializer with Task 58's identity: `c` takes the count the history term does not."""
    rows = [None] * pack["n_wings"] if pack["cond"] is None else pack["cond"].cpu().numpy()
    unit = sum(integrate_2d(np.logaddexp(0.0, as_field_fn(net, row)(LAM_2D[:, None], S_2D[None, :])
                                         - B_CRIT) ** ALPHA) for row in rows)
    a_total = 0.0 if beta is None else float(pack["A"].cpu().numpy() @ np.asarray(beta, dtype=float))
    return float(np.log(max(pack["n_events"] - a_total, 1.0)) - np.log(unit))


class HistoryLikelihood(BeltLikelihood):
    """Week 5's module plus eight non-negative history coefficients, fitted jointly with the field."""

    def __init__(self, net, train, log_c_start=0.0, lr=3.0e-3, monitor=None, nu_max=None,
                 channels=ALL_CHANNELS):
        super().__init__(net, train, log_c_start=log_c_start, lr=lr, monitor=monitor)
        self.cap = float(N_CAP if nu_max is None else nu_max)
        mask = np.zeros(N_CHANNELS)
        mask[list(channels)] = 1.0
        self.z = nn.Parameter(torch.full((N_CHANNELS,), float(np.log(NU_START / self.cap))))
        self.register_buffer("mask", torch.as_tensor(mask, dtype=TORCH_DTYPE))
        self.register_buffer("mass", torch.as_tensor(CHANNEL_MASS, dtype=TORCH_DTYPE))
        for tag, pack in (("train", train), ("monitor", monitor)):
            if pack is None:
                continue
            self.register_buffer(f"{tag}_H", pack["H"])
            self.register_buffer(f"{tag}_A", pack["A"])

    def nu(self):
        """Task 58's `nu_from_z`, in torch: non-negative, and the total strictly below the cap."""
        weights = torch.exp(self.z.clamp(-60.0, 60.0)) * self.mask
        return self.cap * weights / (1.0 + weights.sum())

    def nll(self, tag):
        lat, when = getattr(self, f"{tag}_lat"), getattr(self, f"{tag}_s")
        index, cond = getattr(self, f"{tag}_index"), getattr(self, f"{tag}_cond")
        bank_h, bank_a = getattr(self, f"{tag}_H"), getattr(self, f"{tag}_A")
        n_wings = self.sizes[tag]
        beta = self.nu() / self.mass
        b_events = self.net.field(lat, when, None if cond is None else cond[index])
        lam_mesh, s_mesh, rows = mesh_batch(n_wings, cond, device=self.device)
        b_mesh = self.net.field(lam_mesh, s_mesh, rows)
        at_events = emission_torch(b_events, self.log_c) + (bank_h * beta).sum(-1)
        on_mesh = emission_torch(b_mesh, self.log_c)
        s_grid = torch.as_tensor(S_2D, dtype=TORCH_DTYPE, device=self.device)
        lam_grid = torch.as_tensor(LAM_2D, dtype=TORCH_DTYPE, device=self.device)
        integral = torch.trapezoid(torch.trapezoid(on_mesh, lam_grid, dim=-2), s_grid, dim=-1).sum()
        integral = integral + (bank_a * beta).sum()
        return -(torch.log(at_events).sum() - integral) / (n_wings * WINDOW_YEARS)


def fit_history_neural(wings, bank, cond_of=None, n_cond=0, net_fn=None, lr=None, epochs=None,
                       seed=SEED, target=None, monitor=None, monitor_bank=None,
                       channels=ALL_CHANNELS, nu_max=None, pre_epochs=None, val_every=None):
    """Warm-start a field, then train the field and the eight coefficients together."""
    torch.manual_seed(seed)
    epochs = EPOCHS if epochs is None else epochs
    net = (net_fn or (lambda: BeltNet(n_cond=n_cond)))()
    cond_rows = None if cond_of is None else np.stack([cond_of[h] for h in wings])
    warm_start(net, wings, cond=cond_rows, epochs=pre_epochs, target=target)
    train = pack_history(wings, bank, cond_of)
    watch = None if monitor is None else pack_history(monitor, monitor_bank, cond_of)
    module = HistoryLikelihood(net, train, log_c_start=count_matched_log_c_history(net, train),
                               lr=BEST_LR if lr is None else lr, monitor=watch, nu_max=nu_max,
                               channels=channels)
    trainer = make_trainer(epochs, val_every=val_every or (0 if monitor is None
                                                           else max(1, epochs // 60)))
    trainer.fit(module, single_batch(), None if monitor is None else single_batch())
    return module


print("neural history arm ready · 8 parameters added to Week 5's module, the field untouched")

In [ ]:
# Given: the time-rescaling machinery, with the history term's compensator in closed form. Week 2's
# `rescaled_gaps` integrates the marginal rate on `S_MESH`, whose spacing is 1.55 days — still coarser
# than the 2-day channel, so a mesh-only compensator would make the KS column wrong for exactly the
# channel this week cares about. The base keeps the mesh; the history does not need one.
def history_rate(lat_par, s_par, beta, s_at):
    """The latitude-integrated history rate at each time in `s_at`, in events per year."""
    s_at = np.asarray(s_at, dtype=float)
    s_par = np.asarray(s_par, dtype=float)
    beta = np.asarray(beta, dtype=float)
    out = np.zeros(s_at.shape)
    for m in np.flatnonzero(beta):
        j, k = CHANNELS[int(m)]
        mass = latitude_mass(lat_par, ELLS[k])
        ds = s_at[:, None] - s_par[None, :]
        out += beta[int(m)] * (np.where(ds > 0.0, np.exp(-np.abs(ds) / TAUS[j]), 0.0) @ mass)
    return out


def history_cumulative(lat_par, s_par, beta, s_at):
    """The history term's compensator up to each time in `s_at`, exactly.

    Each past parent contributes tau*(1 - exp(-(s - s_i)/tau)) times its own latitude mass.
    """
    s_at = np.asarray(s_at, dtype=float)
    s_par = np.asarray(s_par, dtype=float)
    beta = np.asarray(beta, dtype=float)
    out = np.zeros(s_at.shape)
    for m in np.flatnonzero(beta):
        j, k = CHANNELS[int(m)]
        mass = latitude_mass(lat_par, ELLS[k])
        ds = s_at[:, None] - s_par[None, :]
        grown = np.where(ds > 0.0, TAUS[j] * (1.0 - np.exp(-np.abs(ds) / TAUS[j])), 0.0)
        out += beta[int(m)] * (grown @ mass)
    return out


def rescaled_gaps_history(bank_wing, base_marginal, beta, rng):
    """Week 2's rescaled gaps, with the history compensator added analytically.

    The tie policy is Week 2's and is stated rather than applied silently: each event is jittered inside
    its own day with a declared seed. The jittered times are used as the parents as well, so the
    compensator stays consistent with the events it is being compared against.
    """
    order = np.argsort(bank_wing["s_true"])
    lat, s = bank_wing["lat"][order], bank_wing["s_true"][order]
    jittered = s + rng.uniform(0.0, ONE_DAY, len(s))
    keep = np.argsort(jittered)
    lat, jittered = lat[keep], jittered[keep]
    base_compensator = np.concatenate([[0.0], cumulative_trapezoid(base_marginal, S_MESH)])
    at_events = np.interp(jittered, S_MESH, base_compensator)
    if np.any(np.asarray(beta) > 0.0):
        at_events = at_events + history_cumulative(lat, jittered, beta, jittered)
    return np.diff(np.concatenate([[0.0], at_events]))


print("time rescaling with history · base on S_MESH (1.55 d), history in closed form")

In [ ]:
# Given: the five arms, one question each, named here because the names travel to Weeks 7 and 8.
HISTORY_TAGS = ("nohist", "short", "bank", "bankrev", "bankleak")
ARM_CHANNELS = {"nohist": (), "short": SHORT_CHANNELS, "bank": ALL_CHANNELS,
                "bankrev": ALL_CHANNELS, "bankleak": ALL_CHANNELS}
ARM_QUESTION = {
    "nohist": "the base alone — the row everything else is nested inside",
    "short": "do the two shortest timescales carry it on their own?",
    "bank": "does the whole bank buy anything the base cannot already do?",
    "bankrev": "does the same bank work as well read backwards? (Section 3, not a fold run)",
    "bankleak": "what does writing <= instead of < buy? (Section 6)",
}

In [ ]:
# Given (Week 6, Task 59): The two-arm runner — its cache key, the digest it checks, and a held-out wing's own past
def run_tag(base, arm, seed, epochs):
    """A digest over everything a row depends on apart from which folds it scored."""
    recipe = dict(fingerprint=protocol_fingerprint()["recipe"], base=str(base), arm=str(arm),
                  channels=list(ARM_CHANNELS[arm]), seed=int(seed), epochs=epochs, lr=BEST_LR,
                  n_cap=(NU_MAX_LEAK if arm == "bankleak" else N_CAP),
                  strictly_past=(arm != "bankleak"), reflected=(arm == "bankrev"))
    text = json.dumps(recipe, sort_keys=True, default=str)
    return hashlib.sha1(text.encode()).hexdigest()[:10]


def history_key(base, arm, folds):
    """The cache name, generated from the configuration. `arm` is in it because beta is NOT a stage.

    Week 4's `variant_name(spec)` reads the stage dictionary, and the history coefficients are not in
    it — so a history champion and a plain champion produce the *identical* variant name and would
    collide on one JSON, with whichever ran first winning silently.
    """
    return f"{base}__{arm}__cycles{'-'.join(str(c) for c in folds)}"


def check_cached(result, base, arm, seed, epochs):
    """The check Week 5 stored and never performed: assert the digest on a cache HIT, not on the write."""
    expected = run_tag(base, arm, seed, epochs)
    found = result.get("run_tag")
    if found != expected:
        raise RuntimeError(f"stale cache for {result.get('name')}: digest {found} on disk, "
                           f"{expected} for the protocol now in force. Delete it and rerun.")
    return result


def score_held_wing(label, alignment, base_fn, beta, a_hc, strictly_past=True,
                    channels=ALL_CHANNELS):
    """One held-out wing's row: its own clock, its own strictly-past history, both metrics.

    The wing's own earlier events are an input here, and that is not leakage — a conditional intensity
    is *defined* by them, and a forecaster standing at time s has them. Its future never enters, and no
    other wing's events enter at all.
    """
    lat_held, t_held = RAW_EVENTS[label]
    shift = align_one(t_held, lat_held, T0_START[label], alignment["mu_b"])
    s_held = t_held - (T0_START[label] + shift)
    inside = (s_held > S_A) & (s_held < S_B)
    lat_in, s_in = lat_held[inside], s_held[inside]
    bank = history_of({label: (lat_in, s_in)}, [label], strictly_past=strictly_past,
                      channels=channels)[label]
    value = log_likelihood_history(bank, base_fn, beta)
    marginal = time_marginal(base_fn)
    gaps = rescaled_gaps_history(bank, marginal, beta,
                                 np.random.default_rng(SEED + 1000 + FIT_WINGS.index(label)))
    predicted = (integrate_2d(base_fn(LAM_2D[:, None], S_2D[None, :]))
                 + float(bank["A"] @ np.asarray(beta, dtype=float)))
    return dict(hemicycle=label, n=int(inside.sum()), outside=int((~inside).sum()),
                delta=float(shift),
                **{f"a ({HEADLINE})": float(a_hc),
                   f"nll/yr ({HEADLINE})": float(-value / WINDOW_YEARS),
                   f"predicted ({HEADLINE})": float(predicted),
                   f"ks ({HEADLINE})": float(ks_vs_exp1(gaps)),
                   f"ks_p ({HEADLINE})": float(kstest(gaps, "expon").pvalue)})

In [ ]:
# Given: the fold loop that calls Task 59's pieces. One row per held-out hemicycle, exactly as Week 4's
# `loco_v2` and Week 5's `run_neural` produce — the same column names, so `rows_of` and `paired_stats`
# work on these results unchanged.
#
# `frozen_from` is the one place the protocol bends, in two places only: Section 3's reversed reading,
# where the base *cancels* between the directions, and Section 6's leaky refit. Everywhere else the
# coefficients are fitted jointly with the base — a frozen base measures "does history matter given a
# base never allowed to compete", which inflates the neural arm and wrecks Section 4.
BUDGET_WINGS = [h for h in FIT_WINGS if h not in VALID_WINGS]


def run_history(base, arm, folds=None, seed=SEED, epochs=None, frozen_from=None, sweeps=2):
    """Fit and score one (base, arm) pair over the folds. Cached, with the digest checked on lookup."""
    folds = tuple(folds or FOLD_CYCLES)
    key = history_key(base, arm, folds)
    path = HISTORY_CACHE / f"{key}.json"
    if path.exists():
        with open(path) as handle:
            return check_cached(json.load(handle), base, arm, seed, epochs)

    assert not set(folds) & set(WITHHELD_CYCLES), "a withheld cycle reached the history runner"
    channels = ARM_CHANNELS[arm]
    strictly_past = arm != "bankleak"
    reflected = arm == "bankrev"
    nu_max = NU_MAX_LEAK if arm == "bankleak" else None
    started, rows, fold_params, in_nll = time.time(), [], {}, []

    for held_out in folds:
        train = [h for h in FIT_WINGS if int(h[:-1]) != held_out]
        events, alignment = fold_events(train)
        bank = history_of(events, train, strictly_past=strictly_past, reflected=reflected,
                          channels=channels)
        frozen = None if frozen_from is None else fold_base_of(frozen_from, held_out)

        if base == "champion":
            model = Model(CHAMPION_SPEC)
            if frozen is None:
                fit = fit_pooled_history(events, train, model, bank, q_start=CHAMPION_RESULT["q"],
                                         amp_start=CHAMPION_RESULT["amplitudes"], sweeps=sweeps,
                                         channels=channels, nu_max=nu_max)
                q, c, a_ref = fit["q"], fit["shared"]["c"], fit["a_ref"]
                amps, wing, nu = fit["amplitudes"], fit["wing"], fit["nu"]
                in_nll.append(fit["nll"] / (len(train) * WINDOW_YEARS))
            else:
                nu, _ = fit_nu_only(bank, train,
                                    lambda h: frozen["base_of"](h, frozen["amps"][h]),
                                    channels=channels, nu_max=nu_max)
                q, c, a_ref = frozen["q"], frozen["c"], frozen["a_ref"]
                amps, wing = frozen["amps"], frozen["wing"]
                in_nll.append(np.nan)
            shared = model.unpack(np.array(q), c, a_ref)
            intercept, slope = np.polyfit(np.log([AREA_AMPLITUDE[h] for h in train]),
                                          np.log([amps[h] for h in train]), 1)[::-1]
            fold_params[str(held_out)] = dict(train=train, mu_b_clock=float(alignment["mu_b"]),
                                              nu=[float(v) for v in nu],
                                              amp_map=[float(intercept), float(slope)],
                                              q=[float(v) for v in q], c=float(c), a_ref=float(a_ref),
                                              amps={h: float(amps[h]) for h in train},
                                              wing={h: [float(x) for x in wing[h]] for h in train})
            n_params = model.n_params + int(np.count_nonzero(nu))
            def base_for(label, a_hc):
                return model.intensity(label, a_hc, shared, None if not model.wing_idx
                                       else model.start_wing())
        else:
            cond_train, scaler = conditioning_of(train)
            cond = dict(cond_train, **conditioning_of(FOLD_UNITS[held_out], scaler)[0])
            if frozen is None:
                module = fit_history_neural(
                    train, bank, cond_of=cond, n_cond=N_COND,
                    net_fn=lambda: FiLMBeltNet(n_cond=N_COND), epochs=epochs, seed=seed,
                    target=fold_baseline_field(BASELINE_RESULT, held_out, train),
                    channels=channels, nu_max=nu_max)
                nu = module.nu().detach().cpu().numpy()
                log_c = float(module.log_c.detach())
                net = module.net
                in_nll.append(module.history[-1][1])
                n_params = int(sum(p.numel() for p in net.parameters())) + 1 + int(np.count_nonzero(nu))
            else:
                net, log_c, cond = frozen["net"], frozen["log_c"], frozen["cond"]
                nu, _ = fit_nu_only(bank, train,
                                    lambda h: as_intensity_fn(net, cond[h], log_c),
                                    channels=channels, nu_max=nu_max)
                in_nll.append(np.nan)
                n_params = int(sum(p.numel() for p in net.parameters())) + 1 + int(np.count_nonzero(nu))
            torch.save(dict(state=net.state_dict(), log_c=log_c,
                            cond={h: [float(v) for v in row] for h, row in cond.items()}),
                       HISTORY_CACHE / f"{key}__cycle{held_out}.pt")
            fold_params[str(held_out)] = dict(train=train, mu_b_clock=float(alignment["mu_b"]),
                                              nu=[float(v) for v in nu], amp_map=[0.0, 0.0],
                                              log_c=float(log_c))
            def base_for(label, a_hc):
                return as_intensity_fn(net, cond[label], log_c)
            intercept = slope = 0.0

        beta = beta_of(nu)
        for label in FOLD_UNITS[held_out]:
            a_hc = (float(np.exp(intercept + slope * np.log(AREA_AMPLITUDE[label])))
                    if base == "champion" else float(AREA_AMPLITUDE[label]))
            rows.append(dict(cycle=int(held_out),
                             **score_held_wing(label, alignment, base_for(label, a_hc), beta, a_hc,
                                               strictly_past=strictly_past, channels=channels)))

    result = dict(name=key, base=base, arm=arm, kind="history", question=ARM_QUESTION[arm],
                  conditional=False, seed=int(seed), folds=list(folds), epochs=epochs, lr=BEST_LR,
                  channels=list(channels), n_params=int(n_params),
                  in_sample=float(-np.nanmean(in_nll)) if np.any(np.isfinite(in_nll)) else float("nan"),
                  nu=[float(v) for v in np.mean([fold_params[str(c)]["nu"] for c in folds], axis=0)],
                  rows=rows, fold_params=fold_params, seconds=float(time.time() - started),
                  fingerprint=protocol_fingerprint(), run_tag=run_tag(base, arm, seed, epochs))
    with open(path, "w") as handle:
        json.dump(result, handle, indent=1)
    return result


def fold_base_of(result, cycle):
    """Rebuild one fold's fitted base from a cached history result, for a reading that freezes it."""
    fold = result["fold_params"][str(cycle)]
    nu = np.array(fold["nu"], dtype=float)
    out = dict(nu=nu, beta=beta_of(nu), mu_b=float(fold["mu_b_clock"]), train=fold["train"],
               amp_map=fold["amp_map"])
    intercept, slope = fold["amp_map"]
    out["a_of"] = ((lambda label: float(np.exp(intercept + slope * np.log(AREA_AMPLITUDE[label]))))
                   if result["base"] == "champion"
                   else (lambda label: float(AREA_AMPLITUDE[label])))
    if result["base"] == "champion":
        model = Model(CHAMPION_SPEC)
        shared = model.unpack(np.array(fold["q"]), fold["c"], fold["a_ref"])
        out.update(q=fold["q"], c=fold["c"], a_ref=fold["a_ref"],
                   amps={h: float(v) for h, v in fold["amps"].items()},
                   wing={h: np.array(v) for h, v in fold["wing"].items()},
                   base_of=lambda label, a_hc: model.intensity(label, a_hc, shared, None))
    else:
        saved = torch.load(HISTORY_CACHE / f"{result['name']}__cycle{cycle}.pt", weights_only=True)
        net = FiLMBeltNet(n_cond=N_COND)
        net.load_state_dict(saved["state"])
        cond = {h: np.array(row) for h, row in saved["cond"].items()}
        out.update(net=net, log_c=float(saved["log_c"]), cond=cond,
                   base_of=lambda label, a_hc=None: as_intensity_fn(net, cond[label],
                                                                    float(saved["log_c"])))
    return out


print(f"runner ready · arms {HISTORY_TAGS} · cache {HISTORY_CACHE}")

In [ ]:
# Given: the epoch budget, re-read on cycle 13 for *both* neural arms, because eight more parameters
# move a validation curve and reusing Week 5's number for one arm only would be two knobs moving at
# once. Cycle 13 is never a scored fold. About 40 s against the week's most likely false positive.
BUDGET_BANK = history_of(EVENTS, BUDGET_WINGS)
VALID_BANK = history_of(EVENTS, VALID_WINGS)
COND_FULL = dict(COND)

BUDGETS, BUDGET_RUNS = {}, {}
for arm in NEURAL_ARMS:
    started = time.time()
    BUDGET_RUNS[arm] = fit_history_neural(
        BUDGET_WINGS, BUDGET_BANK, cond_of=COND_FULL, n_cond=N_COND,
        net_fn=lambda: FiLMBeltNet(n_cond=N_COND), epochs=EPOCHS, monitor=VALID_WINGS,
        monitor_bank=VALID_BANK, channels=ARM_CHANNELS[arm])
    best = min(BUDGET_RUNS[arm].monitor_history, key=lambda row: row[1])
    BUDGETS[arm] = best[0] + 1
    flag = " (the cap, not the curve)" if BUDGETS[arm] >= EPOCHS else ""
    print(f"   neural '{arm}': budget {BUDGETS[arm]:4d} epochs, monitored {best[1]:9.4f} "
          f"on cycle {VALID_CYCLE}{flag} · {time.time() - started:4.0f} s")

print(f"\nWeek 5's FiLM budget on the same validation cycle was {FILM_BUDGET_W05} epochs; these are "
      f"re-read because beta moves the curve.")

In [ ]:
# Given: the resolution floor, declared BEFORE anything is compared, which is the only time declaring it
# means anything. Week 5 measured the spread of a neural score across seeds 42-44: 1.0019 nats per
# hemicycle-year on three folds, 0.2844 on nine. Below the larger of that and the paired standard error
# is NO DETECTION, not a small effect.
SEED_SPREAD = 1.0019 if len(FOLD_CYCLES) <= 3 else 0.2844


def resolution_floor(stats):
    """The larger of a comparison's own paired standard error and the seed spread."""
    return float(max(stats["se"] if np.isfinite(stats["se"]) else 0.0, SEED_SPREAD))


print(f"resolution floor: seed spread {SEED_SPREAD:.4f} nats per hemicycle-year on "
      f"{len(FOLD_CYCLES)} fold cycles, or the paired standard error, whichever is larger")

In [ ]:
# Given: the fold runs this week reads three different ways. Sections 3, 4 and 5 are three readings of
# THIS cached set, not three experiments — so it is fitted once, here, and nothing below refits it.
RUNS = {}
for arm in ("nohist", "short", "bank"):
    started = time.time()
    RUNS[("champion", arm)] = run_history("champion", arm)
    print(f"   champion / {arm:8s} {RUNS[('champion', arm)]['seconds']:5.0f} s "
          f"({time.time() - started:4.0f} s this run) · {ARM_QUESTION[arm]}")
for arm in NEURAL_ARMS:
    started = time.time()
    RUNS[("film", arm)] = run_history("film", arm, epochs=BUDGETS[arm])
    print(f"   film     / {arm:8s} {RUNS[('film', arm)]['seconds']:5.0f} s "
          f"({time.time() - started:4.0f} s this run) · {ARM_QUESTION[arm]}")

print()
for (base, arm), result in RUNS.items():
    table = rows_of(result)
    print(f"{base:9s} {arm:9s} {result['n_params']:6,d} fitted · held out "
          f"{table[f'nll/yr ({HEADLINE})'].mean():9.4f} · n = {np.sum(result['nu']):.4f} "
          f"offspring per event")

# The stability condition, checked here rather than discovered in Section 7. A fitted n at or above 1
# is not a bug and not a bigger result: it says the history term is being used as a rate booster, which
# is what happens when the base has too few wings to pin its own amplitude down.
PRESSING = {(b, a): float(np.sum(r["nu"])) for (b, a), r in RUNS.items()
            if np.sum(r["nu"]) > 0.9 * N_CAP}
if PRESSING:
    print(f"\n*** these arms are pressing the stability bound (n cap = {N_CAP}):")
    for (b, a), value in PRESSING.items():
        print(f"      {b} / {a}: n = {value:.4f}")
    print("    A fit against the bound reports the bound, not a measurement: the history term is "
          "supplying rate\n    rather than describing triggering. Expected on `taste`; a warning on `full`.")
else:
    print(f"\nno arm presses the stability bound "
          f"(largest n = {max(np.sum(r['nu']) for r in RUNS.values()):.4f} against a cap of {N_CAP}), "
          f"so every fitted process is stable and Section 7 can generate from any of them")

In [ ]:
# Given: Week 6's headline, reproduced. The carried stack above is not a copy of Week 6's words, it is
# Week 6's code, so this number is a real check and not a quotation.
W06_CHAMP_GAIN, W06_CHAMP_SE = 2.5895, 0.4286
W06_FILM_GAIN, W06_FILM_SE = 0.3750, 0.2328
_champ = paired_stats(RUNS[("champion", "bank")], RUNS[("champion", "nohist")])
print(f"Week 6 champion: history worth {-_champ['mean']:+.4f} +/- {_champ['se']:.4f} nats/hemicycle-yr "
      f"(Week 6 reported {W06_CHAMP_GAIN:+.4f} +/- {W06_CHAMP_SE:.4f})")
if ("film", "bank") in RUNS:
    _film = paired_stats(RUNS[("film", "bank")], RUNS[("film", "nohist")])
    print(f"Week 6 neural  : history worth {-_film['mean']:+.4f} +/- {_film['se']:.4f}, against a seed "
          f"floor of {SEED_SPREAD:.4f} — still below it")
_nu = np.asarray(RUNS[("champion", "bank")]["nu"], dtype=float)
print(f"offspring per event, 2 d and 5 d channels: "
      f"{100.0 * np.sum(_nu[list(SHORT_CHANNELS)]) / max(np.sum(_nu), 1e-12):.1f}% of the total, "
      f"all the rest on {CHANNEL_NAMES[int(np.argmax(_nu))]} — the result that sent us to longitude")

In [ ]:
# Given: the carried Week 6 open question that is cheap to close here — an error bar on the eight
# coefficients. Week 6 reported the 60 d / 3 deg coefficient as its whole headline with no uncertainty,
# and flagged the beta-c trade as its main identifiability risk. A finite-difference Hessian of the beta
# block, with c and the base frozen, gives both the error bars and the correlations that trade is made of.
def beta_block_errors(result, cycle=None, step=0.08):
    """Finite-difference Hessian of the NLL in the eight beta coefficients, at the fitted point.

    One fold's training wings, with the base and `c` frozen at what that fold fitted. The base's mesh
    integral does not depend on beta, so it is left out: a constant cannot change a Hessian.
    """
    cycle = result["folds"][0] if cycle is None else cycle
    fold = fold_base_of(result, cycle)
    train = fold["train"]
    events, _ = fold_events(train)
    bank = history_of(events, train, channels=ALL_CHANNELS)
    pieces = []
    for label in train:
        fn = fold["base_of"](label, fold["a_of"](label))
        at = np.asarray(fn(bank[label]["lat"], bank[label]["s_true"]), dtype=float)
        pieces.append((at, bank[label]["H"], bank[label]["A"]))
    beta = np.asarray(fold["beta"], dtype=float)

    def nll(b):
        b = np.asarray(b, dtype=float)
        total = 0.0
        for at, H, A in pieces:
            lam = at + H @ b
            if np.any(lam <= 0.0) or not np.all(np.isfinite(lam)):
                return 1.0e12
            total -= float(np.sum(np.log(lam)) - A @ b)
        return total

    n = len(beta)
    scale = np.where(beta > 0, np.abs(beta) * step, step * np.max(np.abs(beta) + 1e-12))
    hess = np.zeros((n, n))
    f0 = nll(beta)
    for i in range(n):
        for j in range(i, n):
            ei, ej = np.zeros(n), np.zeros(n)
            ei[i], ej[j] = scale[i], scale[j]
            fpp, fpm = nll(beta + ei + ej), nll(beta + ei - ej)
            fmp, fmm = nll(beta - ei + ej), nll(beta - ei - ej)
            hess[i, j] = hess[j, i] = (fpp - fpm - fmp + fmm) / (4.0 * scale[i] * scale[j])
    # a Hessian this crude can come back slightly indefinite; keep only the directions it resolves
    evals, evecs = np.linalg.eigh(hess)
    keep = evals > 1e-9 * max(evals.max(), 1e-12)
    cov = (evecs[:, keep] / evals[keep]) @ evecs[:, keep].T
    sd = np.sqrt(np.clip(np.diag(cov), 0.0, None))
    corr = cov / np.outer(np.where(sd > 0, sd, 1.0), np.where(sd > 0, sd, 1.0))
    return dict(beta=beta, beta_se=sd, nu=beta * CHANNEL_MASS, nu_se=sd * CHANNEL_MASS,
                corr=corr, f0=f0, cycle=int(cycle))


BETA_ERR = beta_block_errors(RUNS[("champion", "bank")])
_lead = int(np.argmax(BETA_ERR["beta"] * CHANNEL_MASS))
print(f"Week 6 open question #2, closed on fold {RUNS[('champion', 'bank')]['folds'][0]}: the "
      f"{CHANNEL_NAMES[_lead]} channel carries nu = {BETA_ERR['nu'][_lead]:.4f} "
      f"+/- {BETA_ERR['nu_se'][_lead]:.4f} offspring per event, with a largest off-diagonal "
      f"correlation of {np.abs(BETA_ERR['corr'] - np.eye(N_CHANNELS)).max():.3f} — the beta-c trade")

In [ ]:
# Given: the rotating frame, as arithmetic. Carrington longitude is FIXED TO THE SUN: it is measured
# from a meridian that turns with the Sun, so a feature that stays put on the surface keeps the same
# Carrington longitude forever. What moves is the sub-Earth point — the middle of the disk we can see —
# whose Carrington longitude `L0` falls by 13.1992 degrees a day and comes back round in 27.3 days.
#
# Only the arithmetic is given. Task 68 is where the frame meets our events, and that is where a sign
# error or a wrong epoch would do real damage.
def jd_from_decimal_year(t):
    """Julian Date from the decimal year our clock is built on. Exact to a microsecond."""
    t = np.asarray(t, dtype=float)
    year = np.floor(t).astype(int)
    leap = ((year % 4 == 0) & (year % 100 != 0)) | (year % 400 == 0)
    length = np.where(leap, 366.0, 365.0)
    jan1 = (pd.to_datetime(pd.Series(year), format="%Y") - pd.Timestamp("1858-11-17")
            ).dt.total_seconds().to_numpy() / 86400.0 + MJD_OFFSET
    return jan1 + (t - year) * length


def carrington_l0(jd):
    """Carrington longitude of the central meridian — the sub-Earth point — at Julian Date `jd`."""
    rotations = (np.asarray(jd, dtype=float) - CARRINGTON_EPOCH_JD) / CARRINGTON_PERIOD
    return np.mod(360.0 * (1.0 - np.mod(rotations, 1.0)), 360.0)


def wrap180(delta_deg):
    """Any angular difference folded into (-180, +180]. Negative = east of the reference meridian."""
    return ((np.asarray(delta_deg, dtype=float) + 180.0) % 360.0) - 180.0


dev["jd"] = jd_from_decimal_year(dev["t"].to_numpy())
dev["L0"] = carrington_l0(dev["jd"].to_numpy())
assert dev["longitude"].notna().all(), "the composite is missing a longitude on the analysis set"
print(f"frame: rotation 1 at JD {CARRINGTON_EPOCH_JD}, period {CARRINGTON_PERIOD} d, L0 sweeping "
      f"{SYNODIC_DEG_PER_DAY} deg/day · {dev['longitude'].nunique():,} distinct longitudes, so there "
      f"is no lattice for a sharp kernel to latch onto")

In [ ]:
# Given: the drawing primitives for the lead figure, and the wing it is drawn on. `STRONGEST` is the
# biggest development wing, already chosen by the carried code above.
STROBE_DAYS = 220.0                # a window long enough to show eight rotations of the visibility arc


def draw_carrington_strobe(ax, frame, t_start, days=STROBE_DAYS):
    """Longitude against date for one window, with the central meridian's sawtooth drawn through it."""
    jd0 = jd_from_decimal_year(np.array([t_start]))[0]
    sel = frame[(frame["jd"] >= jd0) & (frame["jd"] < jd0 + days)]
    grid = np.linspace(jd0, jd0 + days, int(days) * 6)
    track = carrington_l0(grid)
    cut = np.abs(np.diff(track)) > 180.0            # break the line where L0 wraps through zero
    seg_jd, seg_l0 = np.insert(grid.astype(float), np.where(cut)[0] + 1, np.nan), \
        np.insert(track, np.where(cut)[0] + 1, np.nan)
    ax.plot(seg_jd - jd0, seg_l0, lw=1.0, color="tab:red", alpha=0.8,
            label="central meridian $L_0$")
    ax.scatter(sel["jd"] - jd0, sel["longitude"], s=16, color="tab:blue", alpha=0.75,
               edgecolor="none", label="first observations")
    ax.set_xlabel(f"days from {t_start:.1f}")
    ax.set_ylabel(r"Carrington longitude $\phi$ [deg]")
    ax.set_ylim(0, 360)
    ax.set_title("(a) events track the sub-Earth point", fontsize=10)
    ax.legend(fontsize=7, loc="upper right", framealpha=0.9)


def draw_cmd_scatter(ax, frame):
    """Central-meridian distance against the clock, for one whole wing."""
    cmd = wrap180(frame["longitude"].to_numpy() - frame["L0"].to_numpy())
    ax.scatter(frame["s"], cmd, s=10, color="tab:purple", alpha=0.5, edgecolor="none")
    ax.axhline(0.0, color="tab:red", lw=1.2)
    ax.axhspan(-90, 90, color="0.85", zorder=0)
    ax.set_xlabel("s [yr]")
    ax.set_ylabel("central-meridian distance [deg]")
    ax.set_ylim(-180, 180)
    ax.set_title(f"(b) {100.0 * (cmd < 0).mean():.0f}% are caught east of centre", fontsize=10)


def draw_butterfly_longitude(ax, frame):
    """The familiar wing, with longitude as the new third coordinate in colour."""
    art = ax.scatter(frame["s"], frame["abs_latitude"], c=frame["longitude"], s=9,
                     cmap="twilight", vmin=0, vmax=360, alpha=0.85, edgecolor="none")
    ax.set_xlabel("s [yr]")
    ax.set_ylabel(r"$|\lambda|$ [deg]")
    ax.set_title("(c) the same wing, coloured by longitude", fontsize=10)
    return art

In [ ]:
# Given: Figure 1 — the week in one figure. Panel (a) carries the whole argument: the groups are not
# scattered over the Sun, they crowd along a diagonal, and that diagonal is our own line of sight.
# Panel (b) measures the same crowding. Panel (c) is the butterfly diagram the program has drawn since
# Week 1, now coloured by longitude, and it should look like nothing at all: no wing in longitude.
_strobe_wing = dev[dev["hemicycle"] == STRONGEST]
_t_start = float(_strobe_wing["t"].quantile(0.45))
fig, axes = plt.subplots(1, 3, figsize=(15.0, 4.1))
draw_carrington_strobe(axes[0], _strobe_wing, _t_start)
draw_cmd_scatter(axes[1], _strobe_wing)
_art = draw_butterfly_longitude(axes[2], _strobe_wing)
fig.colorbar(_art, ax=axes[2], label=r"$\phi$ [deg]", pad=0.02)
fig.suptitle(f"Figure 1 — hemicycle {STRONGEST}: the Sun's longitudes, seen through a rotating window")
fig.tight_layout()
plt.show()

---
## 2 · The instrument, measured

A group's **Carrington longitude** is fixed to the Sun: the coordinate is measured from a meridian
that turns with the Sun, so a feature that does not move across the surface keeps the same longitude
forever. What moves is us. The point facing Earth — its Carrington longitude is `L0(t)` — moves on at
13.1992° a day, so it comes back round every 27.3 days.

A group enters the catalogue when somebody sees it, which can only happen while its longitude faces
Earth. So our table of first observations is a sample taken through a moving window: think of a
spotlight sweeping a dark stage, and us only able to list the actors it happens to light. That window
is not a nuisance to correct away. It is a **selection function** — a probability of being detected
that depends on where you are — so we measure it and put it in the model. The section headings and the
code call this window the *aperture*, the way a telescope's aperture decides what reaches the film.

Its natural coordinate is the **central meridian distance**, `CMD = φ − L0(t)`: how far a group (at
longitude `φ`) is from the middle of the visible disk when it is first recorded. Negative is east, the
side rotating *into* view, so most first observations are negative — a group is usually caught on its
way in rather than on its way out.

```text
TUTOR DIRECTIVE — Section 2. The aperture is the instrument.

Destinations, in this order. Do not state them; get the student to them.

1. [ESSENTIAL] A group's Carrington longitude is fixed to the Sun. What moves is us. Make them say
   which of the two is rotating in the phrase "the visibility window rotates".
2. [ESSENTIAL] Detection is CONDITIONAL on facing Earth, so the catalogue's longitude distribution is
   a product of what the Sun did and what we could see. Ask what the CMD histogram would look like if
   the Sun emerged evenly in longitude — they should reach "not flat", and that is the whole section.
   Figure 3(c) draws the geometry; reach for it if the words are not landing.
3. [IF TIME] Why CMD and not longitude: at a single moment they differ by a constant, but over a
   century longitude averages the window away and CMD does not. The window is only visible in CMD.
4. [IF TIME] Task 70 chooses a kernel width by cross-validation. Ask why it may not be chosen by eye.
   They do not need the answer yet — Section 4 measures it — but the question has to be live first.

Do NOT discuss what the aperture is worth in nats. That is Section 4's prediction and naming a number
here destroys it.
```

In [ ]:
raise NotImplementedError("Task 68: CMD, the mark table, and the geometry of the viewing window")
# Reference workflow:
#   1. `add_cmd`: subtract `L0` from `longitude`, then fold the answer into (-180, +180] with `wrap180`.
#      Get the sign the right way round and check it against the figure below: the great majority of
#      first observations must come out NEGATIVE, because groups are caught as they rotate into view.
#   2. `marks_of`: one dict per hemicycle holding that wing's mark coordinates. Build it from the same
#      per-hemicycle selection `EVENTS` used, and keep a copy of every angle in radians — the whole
#      likelihood works in radians, and converting halfway through it is this week's 4-nat blunder.
#   3. `aperture_geometry`: the median |CMD| and the fraction east of centre. Two numbers, and they are
#      the evidence that there is a viewing window at all.
#   4. `draw_disk_geometry`: those two numbers as a picture — the visible disk, the central meridian, the
#      east limb that turns into view, and the median |CMD| marked on it. It becomes Figure 3(c).
#   What you should see: a median |CMD| near 65 degrees, and about 83% of first observations east of
#      centre. If that fraction comes out near 17%, your sign convention is the wrong way round.

In [ ]:
# Put your code for Task 68 here

In [ ]:
# Given: the mark table, with the geometry checked against the values measured when this week was
# written. If these assertions fail, the frame is rotated and everything after this point is wrong with
# no error message to tell you so.
dev = add_cmd(dev)
MARKS = marks_of(dev, DEV_HEMICYCLES)
GEOMETRY = aperture_geometry(dev["cmd"].to_numpy())
assert all(len(MARKS[h]["lon"]) == len(EVENTS[h][0]) for h in DEV_HEMICYCLES), \
    "the mark table and EVENTS disagree on how many events a wing has"
assert all(np.allclose(MARKS[h]["abs_lat"], EVENTS[h][0]) for h in DEV_HEMICYCLES), \
    "the mark table and EVENTS are in different orders"
assert 60.0 < GEOMETRY["median_abs"] < 70.0 and GEOMETRY["east_fraction"] > 0.75, \
    f"the frame looks wrong: {GEOMETRY}"
PER_WING_GEOM = {h: aperture_geometry(MARKS[h]["cmd"]) for h in DEV_HEMICYCLES}
_med = np.array([g["median_abs"] for g in PER_WING_GEOM.values()])
_east = np.array([g["east_fraction"] for g in PER_WING_GEOM.values()])
print(f"median |CMD| {GEOMETRY['median_abs']:.1f} deg and {100 * GEOMETRY['east_fraction']:.1f}% east "
      f"over {GEOMETRY['n']:,} events; across {len(DEV_HEMICYCLES)} wings "
      f"{_med.mean():.1f} +/- {_med.std():.1f} deg and {_east.mean():.3f} +/- {_east.std():.3f} "
      f"— stable enough to pool a fold's aperture")

In [ ]:
raise NotImplementedError("Task 69: Fit the rotation rate the catalogue actually shows")
# Reference workflow:
#   1. Rebuild `jd`, `L0` and `cmd` on the *daily* observations — this test needs a group's whole track,
#      not just its first row — then keep groups with at least five rows spanning at least five days.
#   2. For each, `np.unwrap` the CMD series in radians before fitting: a group followed for ten days
#      sweeps about 130 degrees, so it crosses the +/-180 seam, and unwrapping takes out the jump the
#      seam puts in the series. Fit a straight line and keep the slope, in degrees per day.
#   3. Keep each group's mean `|latitude|` and its cycle alongside the slope. `rotation_profile` then
#      converts the slopes to sidereal (add 0.9856) and fits a straight line in `sin^2(lat)`, which is
#      the standard two-parameter differential-rotation law. The `cycles` argument is what lets a fold
#      re-derive the profile from its training cycles only — do not skip it.
#   4. Draw the slopes as a histogram with three vertical lines on it: the synodic rate the frame
#      assumes, the measured median, and the sidereal rate — which is the wrong answer here, and seeing
#      it sit clearly off the peak is what makes the test worth doing. Beside it, draw one group's
#      unwrapped track with the predicted slope through its first point.
#   5. Draw rate against latitude with the fitted profile over it. A rigid frame would be a horizontal
#      line, and how far the points depart from it at low and high latitude is what Section 6 uses.
#   What you should see: a tight peak within a few tenths of a degree per day of 13.1992, and a profile
#      that falls by a fraction of a degree per day between the equator and 30 degrees of latitude.

In [ ]:
# Put your code for Task 69 here

In [ ]:
# Given: Figure 2 — the frame checked against the data rather than taken on trust. This is the cheapest
# test in the notebook, and the only thing standing between us and an analysis rotated the wrong way.
_peak = obs.groupby("uniqueID")["correctedArea"].max()
_obs_dev = obs[obs["CYCLE"].isin(DEV_CYCLES)
               & obs["uniqueID"].isin(_peak[_peak > PEAK_AREA_MIN].index)]
TRACK = tracked_rotation_rates(_obs_dev)
ROT_PROFILE = rotation_profile(TRACK)
_PROFILE_CACHE = {}


def fold_profile(frame, cycle):
    """The advection profile one fold may use, re-derived from that fold's TRAINING cycles only.

    `rigid` returns None, which is how Section 6 scores the Carrington frame as a variant rather than
    as an assumption.
    """
    if frame == "rigid":
        return None
    key = (str(frame), int(cycle))
    if key not in _PROFILE_CACHE:
        _PROFILE_CACHE[key] = rotation_profile(TRACK, cycles=[c for c in DEV_CYCLES if c != cycle])
    return _PROFILE_CACHE[key]


fig, axes = plt.subplots(1, 3, figsize=(15.5, 4.0))
draw_rate_histogram(axes[0], TRACK["rate"])
_gid, (_d, _trk) = next(iter(TRACK["examples"].items()))
draw_rate_example(axes[1], _d, _trk)
draw_rate_profile(axes[2], TRACK, ROT_PROFILE)
fig.suptitle("Figure 2 — the rotating frame, measured from the groups it is supposed to describe")
fig.tight_layout()
plt.show()

_q25, _q75 = np.percentile(TRACK["rate"], [25, 75])
_drift = [(ROT_PROFILE[0] + ROT_PROFILE[1] * np.sin(np.radians(L)) ** 2 - SIDEREAL_DEG_PER_DAY)
          * CARRINGTON_PERIOD for L in (5.0, 25.0)]
print(f"rotation rate: median {np.median(TRACK['rate']):.2f} deg/day (IQR {_q25:.2f}-{_q75:.2f}) on "
      f"{len(TRACK['rate']):,} groups against {SYNODIC_DEG_PER_DAY} assumed · Omega = "
      f"{ROT_PROFILE[0]:.3f} {ROT_PROFILE[1]:+.3f} sin^2(lat) sidereal, so a feature drifts "
      f"{_drift[0]:+.1f} deg per rotation at 5 deg and {_drift[1]:+.1f} at 25 deg")

In [ ]:
raise NotImplementedError("Task 70: Measure the viewing window, and cross-validate its width")
# Reference workflow:
#   1. `log_von_mises`: the circular version of a Gaussian, with a concentration `kappa` in place of a
#      width (`sigma` is roughly `1/sqrt(kappa)` radians). Build it in LOG space using `ive`, so that a
#      sharp kernel does not overflow.
#   2. `aperture_logdensity`: average the kernel over every observed CMD and take the log. Add the sum up
#      in chunks so the kernel matrix never gets large.
#   3. `aperture_loo`: the same average, but for each event leave that event's own kernel out of it, and
#      return the mean log density. Leaving it out — *leave-one-out* — is the only honest way to compare
#      widths: otherwise the sharpest kernel always wins by sitting on top of each event.
#   4. `aperture_bandwidth`: run the leave-one-out score over `APERTURE_KAPPAS` and return the best, plus
#      the whole curve — Section 4 draws it, and what the curve does matters as much as which width wins.
#   5. Two drawing primitives: the fitted window over the CMD histogram, and every wing's own histogram
#      overlaid. Remember the density is per radian while the axis is in degrees.
#   What you should see: a sharp peak well to the EAST of disk centre, around -75 degrees, with a long
#      tail running west — not a hump centred on zero — and eighteen wings lying almost on top of each
#      other, because the instrument barely changes over a century.

In [ ]:
# Put your code for Task 70 here

In [ ]:
# Given: Figure 3 — the instrument, measured. The bandwidth is cross-validated on the whole development
# set here, because this is a measurement of the window and nothing is being scored against it yet; from
# Section 4 on, `g` is rebuilt inside every fold from that fold's training cycles only. Panel (c) is the
# same geometry drawn on the disk, from the two numbers Task 68 measured.
CMD_ALL_RAD = np.radians(dev["cmd"].to_numpy())
APERTURE_CV = aperture_bandwidth(CMD_ALL_RAD)
KAPPA_G = APERTURE_CV["kappa"]
LOG_G_POOLED = aperture_logdensity(CMD_ALL_RAD, KAPPA_G)

fig, axes = plt.subplots(1, 3, figsize=(16.0, 4.2))
draw_aperture(axes[0], dev["cmd"].to_numpy(), LOG_G_POOLED)
draw_aperture_wings(axes[1], MARKS, DEV_HEMICYCLES)
draw_disk_geometry(axes[2], GEOMETRY)
fig.suptitle("Figure 3 — the visibility aperture, how little it varies, and the geometry behind it")
fig.tight_layout()
plt.show()

print(f"cross-validated aperture: kappa = {KAPPA_G:.0f} (sigma ~ "
      f"{np.degrees(1.0 / np.sqrt(KAPPA_G)):.1f} deg) scoring {APERTURE_CV['score']:+.4f} nats/event "
      f"against uniform {UNIFORM_LOGF:+.4f} — hold on to that difference, Section 4 wants it")

---
## 3 · What the aperture does to the raw signal

This section comes first for a reason. It uses no model of longitude at all, so nothing it finds can
be an artefact of a model we chose, which makes it the most robust reading in the week.

The statistic the literature uses is simple: of all the pairs of events in a wing, what fraction land
within 20° of each other in longitude? If longitude carried no information at all that fraction would
be `2 × 20 / 360 = 0.111`, and the pairs are then split by the time lag between them.

The question is what to compare it against. The published comparison redraws every longitude uniformly
around the circle, which throws the moving window away: the data was taken through the window and the
null was not, so the comparison mostly measures our telescope. (Count cars only while one streetlamp is
lit, and a null that scatters cars over the whole town will always report a crowd under the lamp.) A
**randomization null** must preserve every selection effect it is not testing, so ours keeps each
event's time exactly where it is and redraws only its CMD, from its own wing's CMD values.

```text
TUTOR DIRECTIVE — Section 3. A null that destroys what it should preserve.

REQUIRED OUTPUT: the student writes down, before running anything, what they expect the
aperture-preserving null to do to the raw co-location curve. Not a number — a shape. Push until they
commit to one of: "nothing much", "it flattens the short lags", "it flattens everything".

Destinations:
1. [ESSENTIAL] The rotating window ALREADY produces co-location with no clustering on the Sun at all:
   two groups seen within a day of each other were both near the same visible meridian, so they were
   near each other in longitude. Get them to construct that argument themselves. Figure 4(c) draws the
   same geometry — use it to check the argument they built, never as a substitute for building it.
2. [ESSENTIAL] A null is a hypothesis about what is being tested, and a uniform-longitude null tests
   "longitude carries no information at all" — including the information that we can only see half the
   Sun. That is not the hypothesis anyone wanted to test.
3. [IF TIME] The lag scale is the giveaway: the ratios alternate with a period of about 27 days. A
   physical triggering process has no reason to know the synodic rotation period. Our telescope does.
4. [IF TIME] Where resampling CMD within the wing is still not enough — it keeps the window and the
   rate, but no real time-of-year or observer-coverage structure. Name it as a limit, not a fix.

When they see 3.20 go to 0.937, the move to resist is "so there is nothing there". Panel 5(c) is why:
it averages a core the catalogue suppresses against a ring it does not. Section 7 asks which is real.
```

In [ ]:
raise NotImplementedError("Task 71: Co-location against time lag, as the literature measures it")
# Reference workflow:
#   1. `colocation_counts`: for every ordered pair of events in one wing, work out the time lag and the
#      longitude separation, then count — per lag bin — how many pairs there are, and how many of those
#      fall inside `window`: either an outer edge, or a `(lo, hi)` ring. Build it in blocks of rows so
#      the all-pairs arrays stay small.
#   2. Support an optional set of latitude-separation edges that splits each lag bin further. Task 73
#      needs exactly that and should not need a second function.
#   3. `colocation_by_wing`: the same counts per hemicycle. Every ratio in this section is averaged over
#      wings, never pooled over all events, because the hemispheric cycle is the replication unit.
#   4. `draw_colocation`: ratio against lag on a log x-axis, with a line at 1.0 and the first two
#      multiples of the 27.3-day rotation period marked. Those marks are what turn a wiggly curve into a
#      specific accusation: a process on the Sun has no reason to care about our rotation period.
#   What you should see: a curve well above 1 at the shortest lags, and bumps near 27 and 54 days.

In [ ]:
# Put your code for Task 71 here

In [ ]:
# Given: the raw reading, with no null yet. `UNIFORM_CO_FRAC` is what the fraction would be if longitude
# carried no information at all, which is the comparison the published work draws.
LAG_CENTRES = np.array([0.5 * (b[0] + b[1]) for b in LAG_BINS])
OBS_COUNTS = colocation_by_wing(MARKS, DEV_HEMICYCLES)


def wing_mean_ratio(counts, reference=None):
    """Mean over wings of each wing's own ratio, with the standard error of that mean."""
    rows = []
    for label, (hits, pairs) in counts.items():
        ok = pairs > 200                                   # a wing with too few pairs says nothing
        frac = np.where(ok, hits / np.maximum(pairs, 1.0), np.nan)
        denom = UNIFORM_CO_FRAC if reference is None else reference[label]
        rows.append(frac / denom)
    table = np.array(rows, dtype=float)
    with warnings.catch_warnings():
        warnings.simplefilter("ignore", category=RuntimeWarning)
        mean = np.nanmean(table, axis=0)
        se = np.nanstd(table, axis=0, ddof=1) / np.sqrt(np.sum(np.isfinite(table), axis=0))
        above = np.nansum(table > 1.0, axis=0)
        counted = np.sum(np.isfinite(table), axis=0)
    return dict(mean=mean, se=se, above=above, counted=counted, table=table)


RAW_RATIO = wing_mean_ratio(OBS_COUNTS)
print("raw co-location as a multiple of chance (the alternation period is the solar rotation): "
      + "  ".join(f"{b[0]}-{b[1]}d:{v:.2f}" for b, v in zip(LAG_BINS, RAW_RATIO["mean"])))

In [ ]:
raise NotImplementedError("Task 72: A null that keeps the window and destroys only the coupling")
# Reference workflow:
#   1. `aperture_null_longitudes`: for one wing, resample its own CMD values with replacement and add
#      them back onto each event's `L0`. Times are left exactly where they are. Think about why the
#      resampling has to happen inside the wing rather than across the whole development set.
#   2. `null_reference`: average the null's co-location fraction over `N_NULL` replicates, per wing and
#      lag bin. Use a seeded generator per replicate so the result is reproducible.
#   3. Divide the observed fraction by the null's, per wing, and average over wings — that is what
#      `wing_mean_ratio` does when you hand it a reference.
#   4. Two primitives: both readings on one axis, and every wing's own residual. The second is the one
#      that decides whether a mean of 1.19 is eighteen wings agreeing or three wings shouting.
#   What you should see: a ratio defined in every lag bin, and null replicates that scatter tightly
#      around each other. What the new null does to the short lags is yours to predict first.

In [ ]:
# Put your code for Task 72 here

In [ ]:
# Given: Figure 4 — the week's most robust result, and the one that most published work on this subject
# gets wrong. Panel (c) is why the null matters: the geometry on its own puts two same-day groups close
# together in longitude, with nothing happening on the Sun at all.
NULL_REF = null_reference(MARKS, DEV_HEMICYCLES)
PRESERVED_RATIO = wing_mean_ratio(OBS_COUNTS, reference=NULL_REF)


def draw_window_overlap(ax, days=1.0, geometry=None):
    """The same polar view as Figure 3(c), one day apart. No data in this panel: it is geometry."""
    geometry = GEOMETRY if geometry is None else geometry
    med = geometry["median_abs"]
    step = SYNODIC_DEG_PER_DAY * days          # L0 falls by this much, so the window turns clockwise
    ring = np.linspace(0.0, 2.0 * np.pi, 361)
    ax.plot(np.cos(ring), np.sin(ring), color="0.82", lw=1.0)
    wedge = np.radians(np.linspace(-med - CO_LON_DEG, -med + CO_LON_DEG, 80))
    ax.fill(np.r_[0.0, np.cos(wedge)], np.r_[0.0, np.sin(wedge)], color="0.7", alpha=0.45,
            label=f"$\\pm${CO_LON_DEG:.0f}$^\\circ$: the co-location window")
    for offset, colour, radius, name in ((0.0, "tab:blue", 1.0, "day 0"),
                                         (-step, "tab:orange", 1.11, f"day {days:.0f}")):
        arc = np.radians(np.linspace(-90.0, 90.0, 181) + offset)
        ax.plot(radius * np.cos(arc), radius * np.sin(arc), color=colour, lw=3.5,
                solid_capstyle="butt", label=f"{name}: the half facing Earth")
        ang = np.radians(offset - med)
        ax.plot([radius * np.cos(ang)], [radius * np.sin(ang)], "o", ms=10, color=colour)
    mid = np.radians(-med - 0.5 * step)
    ax.annotate(f"{step:.1f}$^\\circ$ apart", xy=(1.06 * np.cos(mid), 1.06 * np.sin(mid)),
                xytext=(1.12, -1.16), fontsize=9.5,
                arrowprops=dict(arrowstyle="->", color="k", lw=1.0))
    ax.set_xlim(-1.45, 2.05)
    ax.set_ylim(-1.50, 1.30)
    ax.set_aspect("equal")
    ax.axis("off")
    ax.legend(fontsize=7, loc="lower left", framealpha=0.9)
    ax.set_title("(c) where a group is typically first caught,\non two days running", fontsize=10)


fig, axes = plt.subplots(1, 3, figsize=(16.5, 4.4))
draw_null_overlay(axes[0], LAG_CENTRES, RAW_RATIO, PRESERVED_RATIO)
draw_null_wings(axes[1], LAG_CENTRES, PRESERVED_RATIO)
draw_window_overlap(axes[2])
fig.suptitle("Figure 4 — co-location in longitude, before and after the rotating window is preserved")
fig.tight_layout()
plt.show()

_sig = [(b, m, s, a, c) for b, m, s, a, c in zip(
    LAG_BINS, PRESERVED_RATIO["mean"], PRESERVED_RATIO["se"],
    PRESERVED_RATIO["above"], PRESERVED_RATIO["counted"]) if abs(m - 1.0) > 2.0 * s]
print(f"0-1 d: {RAW_RATIO['mean'][0]:.2f}x chance becomes "
      f"{PRESERVED_RATIO['mean'][0]:.3f} +/- {PRESERVED_RATIO['se'][0]:.3f} against the null; "
      f"surviving beyond two standard errors: "
      + ", ".join(f"{b[0]}-{b[1]} d {m:.3f}+/-{s:.3f} ({(m - 1) / s:+.1f} se, {a:.0f}/{c})"
                  for b, m, s, a, c in _sig))

In [ ]:
raise NotImplementedError("Task 73: Where the surviving excess actually lives")
# Reference workflow:
#   1. Call Task 71's counter with `DLAT_EDGES`, and the null the same way, so both come back split by
#      latitude separation as well as by lag. Take the ratio per wing and average over wings.
#   2. For the three lag bins that survived Figure 4, add up the *excess* pair count — observed minus
#      null — and work out what share of it sits in each latitude-separation band.
#   3. Draw the ratio as an image over lag and latitude separation, with a diverging colour map centred
#      on 1.0, and the share as a bar chart beside it.
#   4. `decompose_radius`: call the counter twice more, with `window=(0, 8)` and `window=(8, 20)`, and
#      the null the same way. The published work reports one number for the whole 20-degree window;
#      these are its two halves — a core and a ring around it — and they do not have the same sign.
#   5. Note what this costs you later: an excess that is tight in longitude AND tight in latitude is
#      exactly what a recurring nest looks like, and also exactly what one region catalogued twice looks
#      like. Section 7 is where that has to be paid for.
#   What you should see: a core and a ring that disagree with each other. One number for the whole
#      window averages them together and reports almost nothing.

In [ ]:
# Put your code for Task 73 here

In [ ]:
# Given: Figure 5 — what is left after the null, broken apart. Panels (a) and (b) together set up
# Section 7: whatever survived is close in longitude AND close in latitude at the same time.
RESIDUAL = decompose_residual(MARKS, DEV_HEMICYCLES)
RADIUS = decompose_radius(MARKS, DEV_HEMICYCLES)
fig, axes = plt.subplots(1, 3, figsize=(16.0, 4.2))
_art = draw_residual_map(axes[0], RESIDUAL)
fig.colorbar(_art, ax=axes[0], label="obs / null", pad=0.02)
draw_residual_share(axes[1], RESIDUAL)
draw_radius_split(axes[2], RADIUS)
fig.suptitle(r"Figure 5 — the residual, split by latitude separation and by longitude radius")
fig.tight_layout()
plt.show()

print(f"share of the surviving 13-61 d excess at Delta|lambda| < 2 deg: "
      f"{100.0 * RESIDUAL['share'][0]:.1f}%  (2-5 deg: {100.0 * RESIDUAL['share'][1]:.1f}%)")
print("the 0-1 d window, split: "
      + "  ".join(f"{lo:.0f}-{hi:.0f}deg {RADIUS[(lo, hi)]['mean'][0]:.3f}"
                  f"+/-{RADIUS[(lo, hi)]['se'][0]:.3f} "
                  f"({RADIUS[(lo, hi)]['above'][0]:.0f}/{RADIUS[(lo, hi)]['counted'][0]:.0f} wings)"
                  for lo, hi in RADIUS_WINDOWS))

---
## 4 · The mark likelihood, and the gauge that makes it mean anything

A **marked point process** is a point process whose events each carry extra information: a *mark*.
Here the events are emergences and the mark is the longitude each one came out at. The log-likelihood
then splits cleanly in two:

`log L = [ Σᵢ log Λ(λᵢ, sᵢ) − ∫∫ Λ dλ ds ]  +  Σᵢ log f(φᵢ | λᵢ, sᵢ, Hᵢ)`

In words: how well did we predict *when and at what latitude* groups appeared (the bracket), plus how
well did we predict *where around the Sun* each one appeared (the sum after it). `λ` is latitude, `s`
is our time coordinate, `φ` is longitude, and `Hᵢ` is everything that had already happened in that
hemisphere before event `i`.

The first bracket is the **ground process** — Weeks 3-6, character for character. The second is the
**mark distribution**: a **circular density**, one per event, which means a probability density over a
circle rather than over a line, so that 359° and 1° are two degrees apart. Because the two terms are
added rather than multiplied, *every held-out number this program has reported is unchanged by this
week*.

That holds under one condition, called **unpredictable marks**: `Λ` must not look at past *longitudes*.
Ours does not — Week 6's history bank is latitude and time only. If a later week puts longitude inside
`Λ`, the ground term moves and the whole program's scores stop being comparable.

One thing looks like a contradiction. Week 6 forbade freezing the base while fitting the history term,
because `β` and the base competed to explain the same structure. Here we freeze the ground model and
fit only the mark, which is fine: the two terms are separate sums, and the ground term contains none of
the mark's parameters.

---
### The aperture is a selection function, not a cut

`f(φ | tᵢ) = g(φ − L0(tᵢ)) · q(φ) / ∫ g(φ′ − L0(tᵢ)) · q(φ′) dφ′`

In words: the chance of recording an emergence at longitude `φ` is (what the Sun does there) × (our
chance of seeing it there), divided by the total so that the whole circle adds up to 1. `q` is the Sun;
`g` is the view, written as a function of distance from the centre of the visible disk.

`g` never falls to exactly zero, so every longitude keeps some small chance, and the division is an
ordinary integral over our circular mesh `PHI`: no hard edge, and no special function anywhere this
week. The width of `g` is set by **bandwidth selection**, cross-validated inside each fold — a narrower
or wider `g` changes the score, so choosing it by eye would mean choosing whichever value flatters the
result.

*`g` is measured once per fold and then frozen.* At one instant `g` and `q` simply multiply, so fitting
both at once cannot separate them; only the Sun's rotation does that, and only over spans long compared
with 27 days. Freezing `g` is this week's main identifiability decision.

```text
TUTOR DIRECTIVE — Section 4. The gauge.

REQUIRED OUTPUT, and it is the most important thing in the session: the student writes down a
prediction of the RATIO between what the visibility aperture is worth, in nats per event, and what
everything about the Sun's own longitude preferences is worth. An order of magnitude is enough.
Do not hint at the answer. Do not say "most", "dominant", "small" or any other word that leaks it.
They have seen Figure 3, so they have what they need.

Destinations:
1. [ESSENTIAL] A probability density has no meaning without a baseline, exactly as B has no meaning
   without B_crit = 1. Connect this week's baseline choice to the Week 3 gauge they already know.
2. [ESSENTIAL] The baseline here is APERTURE-ONLY, never uniform. Get them to say why reporting
   against uniform would be dishonest even though the arithmetic is correct.
3. [ESSENTIAL] Units. The ground term is nats per hemicycle-YEAR over a continuous domain; the mark
   term is nats per EVENT over a probability density. Ask what adding the two would mean — nothing,
   like adding miles per hour to gallons.
4. [IF TIME] The width of g is a free parameter that buys nats. Ask what stops us choosing it to make
   the week's result look good: cross-validation inside the fold, and Figure 6(b) shows it happening.

After the measurement, whatever they predicted: do not let the conversation become about whether they
were right. Ask instead what a paper reporting the mark against a uniform baseline would be claiming.
```

In [ ]:
raise NotImplementedError("Task 74: The mark likelihood, in four numbers per event")
# Reference workflow:
#   1. `sample_circular` and `wrap_rad`: read a function tabulated on `PHI` at any angle by taking the
#      nearest mesh node, and fold an angular difference into (-pi, pi]. Everything here is radians.
#   2. `log_mark_kernel`: dispatch to the von Mises, or to a wrapped Cauchy matched to it at equal mean
#      resultant length, so that the knob changes the tail and not the width. Section 6 scores both.
#   3. `aperture_convolved`: smooth `g` by the mark kernel with an FFT. A circular convolution — slide
#      one function over the other and integrate at every offset — is exactly what `rfft`/`irfft`
#      compute on a mesh that wraps around; remember the `dphi`.
#   4. `mark_terms`: build the four arrays. `a` and `c` are the window's own numerator and normalizer;
#      `b` and `d` are the kernel's contribution, accumulated over each event's parents with
#      `np.bincount`. `d` uses the convolved window read at the PARENT's central-meridian distance *at
#      the child's time* — work out why that is the right argument before you code it.
#   5. Check `mass`: `g` must integrate to 1 on the mesh, or the quadrature is wrong. `mark_logf` and
#      `mark_score` are then the formula above and the event-weighted mean over wings.
#   What you should see: `mass` equal to 1 well inside the 1e-6 the cell below asserts, and a finite log
#      density for every event. A `-inf` anywhere means an angle left the mesh.

In [ ]:
# Put your code for Task 74 here

In [ ]:
# Given: the protocol. `g`, its bandwidth and every mark parameter are rebuilt inside each fold from
# that fold's TRAINING cycles only. The bandwidth is chosen on a subsample, because it is a single
# number and does not need fifteen thousand events; the density itself uses all of them.
BANDWIDTH_SUBSAMPLE = 6000


def fold_split(held_out):
    """Training wings and scored wings for one leave-one-cycle-out fold."""
    train = [h for h in FIT_WINGS if int(h[:-1]) != held_out]
    score = [h for h in FIT_WINGS if int(h[:-1]) == held_out]
    return train, score


def fold_aperture(held_out, marks=None, kappas=APERTURE_KAPPAS, seed=SEED):
    """Measure `g` once for this fold: bandwidth by cross-validation, then the density, then frozen."""
    marks = MARKS if marks is None else marks
    train, _ = fold_split(held_out)
    cmd = np.concatenate([marks[h]["cmd_rad"] for h in train])
    rng = np.random.default_rng(seed + held_out)
    sub = cmd if len(cmd) <= BANDWIDTH_SUBSAMPLE else rng.choice(cmd, BANDWIDTH_SUBSAMPLE, replace=False)
    cv = aperture_bandwidth(sub, kappas=kappas)
    return dict(cycle=int(held_out), train=train, kappa=cv["kappa"], cv=cv,
                log_g=aperture_logdensity(cmd, cv["kappa"]), n_train=len(cmd))


APERTURES = {c: fold_aperture(c) for c in FOLD_CYCLES}
for cycle, ap in APERTURES.items():
    print(f"fold {cycle}: g from {ap['n_train']:,} training events, kappa = {ap['kappa']:.0f} "
          f"(sigma ~ {np.degrees(1.0 / np.sqrt(ap['kappa'])):.1f} deg)")

In [ ]:
# Given: the two baselines, scored on the held-out wings of every fold. This is the first half of the
# measurement whose ratio you predicted in your tutor session.
BASELINES = {}
for cycle, ap in APERTURES.items():
    _, score_wings = fold_split(cycle)
    terms = [mark_terms(MARKS[h], ap["log_g"], kappa_phi=100.0) for h in score_wings]
    BASELINES[cycle] = dict(terms=terms, wings=score_wings,
                            aperture=mark_score(terms, 0.0), uniform=UNIFORM_LOGF,
                            per_event=np.concatenate([mark_logf(t, 0.0) for t in terms]))
APERTURE_GAIN = float(np.mean([v["aperture"] - UNIFORM_LOGF for v in BASELINES.values()]))
assert all(abs(t["mass"] - 1.0) < 1.0e-6 for v in BASELINES.values() for t in v["terms"]), \
    "g does not integrate to 1 on PHI — the circular quadrature is wrong"
print(f"\nheld-out aperture-only score "
      + "  ".join(f"cycle {c}: {v['aperture']:+.4f}" for c, v in BASELINES.items())
      + f"  against uniform {UNIFORM_LOGF:+.4f}"
      + f"\n=> the visibility aperture alone is worth {APERTURE_GAIN:+.4f} nats per event")

In [ ]:
# Given: the resolution floor — how big a difference has to be before we are allowed to call it real —
# declared as a rule before anything is compared. The program's convention is the larger of the paired
# standard error and the spread across random seeds. A mark fit is deterministic, so there is no seed
# spread and the floor IS the paired standard error of the per-wing differences. That makes it a number
# per comparison rather than one number for the week, so every comparison below carries its own.
def mark_floor(per_wing_gains):
    """Paired standard error of a comparison's per-wing gains, in nats per event."""
    gains = np.asarray(list(per_wing_gains.values()) if isinstance(per_wing_gains, dict)
                       else per_wing_gains, dtype=float)
    if gains.size < 2:
        return float("nan")
    return float(np.std(gains, ddof=1) / np.sqrt(gains.size))


print(f"resolution floor: the paired standard error of the per-wing differences, computed per "
      f"comparison · {sum(len(v['wings']) for v in BASELINES.values())} wings scored over "
      f"{len(BASELINES)} folds")

In [ ]:
raise NotImplementedError("Task 75: The gauge — what the instrument is worth, and what it costs")
# Reference workflow:
#   1. `draw_score_distributions`: a histogram of the per-event log mark density under the window-only
#      model, with the uniform value and the mean marked. The spread matters — a mean gain of a nat is
#      made of events that were strongly expected and events that were not.
#   2. `draw_bandwidth_curve`: every fold's cross-validation score against concentration, with the
#      chosen point starred and the width floor drawn in. Log x-axis.
#   3. `draw_gauge_bars`: two bars on a log scale, annotated with their values and with the ratio in the
#      title. The second bar is not measured until Section 6, so leave it as an argument.
#   4. Say in the figure what the curve does at its right-hand edge. If cross-validation wants the
#      sharpest kernel on offer, the honest report is that the window's score is bounded by our mesh and
#      our width floor rather than by the data.
#   What you should see: a histogram with a long left tail, and a cross-validation curve that rises and
#      then flattens. The ratio in the title is the number you predicted — read it only after you have.

In [ ]:
# Put your code for Task 75 here

In [ ]:
# Given: Figure 6 — the gauge: what the instrument is worth, beside what the Sun is worth. Panel (c) is
# drawn twice — here with the Sun's bar held at the measured floor as a placeholder, and again in
# Section 6 once there is a real number to put in it.
_pooled = np.concatenate([v["per_event"] for v in BASELINES.values()])
fig, axes = plt.subplots(1, 3, figsize=(15.0, 4.1))
draw_score_distributions(axes[0], _pooled)
draw_bandwidth_curve(axes[1], APERTURES)
draw_gauge_bars(axes[2], APERTURE_GAIN, 0.0187, labels=("the aperture", "the Sun\n(Section 6)"))
fig.suptitle("Figure 6 — what the instrument is worth, and what choosing its width would buy")
fig.tight_layout()
plt.show()

_edge = [c for c, ap in APERTURES.items() if ap["kappa"] == max(APERTURE_KAPPAS)]
print(f"degrees-vs-radians would shift every number here by {DEG_RAD_NATS:+.4f} nats/event — "
      f"larger than anything this week measures about the Sun")
if _edge:
    print(f"cross-validation chose the sharpest kernel offered on fold(s) {_edge}: the aperture's score "
          f"is bounded by the mesh and the 1 deg floor, not by the data. Reported, not hidden.")

---
## 5 · Is there a persistent active longitude? The first held-out test

Berdyugina & Usoskin (2003) claimed two **active longitude**s 180° apart, persisting for about 120
years. Pelt et al. (2006) could not reproduce it, and showed that the method finds structure even in
data with no longitude information in it at all. The dispute was never settled, and in twenty-three
years nobody has reported a held-out score for it — which is odd, because that is the natural test.

So we run it. One **von Mises distribution** — the circle's version of a Gaussian, with a mean
direction `μ` and a concentration `κ` that plays the part of 1/width — on a rigid Carrington frame,
with `μ` and `κ` fitted on the training cycles and scored on the held-out one against the aperture-only
baseline. The frame stays rigid here: a feature fixed on the Sun for 120 years has no single latitude
to rotate differentially at, and searching for the right frame is what went wrong in 2003.

The Rayleigh statistic — add the longitudes up as unit vectors and measure the length of the result,
which is 0 for a perfectly even spread and 1 if they all point the same way — is small here, but it is
not zero. So the honest expectation is "real, and negligible": a case where a held-out likelihood says
more than a p-value does, and where a small positive number is a result rather than a failed task.

In [ ]:
raise NotImplementedError("Task 76: One active longitude, fitted then scored on a held-out cycle")
# Reference workflow:
#   1. `rayleigh`: the mean resultant length of the longitudes mapped to `harmonic * phi`, and the
#      standard large-sample p-value `exp(-n R^2)`. Report it; it is the thing the literature reports.
#   2. `fixed_parents`: express the active-longitude model in Task 74's own language, as a mixture whose
#      single parent sits at a fixed `mu` for every event. No new likelihood code is needed.
#   3. `fit_active_longitude`: take `mu` from the Rayleigh direction on the training wings, then scan a
#      few concentrations and fit the mixture weight `w` at each. Keep the best in-sample combination.
#   4. Two primitives: the fitted density over the pooled histogram, and the per-fold held-out gain as a
#      dot plot with the resolution floor drawn as a band. Never a printed table.
#   What you should see: a gain that is tiny and positive — a few ten-thousandths of a nat per event,
#      against a floor of its own size. That is a result, not a failed task.

In [ ]:
# Put your code for Task 76 here

In [ ]:
# Given: the held-out test, fold by fold, against its own paired standard error.
ACTIVE_ROWS, ACTIVE_WINGS = [], {}
for cycle, ap in APERTURES.items():
    train, score_wings = fold_split(cycle)
    fit = fit_active_longitude({h: MARKS[h] for h in train}, ap["log_g"])
    parents = fixed_parents({h: MARKS[h] for h in score_wings}, fit["mu"])
    rows = []
    for label in score_wings:
        terms = mark_terms(MARKS[label], ap["log_g"], fit["kappa"], parents[label])
        ACTIVE_WINGS[label] = mark_score([terms], fit["w"]) - mark_score([terms], 0.0)
        rows.append(ACTIVE_WINGS[label])
    ACTIVE_ROWS.append(dict(cycle=int(cycle), fit=fit, gain=float(np.mean(rows))))
ACTIVE_FLOOR = mark_floor(ACTIVE_WINGS)

RAY1 = rayleigh(np.radians(dev["longitude"].to_numpy()), 1)
RAY2 = rayleigh(np.radians(dev["longitude"].to_numpy()), 2)
fig, axes = plt.subplots(1, 2, figsize=(11.5, 4.2))
draw_active_longitude(axes[0], np.radians(dev["longitude"].to_numpy()), ACTIVE_ROWS[0]["fit"],
                      APERTURES[ACTIVE_ROWS[0]["cycle"]]["log_g"])
draw_fold_scores(axes[1], ACTIVE_ROWS, ACTIVE_FLOOR)
fig.suptitle("Figure 7 — a persistent active longitude, tested out of sample for the first time")
fig.tight_layout()
plt.show()

print(f"Rayleigh, pooled development set: first harmonic R = {RAY1['R']:.5f} (p = {RAY1['p']:.2e}), "
      f"second harmonic R = {RAY2['R']:.5f} (p = {RAY2['p']:.3f})")
_mean_gain = float(np.mean([r["gain"] for r in ACTIVE_ROWS]))
print(f"held-out gain of one von Mises on a rigid Carrington frame: {_mean_gain:+.5f} +/- "
      f"{ACTIVE_FLOOR:.5f} nats/event over {len(ACTIVE_WINGS)} wings — formally positive and "
      f"physically negligible, and the first held-out number anyone has put on this claim: a persistent "
      f"active longitude is real and worth about "
      f"{APERTURE_GAIN / max(abs(_mean_gain), 1e-12):,.0f} times less than the aperture is.")

---
## 6 · Nests as a conditional mark density

A *nest* is a patch of the Sun that keeps producing groups. Written as a mark density it is a mixture:
with probability `1 − w` an emergence picks its longitude freely, and with probability `w` it picks one
near a recent emergence in the same hemisphere.

`q(φ | H) = (1 − w)/2π  +  w · Σ_{i : tᵢ < t} k(φ − φᵢ; κ_φ) · a(Δt; τ) · b(Δ|λ|; ℓ)`

In words: either the longitude is drawn evenly around the circle, or it is drawn near one of the
earlier events, preferring the ones that happened recently and at a similar latitude. The earlier event
is the *parent*, the new one is the *child*. `k` is the kernel that decides how close "near" is, with
width `κ_φ`; `a` fades with the time gap `Δt` over a timescale `τ`; `b` fades with the latitude gap
`Δ|λ|` over a scale `ℓ`; and `w` is the *nested fraction*, the share of emergences that belong to a
nest at all. Because `w` sits inside a likelihood it comes with an error bar — the first one anybody
has put on it, since every published nested fraction is a membership count from a clustering algorithm.

Parents are strictly earlier, in the same hemisphere, and in the development set only. A held-out
wing's *own earlier* events are not leakage; its later ones are. One knob moves per variant, and one of
those knobs is the kernel's shape: a von Mises against a **wrapped Cauchy distribution**, whose heavier
tail is what fits of earthquake triggering consistently prefer.

One knob is new. `φᵢ` is where a parent was *when it emerged*, but a nest sits in plasma that rotates
at its own rate `Ω(λ)`, so by the time the child appears the nest has moved: **advection** carries the
parent forward to the child's time before the distance between them is measured. Figure 2(c) says how
far that is — a few degrees per rotation, comparable to the kernel's own width — so `rigid` and
`tracking` are both scored.

In [ ]:
raise NotImplementedError("Task 77: The strictly-past mixture, its fit, and one knob at a time")
# Reference workflow:
#   1. `pair_index`: for one wing, list every (child, parent) pair with the parent strictly earlier and
#      inside the lag window, keeping the time gap, the latitude gap, the parent's longitude and the
#      parent's latitude. Build it once per wing — the pair set depends on neither `tau`, `ell`, nor
#      the frame.
#   2. `advect_pairs`: the frame knob. At `Omega(lat)` from Task 69 a parent drifts relative to the rigid
#      frame, so add `(Omega(lat) - 14.1844) * dt` degrees to its longitude before measuring the
#      separation. Compare that drift per rotation against the kernel width before you believe it
#      matters. `None` must return the pairs unchanged, or `rigid` stops being a scoreable variant.
#   3. `nest_parents`: weight each pair by `exp(-dt/tau)` times a Gaussian in latitude separation, then
#      normalize each child's weights to sum to one. Use `np.bincount` for the per-child totals.
#   4. `fit_mark_mixture`: assemble Task 74's terms for the training wings and fit `w` on (0, 0.95) with
#      a bounded scalar minimizer, or hold it at a prescribed value. Allow a per-wing pair mask, because
#      Section 7 asks the same question of a subset of the pairs. Then `score_mark_variant`, the LOCO
#      loop: resolve the frame with `fold_profile` INSIDE the fold and advect there, so the held-out wing
#      never helps measure the rotation law its own parents are carried forward with; fit `w` on the
#      training wings; score the held-out wings against their own window-only baseline; and keep the
#      per-wing gains, which Section 7 needs.
#   5. Two primitives: variants as a dot plot sorted by gain with the floor as a band, and per-wing gains
#      as a bar chart coloured by source catalogue. No printed twin of either.
#   What you should see: every variant's fitted `w` strictly inside (0, 0.95) — a fit pressed against a
#      bound is reporting the bound, not a measurement — and `tracking` scoring above `rigid`.

In [ ]:
# Put your code for Task 77 here

In [ ]:
# Given: the pair index, built once, and the variant scan. One knob moves per row, from a single
# reference spec — the same mechanic as Week 4's scoreboard, in a new coordinate.
PAIRS = {h: pair_index(MARKS[h]) for h in DEV_HEMICYCLES}
print(f"pair index: {sum(len(p['owner']) for p in PAIRS.values()):,} strictly-past same-wing pairs "
      f"within {MARK_MAX_LAG_DAYS:.0f} days")


def variant_label(spec):
    """The experiment's name, generated from the config rather than typed by hand."""
    bits = [spec["family"], f"kap{spec['kappa_phi']:.0f}", f"tau{spec['tau']:.0f}",
            f"ell{spec['ell']:.0f}", "wfit" if spec["w"] is None else f"w{spec['w']:.2f}",
            spec["frame"]]
    return "·".join(bits)


VARIANT_SPECS = [dict(MARK_SPEC)]
for knob, values in MARK_KNOBS.items():
    for value in values:
        if value == MARK_SPEC[knob]:
            continue
        VARIANT_SPECS.append(dict(MARK_SPEC, **{knob: value}))

MARK_RESULTS = []
_started = time.time()
for spec in VARIANT_SPECS:
    result = score_mark_variant(spec, APERTURES, MARKS, PAIRS, BASELINES)
    result["label"] = variant_label(spec)
    MARK_RESULTS.append(result)
REFERENCE = MARK_RESULTS[0]
print(f"{len(MARK_RESULTS)} variants scored in {time.time() - _started:.0f} s · reference "
      f"{REFERENCE['label']}")

In [ ]:
# Given: Figure 8 — the nested fraction, with a likelihood behind it rather than a cluster count.
# Panel (b) is the one Section 7 takes apart: the colour says which catalogue each wing came from, and
# the split by colour is not noise.
fig, axes = plt.subplots(1, 2, figsize=(13.0, 4.6))
draw_variant_dots(axes[0], MARK_RESULTS, REFERENCE["se"], reference=REFERENCE["label"])
draw_wing_gains(axes[1], REFERENCE["per_wing"], era_of=ERA_OF)
fig.suptitle("Figure 8 — the mark's variants, and the wings they are averaged over")
fig.tight_layout()
plt.show()

print(f"reference mark: w = {REFERENCE['w']:.4f} fitted per fold, held-out gain "
      f"{REFERENCE['gain']:+.5f} +/- {REFERENCE['se']:.5f} nats/event over aperture-only "
      f"({REFERENCE['gain'] / max(REFERENCE['se'], 1e-12):.1f} times its own paired standard error)")
_best = max(MARK_RESULTS, key=lambda r: r["gain"])
print(f"best variant {_best['label']}: {_best['gain']:+.5f}, the whole spread "
      f"{max(r['gain'] for r in MARK_RESULTS) - min(r['gain'] for r in MARK_RESULTS):.5f} wide"
      f"\nTHE GAUGE: the aperture {APERTURE_GAIN:+.4f} against the Sun {REFERENCE['gain']:+.4f} "
      f"— a ratio of {APERTURE_GAIN / max(REFERENCE['gain'], 1e-12):.0f} to 1")

In [ ]:
raise NotImplementedError("Task 78: The fitted mark, drawn on the Carrington map")
# Reference workflow:
#   1. `mark_density_at`: at any time inside a wing, rotate `g` to that time's `L0`, build the kernel sum
#      over the events strictly earlier than it, mix with weight `w`, multiply, and normalize on `PHI`
#      with a plain sum times `DPHI`. With a `profile`, carry each past event forward to the time being
#      drawn — and return those carried longitudes, because panel (b) marks them.
#   2. `draw_mark_field`: evaluate that on a grid of times and draw it as a `pcolormesh` over
#      (days, longitude), with the real emergences drawn on top as open circles. Return the mappable so
#      the given cell can add a colour bar, and no `plt.show()`.
#   3. `draw_mark_slices`: three slices through the same field as ordinary curves, with the longitudes
#      of the most recent parents marked as faint vertical lines.
#   4. Look at panel (a) before reading on. The bright band is the viewing window, and it moves; the
#      bumps that stay put are the nest. If you cannot see two different things in that panel, the
#      mixture weight is doing nothing.
#   What you should see: a bright diagonal stripe crossing the panel once every 27.3 days, with small
#      stationary bumps sitting on top of it.

In [ ]:
# Put your code for Task 78 here

In [ ]:
# Given: Figure 9 — the mark, drawn on the Sun's own coordinates.
_wing = MARKS[STRONGEST]
_cycle = int(STRONGEST[:-1]) if int(STRONGEST[:-1]) in APERTURES else sorted(APERTURES)[0]
_log_g = APERTURES[_cycle]["log_g"]   # the aperture of a fold that excludes this wing's own cycle
_w = REFERENCE["w"]
MAP_DAY0 = float(np.quantile(_wing["days"], 0.42))
_prof = fold_profile(MARK_SPEC["frame"], _cycle)
fig, axes = plt.subplots(1, 2, figsize=(13.0, 4.6))
_art = draw_mark_field(axes[0], _wing, _log_g, MARK_SPEC, _w, PAIRS[STRONGEST], MAP_DAY0,
                       profile=_prof)
fig.colorbar(_art, ax=axes[0], label=r"$f(\phi \mid t, H)$ [1/rad]", pad=0.02)
draw_mark_slices(axes[1], _wing, _log_g, MARK_SPEC, _w, PAIRS[STRONGEST], MAP_DAY0, profile=_prof)
fig.suptitle(f"Figure 9 — hemicycle {STRONGEST}: what the fitted mark actually says")
fig.tight_layout()
plt.show()

---
## 7 · The degeneracy, which is this week's headline

Two completely different stories predict the same table, and this catalogue cannot tell them apart.
Figure 10(d) draws them both: *a nest*, one patch of the Sun producing several groups at nearly the same
longitude days or rotations apart; and *one region catalogued twice*, a group that survives a trip
round the far side, comes back with a new identifier, and enters our table as a second emergence at
almost the same place one rotation later. They leave the same dots behind.

Section 3 left an excess at 13-20, 20-34 and 47-61 days — one, two and roughly two rotations — tight in
longitude *and* in latitude, and Figure 5(c) left a second one, same-day, just outside the radius
inside which the catalogue holds no pairs at all. Both stories predict exactly that.

The catalogues say which one they chose. Brouwer & Zwaan (1990), §2: "Less than 10% of the sunspot
groups are listed as recurrent in GPR; *they are treated as separate sunspot groups*." The Debrecen
readme documents minting new numbers by appending letters. The distinction we need was thrown away
before we got the data.

The policy is measurable, though — Figure 10(c). Both the same-day hole and the ring around it belong
to *large* groups and vanish for small ones, and a merging radius that grows with a group's own size is
something a bookkeeping rule knows and the dynamo cannot. The large pairs' ring under two days is then
paid for by a deficit just after it: one object counted twice, not a process that triggers and decays.
What survives all of that is an excess at 2-20 days among the *small* pairs alone — a tenth of the
signal we started with, and the week's only clean candidate for triggering.

Figure 10(e) is the check to run before trusting either end of the bracket. The mask flags *pairs*,
and pairs outnumber events by forty to one, so a few per cent of pairs can be more returns than every event
in the catalogue could possibly supply. Count events, not pairs, and what the ablation removes turns
out to be a *lag band* rather than a set of identified returns.

So the week does not report a nested fraction. It reports a *bracket*.

```text
TUTOR DIRECTIVE — Section 7. Nesting or bookkeeping?

REQUIRED OUTPUT: before the measurement, the student commits in writing to which way they expect the
degeneracy to break — more of the signal from real nesting, or more from re-catalogued recurrences —
and names one observable that would tell them.

Destinations:
1. [ESSENTIAL] The catalogue's identity policy is a physical assumption in disguise. "One group" is a
   decision somebody made, and here it was made in a way that erases the thing we want to measure.
   Panel (d) is the picture of it: two stories, the same dots.
2. [IF TIME] Why the obvious test does not work. One might compare a group's last observation with a
   co-located successor's first, expecting negative gaps if they are the same region. Measured: 5,872
   pairs at 20-34 d, median gap 17.2 d, ZERO negative gaps — not evidence against recurrence, but
   guaranteed by the identifier policy. A test that cannot fail is not a test.
3. [ESSENTIAL] The era contrast is the real evidence, and it points both ways. Get them to propose
   both readings (a changing Sun, a changing catalogue) and then to say what would distinguish them.
   The answer involves data we do not have, and that is the honest end of the argument.
4. [ESSENTIAL] What a bracket claims, and why it is stronger than a point estimate with a hidden
   assumption.
5. [IF TIME] Count events, not pairs. Panel (e) draws it: the mask flags ~6% of PAIRS, so ask what
   fraction of EVENTS could be returns and make them do the division before they read the panel.

If they ask what the right answer is: nobody knows, and saying so is the week's result.
```

In [ ]:
raise NotImplementedError("Task 79: The era contrast, and the ceiling on re-catalogued returns")
# Reference workflow:
#   1. `recurrence_mask`: flag the pairs that arrive within a few days of a whole number of rotations at
#      nearly the same latitude. Do NOT put longitude into this condition, however tempting: the
#      ablation below refits without these pairs and asks what survives, so selecting them on longitude
#      would guarantee the answer. You are defining a suspicion, not detecting a fact — no column in the
#      data can confirm that any one of these pairs is a single region seen twice.
#   2. `wing_gains_all`: score every wing at one fixed `w`, each against the leave-one-out window built
#      for its own cycle, so the era contrast rests on all the wings rather than only the scored folds.
#      Fit nothing here; a fitted `w` per wing would confound the comparison with sample size. Then
#      `era_summary` groups those gains by source catalogue.
#   3. Two primitives. In panel (a) plot every wing, not just the two means — with four Debrecen wings a
#      bare mean claims more than the data supports. In panel (b) draw the nested fraction as the
#      interval between the two refits, with the literature's chance-corrected values for scale.
#   4. `draw_event_ceiling`: the arithmetic that bounds the lower end, as panel (e). The mask flags
#      PAIRS, so compare the flagged count against `3 * N_events` — the most returns the catalogue could
#      possibly hold, since even if every event were a return it could have at most three earlier
#      incarnations. Three bars on a log scale. (`draw_two_stories`, panel (d), is given: it has no data
#      in it, and drawing the ambiguity is not the same as measuring it.)
#   5. `size_channels`: the same ring as Figure 5(c), now split by lag AND by whether both members are
#      above the median peak area. A channel that exists only for big groups knows something the dynamo
#      has no way to know. Watch for an excess immediately followed by a deficit — that shape is one
#      object counted twice, not a process that triggers and decays. The bracket is the deliverable:
#      neither endpoint is the answer, and the week does not pick one.
#   What you should see: a mask that flags only a few per cent of the pairs, and yet a flagged count
#      that very nearly exhausts the event ceiling — which is the whole point of counting events instead.
#      Which way the bracket should be read is yours to predict first.

In [ ]:
# Put your code for Task 79 here

In [ ]:
# Given: Figure 10 — the week's headline, which is an interval rather than a number. Panels (a) to (c)
# are the evidence, panel (d) is the ambiguity itself drawn as two stories, and panel (e) is the
# arithmetic that bounds how much of the signal re-catalogued returns could possibly explain.
RECUR = {h: recurrence_mask(PAIRS[h], MARKS[h]) for h in DEV_HEMICYCLES}
ERA_APERTURES = {c: fold_aperture(c) for c in SETTINGS["cycles"]}
ERA_GAINS = wing_gains_all(MARK_SPEC, ERA_APERTURES, MARKS, PAIRS, REFERENCE["w"])
ERAS = era_summary(ERA_GAINS)
EXCL = score_mark_variant(MARK_SPEC, APERTURES, MARKS, PAIRS, BASELINES,
                          masks={h: ~RECUR[h] for h in DEV_HEMICYCLES})
RECUR_ONLY = score_mark_variant(MARK_SPEC, APERTURES, MARKS, PAIRS, BASELINES,
                                masks=dict(RECUR))

CHANNELS = size_channels(MARKS, PAIRS)
_n_rec = sum(int(m.sum()) for m in RECUR.values())
_n_pair = sum(len(p["owner"]) for p in PAIRS.values())
_n_event = sum(p["n"] for p in PAIRS.values())

fig = plt.figure(figsize=(16.5, 8.8))
gs = fig.add_gridspec(2, 3)
draw_era_contrast(fig.add_subplot(gs[0, 0]), ERAS)
draw_bracket(fig.add_subplot(gs[0, 1]), REFERENCE["w"], EXCL["w"])
draw_size_channels(fig.add_subplot(gs[0, 2]), CHANNELS)
draw_two_stories(fig.add_subplot(gs[1, 0:2]))
draw_event_ceiling(fig.add_subplot(gs[1, 2]), _n_rec, _n_event, _n_pair)
fig.suptitle("Figure 10 — nesting, or the same region counted twice? The catalogue cannot say")
fig.tight_layout()
plt.show()
if np.isfinite(ERAS["DPD"]["mean"]):
    print(f"era contrast: RGO {ERAS['RGO']['mean']:+.4f} over {ERAS['RGO']['n']} wings against "
          f"DPD {ERAS['DPD']['mean']:+.4f} over {ERAS['DPD']['n']} — a factor of "
          f"{ERAS['RGO']['mean'] / ERAS['DPD']['mean']:.1f}")
else:
    print(f"era contrast needs a Debrecen fold cycle; not available on the '{PROFILE}' profile")
print(f"{_n_rec:,} of {_n_pair:,} pairs look like a far-side return "
      f"({100.0 * _n_rec / _n_pair:.2f}%), and they alone are worth "
      f"{RECUR_ONLY['gain']:+.5f} nats/event at w = {RECUR_ONLY['w']:.3f}")
print(f"removing them takes the nested fraction from {REFERENCE['w']:.3f} to {EXCL['w']:.3f} and the "
      f"gain from {REFERENCE['gain']:+.5f} +/- {REFERENCE['se']:.5f} to {EXCL['gain']:+.5f} +/- "
      f"{EXCL['se']:.5f}, which is inside its own standard error"
      f"\nNESTED FRACTION, as a bracket: {min(REFERENCE['w'], EXCL['w']):.3f} to "
      f"{max(REFERENCE['w'], EXCL['w']):.3f} — its width is set by the catalogue's identity policy, "
      f"not by our statistics")
_ceiling = 3 * _n_event        # n in (1, 2, 3): at most three earlier incarnations of one region
print(f"but count events, not pairs: {_n_event:,} events, and n in (1,2,3) allows at most three earlier "
      f"incarnations each, so even if EVERY event were a return the true-return set could not exceed "
      f"{_ceiling:,} pairs. The mask flags {100.0 * _n_rec / _ceiling:.0f}% of that ceiling, so at any "
      f"plausible return rate it is mostly timing coincidence. What the ablation removes is a LAG "
      f"BAND, not a set of identified returns, so w = {EXCL['w']:.3f} is the nested fraction left "
      f"once the lags where recurrence could live are taken out — the right quantity, still degenerate")
_r = CHANNELS["ratio"]
print(f"the 8-20 deg ring by size, obs/null: large pairs {_r[0, 2]:.2f} at 0-1 d and {_r[1, 2]:.2f} at "
      f"1-2 d then {_r[3, 2]:.2f} at 4-8 d, an excess paid for by a deficit; small pairs "
      f"{_r[0, 0]:.2f} at 0-1 d but {_r[2, 0]:.2f} at 2-4 d; at one rotation {_r[6, 2]:.2f} large "
      f"against {_r[6, 0]:.2f} small")

---
## 8 · Anti-gaming, twice

Two ways to fool yourself, and we can put a number on both. A *leak*, here, is a choice that makes a
result look stronger than the data supports without anybody writing down a false number.

*(a) The baseline.* Report the mark against a *uniform* circular density instead of against the
measured aperture, and the number inflates by whatever the telescope is worth. Of the papers on this
subject over forty years, exactly one — Brouwer & Zwaan (1990), one clause in §3 — drew its null
longitudes on the visible disk, and none between 2016 and 2026 applies any visibility weight at all.

*(b) The frame.* Pelt et al. (2006) set *every* longitude in the record to zero, re-ran the published
active-longitude search, and still recovered a specific rotation rate. A search over a grid of fitted
differential-rotation frames finds structure in data with no longitude information in it at all.

```text
TUTOR DIRECTIVE — Section 8. What the literature did.

Destinations:
1. [ESSENTIAL] The two leaks are different in kind. (a) is a reporting choice — the number is right,
   the baseline is wrong. (b) is a search over a parameter nobody counted. Make them separate the two.
2. [IF TIME] Degrees of freedom you do not count are not free. Ask how many effectively independent
   frames a grid of rotation parameters contains, and what that does to a p-value. They will not have
   a number; the point is that neither does the literature.
3. [IF TIME] The unit conversion in panel (a) is worth dwelling on. A per-event quantity becomes a
   per-wing-year quantity only by multiplying by the event rate: legitimate for comparing leak SIZES,
   illegitimate for the headline metric. Get them to say which is which.
4. [ESSENTIAL] The honest reading of this section is not "the literature is wrong". It is "these
   claims have never been scored out of sample, and now one of them has been".
5. [IF TIME] The sharpest question here: Section 6 DID move the frame and it helped — so why is that
   not leak (b)? Make them find the difference. Advection used a rate measured from daily positions
   inside each group, an observable the mark never reads, re-derived per fold and scored held out: one
   frame, declared in advance. Leak (b) searches a grid of frames against the statistic and reports
   the maximum, unscored. Same coordinate change, opposite protocols.
```

In [ ]:
raise NotImplementedError("Task 80: Two ways to fool yourself, measured")
# Reference workflow:
#   1. `uniform_longitude_surrogate`: rebuild the mark table with longitudes that carry no information —
#      drawn independently and evenly around the circle, or all set to one constant. A *surrogate* is
#      data built to contain nothing, so that a method which still finds something has been caught.
#      Recompute CMD from the new longitudes; forgetting that is the one way to build a surrogate that
#      still has the viewing window in it.
#   2. Refit the Section 6 reference model on the uniform surrogate. The held-out gain must come back
#      inside the resolution floor and `w` must collapse. If it does not, something in Section 6 is
#      reading its own answer.
#   3. `frame_scan`: shift every longitude by `(rate - carrington) * t` and take the Rayleigh R, over a
#      grid of rates. Offset the wings from each other in time so the frames cannot coincide by
#      accident. Report the MAXIMUM over the grid — that is what a search returns.
#   4. Two primitives: the leak scale as a horizontal log-scale bar chart, and R against rate for the
#      data and for both zero-information sets on one axis.
#   5. The conversion in panel (a): a per-event gain becomes a per-wing-year gain by multiplying by the
#      events per wing-year. Fine for comparing the size of two leaks, never for the headline metric.
#   What you should see: `w` collapsing towards 0 on the surrogate, and a frame scan whose maximum sits
#      well above its own typical value even on a record with no longitude information in it at all.

In [ ]:
# Put your code for Task 80 here

In [ ]:
# Given: Figure 11 — both leaks, measured. The uniform-baseline leak is this week's own, and it is the
# largest the program has found outside the inequality Week 6 broke on purpose.
EVENTS_PER_WING_YEAR = float(np.mean([len(MARKS[h]["lon"]) for h in FIT_WINGS]) / WINDOW_YEARS)
_rng = np.random.default_rng(SEED + 7)
IID = uniform_longitude_surrogate(MARKS, DEV_HEMICYCLES, _rng)
IID_PAIRS = {h: pair_index(IID[h]) for h in DEV_HEMICYCLES}
IID_RESULT = score_mark_variant(MARK_SPEC, APERTURES, IID, IID_PAIRS, BASELINES)
assert abs(IID_RESULT["gain"]) < 0.25 * REFERENCE["gain"], \
    f"zero-information longitudes bought {IID_RESULT['gain']:+.5f} nats/event — Section 6 leaks"

LEAKS = [("Week 6: leaky inequality", 82.9113),
         ("Week 7: uniform baseline", APERTURE_GAIN * EVENTS_PER_WING_YEAR),
         ("Week 5: stopping on the scored wing", 4.5716),
         ("Week 4: worst protocol violation", 1.94),
         ("Week 7: the Sun itself", REFERENCE["gain"] * EVENTS_PER_WING_YEAR)]
RATE_GRID = np.linspace(12.6, 15.0, 1601)
ZEROED = uniform_longitude_surrogate(MARKS, DEV_HEMICYCLES, _rng, constant=0.0)
CURVES = {"the data": frame_scan(MARKS, DEV_HEMICYCLES, RATE_GRID),
          "uniform longitudes": frame_scan(IID, DEV_HEMICYCLES, RATE_GRID),
          "every longitude zero": frame_scan(ZEROED, DEV_HEMICYCLES, RATE_GRID)}

fig, axes = plt.subplots(1, 2, figsize=(12.5, 4.4))
draw_leak_scale(axes[0], LEAKS)
draw_frame_search(axes[1], RATE_GRID, CURVES)
fig.suptitle("Figure 11 — the two ways this subject has been got wrong, in nats and in frames")
fig.tight_layout()
plt.show()

print(f"zero-information control: w collapses to {IID_RESULT['w']:.4f} and the held-out gain is "
      f"{IID_RESULT['gain']:+.5f} nats/event against the reference's {REFERENCE['gain']:+.5f}")
print(f"reporting against uniform instead of the aperture would inflate this week's result from "
      f"{REFERENCE['gain']:+.4f} to {REFERENCE['gain'] + APERTURE_GAIN:+.4f} nats/event "
      f"— {100.0 * APERTURE_GAIN / (APERTURE_GAIN + REFERENCE['gain']):.1f}% of it instrument")
print(f"frame search over {len(RATE_GRID):,} rates: the data reaches R = {CURVES['the data'].max():.4f}, "
      f"but uniform longitudes reach {CURVES['uniform longitudes'].max():.4f} and a record with every "
      f"longitude set to zero reaches {CURVES['every longitude zero'].max():.4f}")

---
## 9 · Physical plausibility, and what this could have detected

A mark model that cannot reproduce Figure 1 has not understood the instrument. So: generate longitudes
from the fitted mark at the real events' times, and draw that map beside the Sun's own. The same
generator answers the question Section 3 could not — a *power curve*: how strong would a nest have to
be before the co-location statistic could see it at all? That test has to go through the selection
function. Move each child straight to its parent's longitude instead and the ratio goes *down*, because
it puts the event outside the window we could ever have seen it in.

In [ ]:
raise NotImplementedError("Task 81: Generate longitudes from the mark, and what it could detect")
# Reference workflow:
#   1. `sample_longitudes`: walk the wing in time order. At each event rotate `g` to that event's `L0`,
#      build the kernel sum over the longitudes ALREADY DRAWN for its parents, mix with weight `w`,
#      multiply by `g`, and draw from the result by inverse-transform sampling on `PHI`: take the
#      cumulative sum of the density, scale it to 1, and read off where a uniform draw lands.
#   2. Use the convolution trick: put the parents' weights on their mesh nodes and smooth that spike
#      train with the kernel by FFT. Writing the kernel matrix out per event works and is a hundred
#      times slower.
#   3. `synthetic_marks`: keep every event's real time and latitude, and replace only its longitude. The
#      synthetic wing then has the Sun's own rate envelope and the model's longitudes, which is the only
#      comparison that isolates the mark.
#   4. Draw the synthetic Carrington map beside the real one, with the same window and the same meridian
#      track. If the synthetic map does not show the diagonal crowding, `g` is not in your generator.
#   5. `draw_power_curve`: the co-location ratio against the generating `w`, with the null band and the
#      real measurement drawn on it. This is where a null result would have got its meaning.
#   What you should see: a synthetic map that passes for Figure 1(a) at a glance, and a power curve that
#      climbs out of the null band only at a `w` far above anything we measured.

In [ ]:
# Put your code for Task 81 here

In [ ]:
# Given: Figure 12 — the model's Sun beside the real one, and the detection threshold.
_rng = np.random.default_rng(SEED + 31)
SYNTH = synthetic_marks(MARKS, [STRONGEST], APERTURES, MARK_SPEC, REFERENCE["w"], _rng)
POWER_W = (0.0, 0.05, 0.10, 0.20, 0.40)
POWER_BIN = LAG_BINS.index((20, 34))
power_mean, power_se = [], []
for value in POWER_W:
    realization = synthetic_marks(MARKS, DEV_HEMICYCLES, APERTURES, MARK_SPEC, value,
                                  np.random.default_rng(SEED + 500 + int(1000 * value)))
    stats = wing_mean_ratio(colocation_by_wing(realization, DEV_HEMICYCLES), reference=NULL_REF)
    power_mean.append(stats["mean"][POWER_BIN])
    power_se.append(stats["se"][POWER_BIN])

fig, axes = plt.subplots(1, 3, figsize=(15.5, 4.3))
draw_strobe_compare(axes[0], MARKS[STRONGEST], MAP_DAY0, title="(a) the Sun", colour="tab:blue")
draw_strobe_compare(axes[1], SYNTH[STRONGEST], MAP_DAY0, title="(b) the fitted mark", colour="tab:green")
draw_power_curve(axes[2], POWER_W, power_mean, power_se,
                 PRESERVED_RATIO["mean"][POWER_BIN], PRESERVED_RATIO["se"][POWER_BIN])
fig.suptitle(f"Figure 12 — hemicycle {STRONGEST} generated from the mark, and the week's detection floor")
fig.tight_layout()
plt.show()

_synth_geom = aperture_geometry(SYNTH[STRONGEST]["cmd"])
_real_geom = PER_WING_GEOM[STRONGEST]
_thresh = next((v for v, m, s in zip(POWER_W, power_mean, power_se) if v > 0 and m - 2 * s > 1.0), None)
print(f"synthetic wing: median |CMD| {_synth_geom['median_abs']:.1f} deg and "
      f"{100 * _synth_geom['east_fraction']:.1f}% east, against the Sun's "
      f"{_real_geom['median_abs']:.1f} deg and {100 * _real_geom['east_fraction']:.1f}%"
      f"\ndetection threshold on the 20-34 d statistic: "
      + (f"w of about {_thresh:.2f}" if _thresh else "not reached in the scanned range")
      + f", against the fitted w = {REFERENCE['w']:.3f}")

In [ ]:
# Given: the week's board, and the only printed table in this notebook. Both units appear, in separate
# columns, and they are never added together: the ground process is scored in nats per hemicycle-year
# and the mark in nats per event. Adding those would be like adding miles per hour to gallons.
BOARD = pd.DataFrame([
    dict(model="ground process only (Weeks 4-6)", mark_nats_per_event=np.nan,
         w=np.nan, se=np.nan, rgo=np.nan, dpd=np.nan),
    dict(model="+ aperture only (the instrument)", mark_nats_per_event=APERTURE_GAIN,
         w=0.0, se=np.nan, rgo=np.nan, dpd=np.nan),
    dict(model="+ one active longitude, rigid frame", w=ACTIVE_ROWS[0]["fit"]["w"],
         mark_nats_per_event=float(np.mean(list(ACTIVE_WINGS.values()))), se=ACTIVE_FLOOR,
         rgo=np.nan, dpd=np.nan),
    dict(model="+ nest mixture (every pair)", mark_nats_per_event=REFERENCE["gain"],
         w=REFERENCE["w"], se=REFERENCE["se"], rgo=ERAS["RGO"]["mean"], dpd=ERAS["DPD"]["mean"]),
    dict(model="+ nest mixture, far-side returns removed", mark_nats_per_event=EXCL["gain"],
         w=EXCL["w"], se=EXCL["se"], rgo=np.nan, dpd=np.nan),
    dict(model="zero-information control", mark_nats_per_event=IID_RESULT["gain"],
         w=IID_RESULT["w"], se=IID_RESULT["se"], rgo=np.nan, dpd=np.nan),
]).set_index("model")
BOARD.columns = ["mark [nats/event]", "w", "floor (paired se)", "RGO [nats/ev]", "DPD [nats/ev]"]
print(f"Week 7 board · held-out, LOCO over cycles {tuple(APERTURES)} · the floor is each row's own "
      f"paired standard error over {len(REFERENCE['per_wing'])} scored wings")
print(BOARD.to_string(float_format=lambda v: f"{v:+.5f}", na_rep="--"))
print(f"\nthe ground process is unchanged: champion held-out NLL "
      f"{rows_of(RUNS[('champion', 'bank')])[f'nll/yr ({HEADLINE})'].mean():.4f} nats per "
      f"hemicycle-year, exactly as Week 6 left it")

In [ ]:
# Given: Figure 13 — the week on one axis. Nothing is computed here that was not computed already:
# both panels redraw numbers from the board above and from Section 3, which is the whole point. Panel
# (a) puts every magnitude the week measured on one log axis, so the instrument and the Sun can be
# compared at a glance; panel (b) is the co-location story in the order it happened.
def draw_value_ledger(ax, rows, floor):
    """Everything the week measured, in nats per event, on one log axis."""
    values = np.array([value for _, value in rows], dtype=float)
    shown = np.maximum(np.abs(values), 1.0e-6)
    ax.barh(np.arange(len(rows)), shown, height=0.6,
            color=["tab:orange" if value > floor else "0.6" for value in values])
    for y, value in enumerate(values):
        ax.annotate(f"{value:+.4f}", (shown[y], y), xytext=(6, 0), textcoords="offset points",
                    fontsize=8, va="center")
    ax.axvline(floor, color="tab:red", lw=1.2, ls="--", label=f"resolution floor, {floor:.4f}")
    ax.set_yticks(np.arange(len(rows)))
    ax.set_yticklabels([name for name, _ in rows], fontsize=8)
    ax.set_xscale("log")
    ax.set_xlim(right=shown.max() * 20.0)
    ax.set_xlabel("held-out gain [nats per event], log scale")
    ax.legend(fontsize=7, loc="lower right")
    ax.set_title("(a) what the instrument is worth, and what the Sun is worth", fontsize=10)


def draw_colocation_story(ax, steps):
    """Three times chance, then below chance, then split into channels that disagree."""
    values = np.array([value for _, value, _ in steps], dtype=float)
    ax.bar(np.arange(len(steps)), values, color=[colour for _, _, colour in steps], width=0.62)
    for x, value in enumerate(values):
        ax.annotate(f"{value:.2f}", (x, value), xytext=(0, 4), textcoords="offset points",
                    fontsize=8, ha="center")
    ax.axhline(1.0, color="k", lw=1.0)
    ax.set_xticks(np.arange(len(steps)))
    ax.set_xticklabels([name for name, _, _ in steps], fontsize=7.5)
    ax.set_ylabel("co-location, as a multiple of its null")
    ax.set_title("(b) the strongest signal in the data, in three steps", fontsize=10)


def _channel(result, lag_days, size_class):
    """One cell of the size split, found by lag rather than by a hard-coded row index."""
    index = next(i for i, (lo, hi) in enumerate(result["lag_bins"]) if lo <= lag_days < hi)
    return float(result["ratio"][index, size_class])


LEDGER = [("the viewing window\n(the instrument)", APERTURE_GAIN),
          ("the nest mixture\n(every pair)", REFERENCE["gain"]),
          ("one active longitude\n(rigid frame)", float(np.mean(list(ACTIVE_WINGS.values())))),
          ("zero-information\ncontrol", IID_RESULT["gain"])]
STORY = [("raw, against\nuniform chance\n0-1 d", RAW_RATIO["mean"][0], "tab:red"),
         ("the same data, against\na window-preserving null\n0-1 d", PRESERVED_RATIO["mean"][0],
          "tab:blue"),
         ("big pairs\n0-1 d", _channel(CHANNELS, 0.5, 2), "0.55"),
         ("big pairs\none rotation", _channel(CHANNELS, CARRINGTON_PERIOD, 2), "0.55"),
         ("small pairs\n2-4 d", _channel(CHANNELS, 3.0, 0), "tab:green")]

fig, axes = plt.subplots(1, 2, figsize=(14.5, 4.6))
draw_value_ledger(axes[0], LEDGER, REFERENCE["se"])
draw_colocation_story(axes[1], STORY)
fig.suptitle("Figure 13 — the week on one axis: the instrument, the Sun, and what is left over")
fig.tight_layout()
plt.show()

---
## 10 · Where this leaves us

Figure 13 is the week on one axis. Five findings:

1. *Longitude is in the model and the ground process never moved.* Every held-out NLL this program has
   reported still stands: the mark term is simply added on, and `Λ` never reads a longitude.
2. *The instrument is most of what there is to know* — more than a nat per event against a few
   hundredths, Figure 13(a) — so a result quoted against a uniform baseline is almost all telescope.
3. *The strongest clustering signal in the data was the telescope*, 13(b): three times chance under
   a day becomes slightly below chance once the moving window is preserved, and what looked like one
   signal splits into three channels, only one of which is still a candidate.
4. *The active longitude is real and negligible*, and it is the first held-out number on a claim argued
   over since 2003. *Measuring* a frame is not *searching* for one: advection was measured from tracked
   groups and scored held out, where 2003 searched a grid of frames against its own answer.
5. *The nested fraction is a bracket, not a number*, and the bracket's width is set by the catalogue's
   identity policy rather than by our statistics.

What 3.0 would need: a catalogue whose recurrence policy is recoverable; the global cycle, deferred
throughout; and a stochastic latent, so that "a currently productive patch" can be a state rather than
a sum over past events.

```text
TUTOR DIRECTIVE — Section 10. What the week brackets.

Destinations:
1. [ESSENTIAL] Check all three written predictions now — the null's effect (Section 3), the
   instrument-to-Sun ratio (Section 4), and which way the degeneracy breaks (Section 7). Whatever they
   wrote, the question is what the prediction assumed, not whether it was right.
2. [ESSENTIAL] The distinction the week turns on: a signal can survive a correct null and still be
   degenerate with how the data was assembled. A null tests a hypothesis about the process; it cannot
   test a hypothesis about the catalogue.
3. [IF TIME] What would close the bracket. Push for something concrete and observable — magnetogram
   continuity across a far-side passage, a catalogue that records recurrence, far-side helioseismology.
4. [IF TIME] Three things are deferred, each for its own reason: the global cycle (the program models
   one hemispheric wing from that hemisphere's own observables), cross-hemisphere active longitudes
   (the same reason), and a stochastic latent (out of scope all program long). Longitude inside the
   ground intensity is not deferred, it is ruled out — it would void every number Weeks 4-6 reported.

Next week is Scoreboard 2 and the withheld cycles. The mark adds four fields to a report row — the
kernel family, w, the aperture bandwidth and the frame — and its column is in nats per event, never
added to the headline.
```

---
## Before you push: save the conversation

Your deliverables are this notebook run top to bottom and `vocal_handoff.md`, the record of your
tutor session including the three predictions you wrote down. Both go to `submissions/weeks/week_07/`.